# System Design & Architecture — Complete Interview Notebook

> **Purpose:** Comprehensive study guide + interview Q&A covering distributed systems, scalability, microservices, API design, event-driven architecture, high availability, design patterns, and classic system design problems.
>
> **Format:** Each section starts with **concept notes** (tables, comparisons, visual aids) followed by **interview Q&A tables**.
>
> **How to use:** Read the notes to learn concepts → then test yourself with the Q&A tables → review weak areas.


## 📋 Table of Contents

| # | Section | Topics |
|---|---------|--------|
| 1 | [Distributed Systems](#distributed-systems) | CAP Theorem · Consistency Models · Consensus (Raft, Paxos) · Leader Election · Replication · Sharding · Distributed Transactions · Idempotency |
| 2 | [Scalability](#scalability) | Vertical vs Horizontal · Load Balancing · Auto Scaling · Caching (Redis, CDN) · Database Indexing · Read Replicas · Rate Limiting |
| 3 | [Microservices Architecture](#microservices) | Monolith vs Micro · Service Discovery · API Gateway · gRPC · Circuit Breaker · Saga Pattern · Observability |
| 4 | [API Design](#api-design) | REST Principles · HTTP Methods/Status · Versioning · JWT/OAuth · Authorization · Pagination · OpenAPI |
| 5 | [Event-Driven Architecture](#event-driven) | Pub/Sub · Kafka · Event Sourcing · CQRS · Delivery Guarantees |
| 6 | [High Availability & Reliability](#high-availability) | Fault Tolerance · Redundancy · Failover · Health Checks · DR · Chaos Engineering |
| 7 | [Design Patterns (Architecture)](#design-patterns) | Layered · Clean · Hexagonal · DDD |
| 8 | [System Design Problems](#design-problems) | URL Shortener · Chat System · YouTube/Netflix · Ride Sharing · AI Chatbot |


---
<a id="distributed-systems"></a>
# Section 1: Distributed Systems

A **distributed system** is a collection of independent computers that appear to users as a single coherent system. Data and computation are spread across multiple machines connected by a network.


## 1.1 CAP Theorem

The **CAP Theorem** (Brewer's Theorem) states that a distributed data store can provide **at most two** of the following three guarantees simultaneously:

| Property | Meaning | Example |
|----------|---------|---------|
| **Consistency (C)** | Every read receives the most recent write or an error | After writing `x=5`, all nodes return `5` — no stale data |
| **Availability (A)** | Every request receives a non-error response (may not be latest) | System always responds, even if data might be slightly outdated |
| **Partition Tolerance (P)** | System continues to operate despite network partitions between nodes | If the network between Node A and Node B fails, the system still works |

> **Key Insight:** In a distributed system, **network partitions (P) are inevitable** — cables fail, switches die, data centers disconnect. So you must always have P, and the real choice is between **C** and **A**.

### CAP Trade-off in Practice

| Combination | What You Get | What You Lose | Real-World System |
|-------------|-------------|---------------|-------------------|
| **CP** (Consistency + Partition Tolerance) | Strong consistency, survives partitions | May reject requests during partitions | MongoDB (default), HBase, Redis Cluster, Zookeeper |
| **AP** (Availability + Partition Tolerance) | Always responds, survives partitions | May return stale data | Cassandra, DynamoDB, CouchDB, DNS |
| **CA** (Consistency + Availability) | Strong consistency, always available | Cannot survive partitions (single node only) | Traditional RDBMS (PostgreSQL on single node) |

### Visual: CAP Decision Tree

```
Network Partition Occurs?
├── YES (must choose)
│   ├── Prioritize Consistency → CP → reject requests until partition heals
│   └── Prioritize Availability → AP → serve possibly stale data
└── NO (no partition)
    └── CA → both consistency and availability (normal operation)
```

### PACELC Extension

| Condition | Choice | Meaning |
|-----------|--------|---------|
| **P** (Partition) | **A** vs **C** | Same as CAP — choose availability or consistency during partition |
| **E** (Else / No partition) | **L** vs **C** | Choose **Latency** vs **Consistency** during normal operation |

> **Example:** DynamoDB is **PA/EL** — during partition, chooses availability; during normal ops, chooses low latency (eventually consistent reads are faster).
> **Example:** MongoDB is **PC/EC** — during partition and normal ops, chooses consistency.


## 1.2 Consistency Models

| Model | Guarantee | Latency | Use Case |
|-------|-----------|---------|----------|
| **Strong Consistency** | Read always returns latest write | High (synchronous replication) | Banking, inventory, leader election |
| **Eventual Consistency** | Read will eventually return latest write | Low (async replication) | Social media feeds, DNS, session caches |
| **Causal Consistency** | Operations with causal relationship are seen in order | Medium | Chat messages, comments on posts |
| **Read-Your-Writes** | After writing, same client always sees its own write | Medium | User profile updates, settings changes |
| **Monotonic Reads** | Once you read a value, you never see an older value | Medium | Time-series data, progress tracking |
| **Linearizability** | Strongest — operations appear to happen at a single point in time | Highest | Critical financial transactions, locks |

### Strong vs Eventual Consistency — Detailed Comparison

| Aspect | Strong Consistency | Eventual Consistency |
|--------|-------------------|-----------------------|
| **Read guarantee** | Always latest value | May return stale value |
| **Write latency** | Higher (wait for all replicas to acknowledge) | Lower (write to one, replicate async) |
| **Availability** | Lower (must wait for quorum) | Higher (always responds) |
| **Implementation** | Synchronous replication, 2PC, Raft | Gossip protocol, async replication |
| **Conflict resolution** | Prevented by design | Last-write-wins, vector clocks, CRDTs |
| **When to use** | Money transfers, inventory counts | Likes count, activity feeds, caches |

### Quorum-Based Consistency

```
N = total replicas,  W = write quorum,  R = read quorum

Strong consistency when: W + R > N

Example (N=3):
  W=2, R=2 → Strong (2+2=4 > 3)  ← at least one node overlaps
  W=1, R=1 → Eventual (1+1=2 ≤ 3) ← might miss latest write
  W=3, R=1 → Strong but slow writes (all must ack)
  W=1, R=3 → Strong but slow reads (read from all)
```


## 1.3 Consensus Algorithms

Consensus algorithms allow a group of distributed nodes to **agree on a single value** even if some nodes fail. Essential for leader election, replicated state machines, and distributed logs.

### Raft Algorithm (Understandable Consensus)

| Concept | Description |
|---------|-------------|
| **Leader** | One node elected as leader; handles all client writes |
| **Follower** | Passive nodes that replicate leader's log |
| **Candidate** | Follower that starts an election when leader is suspected dead |
| **Term** | Logical clock — monotonically increasing election number |
| **Log** | Ordered sequence of commands replicated from leader to followers |
| **Heartbeat** | Leader sends periodic heartbeats to maintain authority |

### Raft: Step-by-Step Leader Election

```
1. Follower doesn't receive heartbeat within timeout
2. Follower increments term, becomes Candidate
3. Candidate votes for itself, sends RequestVote RPCs to all others
4. Other nodes vote (at most one vote per term)
5. If Candidate receives majority votes → becomes Leader
6. If another Leader's heartbeat received → steps down to Follower
7. If election timeout with no winner → new election with higher term
```

### Raft vs Paxos Comparison

| Aspect | Raft | Paxos |
|--------|------|-------|
| **Design goal** | Understandability | Theoretical correctness |
| **Leader** | Single strong leader (required) | Leader optional (multi-proposer) |
| **Phases** | 2 phases (election + log replication) | 3 phases (prepare, accept, learn) |
| **Complexity** | Simple to implement | Notoriously difficult to implement |
| **Real implementations** | etcd, CockroachDB, TiKV | Google Chubby, Apache Zookeeper (ZAB variant) |
| **Split-brain prevention** | Majority quorum per term | Majority quorum per ballot |
| **Industry preference** | Preferred for new systems | Legacy / theoretical reference |

### Paxos: Three Phases

| Phase | Actor | Action |
|-------|-------|--------|
| **Prepare** | Proposer | Sends `Prepare(n)` to acceptors (n = proposal number) |
| **Promise** | Acceptor | Promises not to accept proposals < n; returns any accepted value |
| **Accept** | Proposer | Sends `Accept(n, value)` if majority promised |
| **Learn** | Learner | Learns the chosen value once majority accepted |


## 1.4 Leader Election

Leader election selects **one node** from a group to act as the coordinator. Critical for replicated databases, distributed locks, and task scheduling.

### Leader Election Approaches

| Approach | How It Works | Pros | Cons |
|----------|-------------|------|------|
| **Bully Algorithm** | Highest ID node wins; lower nodes yield | Simple | Doesn't handle network partitions well |
| **Ring Election** | Nodes arranged in ring; election message circulates | Fair, no bottleneck | Slow (O(n) messages) |
| **Raft Election** | Random timeout → candidate → majority vote | Partition-safe, fast | Requires majority alive |
| **Zookeeper (ZAB)** | Ephemeral znodes — first to create becomes leader | Battle-tested, watches for failover | External dependency |
| **etcd Lease** | Acquire lease with TTL — holder is leader, renewal required | Simple, built on Raft | Requires etcd cluster |

### Split-Brain Problem

```
Normal:  [Node-A (Leader)] ←→ [Node-B] ←→ [Node-C]

Partition: [Node-A (Leader)] ✕✕✕ [Node-B] ←→ [Node-C]
                                    ↑
                               Node-B might elect itself leader!
                               
Result:   TWO leaders → conflicting writes → DATA CORRUPTION

Solution: Quorum-based election
  - 3-node cluster → need 2 votes to become leader
  - Node-A: has 1 vote (itself) → NOT leader
  - Node-B: has 2 votes (B + C) → becomes leader
  - Only ONE leader possible ✓
```


## 1.5 Replication Strategies

Replication copies data across multiple nodes for **fault tolerance** and **read scalability**.

### Replication Types

| Type | How It Works | Consistency | Latency | Durability |
|------|-------------|-------------|---------|------------|
| **Synchronous** | Write waits for ALL replicas to acknowledge | Strong | High | Highest |
| **Asynchronous** | Write returns after primary confirms; replicas updated later | Eventual | Low | Risk of data loss on primary failure |
| **Semi-synchronous** | Write waits for at least ONE replica (+ primary) | Strong-ish | Medium | Good (at least 2 copies guaranteed) |

### Replication Topologies

| Topology | Description | Use Case |
|----------|-------------|----------|
| **Single-Leader** | One primary (writes) → multiple replicas (reads) | Most RDBMS (PostgreSQL, MySQL) |
| **Multi-Leader** | Multiple primaries (each accepts writes) → cross-replicate | Multi-region writes (CockroachDB) |
| **Leaderless** | All nodes accept reads and writes; quorum for consistency | Cassandra, DynamoDB |

### Single-Leader Replication Flow

```
Client Write → Primary DB
                ├── WAL (Write-Ahead Log) on primary
                ├── Sync replica ← waits for ACK (strong)
                └── Async replicas ← doesn't wait (eventual)

Client Read → any replica (eventual) or primary (strong)
```

### Conflict Resolution in Multi-Leader / Leaderless

| Strategy | How It Works | Pros | Cons |
|----------|-------------|------|------|
| **Last-Write-Wins (LWW)** | Timestamp comparison — latest write wins | Simple | Data loss (earlier write discarded), clock skew issues |
| **Vector Clocks** | Per-node logical counters track causality | Detects concurrent writes | Complex, growing metadata |
| **CRDTs** | Data structures that automatically merge without conflicts | No coordination needed | Limited data types (counters, sets, maps) |
| **Application-level** | App-specific merge logic (e.g., merge shopping carts) | Most flexible | Application complexity |


## 1.6 Sharding / Partitioning

Sharding splits a large dataset across multiple machines. Each shard holds a **subset** of the total data.

### Why Shard?

| Problem | Without Sharding | With Sharding |
|---------|-----------------|---------------|
| Storage | Single machine disk limit (e.g., 2TB) | Unlimited — add more shards |
| Write throughput | Single machine bottleneck | Parallel writes across shards |
| Read throughput | Single machine CPU/IO | Distributed reads |
| Memory | Limited by single machine RAM | Aggregate RAM of all shards |

### Sharding Strategies

| Strategy | How It Works | Pros | Cons |
|----------|-------------|------|------|
| **Range-based** | Shard by key range (A-M → Shard 1, N-Z → Shard 2) | Range queries efficient, simple | Hotspots (uneven distribution) |
| **Hash-based** | `hash(key) % num_shards` determines shard | Even distribution, no hotspots | Range queries span all shards |
| **Geo-based** | Shard by geographic region | Low latency for regional users | Cross-region queries expensive |
| **Directory-based** | Lookup table maps each key → shard | Flexible, any mapping | Lookup table is SPOF and bottleneck |

### Consistent Hashing (Key Concept)

```
Problem with hash(key) % N:
  - Adding/removing a shard changes N
  - Most keys remap → massive data migration

Consistent Hashing:
  - Nodes and keys both mapped to a ring (0 to 2^32)
  - Key assigned to first node clockwise from its position
  - Adding a node: only keys between new node and predecessor move
  - Result: only K/N keys remapped (vs nearly all with modulo)

Ring visualization:
       0
    /     \
  Node-C   Node-A
    \     /
     Node-B
      180
      
Key "user:123" hashes to position 45 → assigned to Node-A (next clockwise)
Add Node-D at position 30 → only keys 0-30 move from Node-A to Node-D
```

### Virtual Nodes

| Concept | Explanation |
|---------|-------------|
| **Problem** | With few physical nodes, distribution is uneven on the ring |
| **Solution** | Each physical node gets multiple "virtual nodes" (e.g., 150 each) spread around the ring |
| **Result** | Much more even data distribution, smoother rebalancing |
| **Used by** | Cassandra, DynamoDB, Riak |


## 1.7 Distributed Transactions

A distributed transaction spans multiple services or databases and must maintain **ACID** properties across all participants.

### Two-Phase Commit (2PC)

| Phase | Coordinator Action | Participant Action |
|-------|-------------------|-------------------|
| **Phase 1: Prepare** | Sends `PREPARE` to all participants | Writes to WAL, locks resources, responds `VOTE_COMMIT` or `VOTE_ABORT` |
| **Phase 2: Commit** | If ALL voted commit → sends `COMMIT` | Applies transaction, releases locks |
| **Phase 2: Abort** | If ANY voted abort → sends `ABORT` | Rolls back transaction, releases locks |

```
Coordinator               Participant-A           Participant-B
    |--- PREPARE ------------>|                        |
    |--- PREPARE ---------------------------------->|
    |<-- VOTE_COMMIT ---------|                        |
    |<-- VOTE_COMMIT -------------------------------|
    |--- COMMIT ------------->|                        |
    |--- COMMIT ---------------------------------->|
    |<-- ACK -----------------|                        |
    |<-- ACK ----------------------------------|
```

### 2PC vs 3PC vs Saga

| Aspect | 2PC | 3PC | Saga |
|--------|-----|-----|------|
| **Blocking** | Yes — participants block if coordinator dies | Non-blocking (adds pre-commit phase) | Non-blocking (each step independent) |
| **Coordinator SPOF** | Yes — if coordinator dies after prepare, participants hang | Reduced — timeout leads to abort | No coordinator — choreography or orchestration |
| **Performance** | Slow (synchronous locks) | Slower (extra round trip) | Fast (async, no global locks) |
| **Consistency** | Strong (atomicity guaranteed) | Strong | Eventual (compensating transactions fix failures) |
| **Use case** | Database-level distributed transactions | Rarely used (complex, marginal benefit) | Microservices (most common) |

### Saga Pattern — Two Styles

| Style | How It Works | Pros | Cons |
|-------|-------------|------|------|
| **Choreography** | Each service publishes events; next service reacts | Loosely coupled, simple | Hard to track, circular dependencies possible |
| **Orchestration** | Central orchestrator tells each service what to do | Easy to understand, centralized logic | Orchestrator is coupling point, SPOF |

```
Choreography Saga:
  OrderService → publishes "OrderCreated"
    → PaymentService processes payment → publishes "PaymentCompleted"  
      → InventoryService reserves stock → publishes "StockReserved"
        → ShippingService creates shipment → publishes "OrderShipped"

If PaymentService fails:
  → publishes "PaymentFailed"
    → OrderService runs compensating transaction → cancels order
```


## 1.8 Idempotency

An operation is **idempotent** if performing it multiple times has the same effect as performing it once.

### HTTP Methods & Idempotency

| Method | Idempotent? | Safe? | Explanation |
|--------|:-----------:|:-----:|-------------|
| GET | ✅ Yes | ✅ Yes | Reading data — same result every time |
| PUT | ✅ Yes | ❌ No | Replace resource — doing it twice = same state |
| DELETE | ✅ Yes | ❌ No | Delete resource — deleting twice = still deleted |
| POST | ❌ No | ❌ No | Create resource — doing it twice = TWO resources |
| PATCH | ❌ No* | ❌ No | Depends on implementation (increment vs set) |

### Idempotency Patterns for Non-Idempotent Operations

| Pattern | How It Works | Use Case |
|---------|-------------|----------|
| **Idempotency Key** | Client sends unique key with request; server checks if already processed | Payment APIs (Stripe, PayPal) |
| **Database Unique Constraint** | Insert fails on duplicate key → return existing result | User registration, order creation |
| **Token-based** | Server issues one-time-use token; consumed on first request | Form submissions, CSRF protection |
| **Conditional Writes** | `UPDATE ... WHERE version = X` — fails if version changed | Optimistic concurrency control |

```
Idempotency Key Flow:
1. Client generates UUID: "abc-123"
2. POST /payments  {idempotency_key: "abc-123", amount: 100}
3. Server checks Redis: key "abc-123" exists? NO → process payment
4. Server saves to Redis: "abc-123" → {status: "success", id: "pay_456"} with 24h TTL
5. Network error — client retries same request
6. Server checks Redis: key "abc-123" exists? YES → return cached result
7. Payment processed exactly once ✓
```


---
## Section 1 — Interview Questions: Distributed Systems (80 Questions)


| # | Question | Keywords | Answer |
|---|----------|----------|--------|
| 1 | What is a distributed system? | distributed, definition | • Collection of independent computers that appear as a single system to users<br>• Data and computation spread across multiple machines connected by network<br>• Goals: scalability, fault tolerance, geographic distribution<br>• Challenges: network failures, clock sync, partial failures, consensus<br>• Examples: Google Search, Amazon DynamoDB, Cassandra, Kafka |
| 2 | What is the CAP theorem? | CAP, Brewer | • States that a distributed data store can provide at most 2 of 3 guarantees: Consistency, Availability, Partition Tolerance<br>• Network partitions are inevitable → real choice is between C and A<br>• CP systems: reject requests during partition (MongoDB, Zookeeper)<br>• AP systems: serve stale data during partition (Cassandra, DynamoDB)<br>• CA systems: only possible on single node (no partitions to tolerate) |
| 3 | Why is partition tolerance mandatory in distributed systems? | partition tolerance, mandatory | • Network partitions are a reality — cables fail, switches die, data centers disconnect<br>• You cannot "opt out" of partitions — they happen regardless of your design<br>• A system without P is just a single-node system (not distributed)<br>• The practical choice is always CP or AP<br>• Even within a single data center, partitions occur (rack switches fail) |
| 4 | Give a real-world example of a CP system. | CP, example | • MongoDB (default config): during network partition, the minority partition becomes read-only<br>• Primary node must be in the majority partition to accept writes<br>• Reads from secondaries may be stale but writes are consistent<br>• Trade-off: unavailability of writes during partition vs data consistency<br>• Other examples: HBase, Zookeeper, etcd, Redis Cluster |
| 5 | Give a real-world example of an AP system. | AP, example | • Cassandra: during network partition, any node can accept writes<br>• Uses last-write-wins conflict resolution with timestamps<br>• Always available for reads and writes, but data may be temporarily inconsistent<br>• Eventually consistent — replicas sync after partition heals<br>• Other examples: DynamoDB, CouchDB, DNS |
| 6 | What is the PACELC theorem? | PACELC, extension | • Extension of CAP: during Partition → A vs C; Else → Latency vs Consistency<br>• Addresses trade-off during normal operation (no partition)<br>• DynamoDB: PA/EL — prioritizes availability during partition, low latency normally<br>• MongoDB: PC/EC — prioritizes consistency always<br>• More practical than CAP because partitions are rare; normal-mode trade-off matters more |
| 7 | What is strong consistency? | strong, consistency | • Every read returns the most recent write — no stale data ever<br>• Achieved via synchronous replication or quorum reads/writes<br>• Higher latency: must wait for replicas to confirm<br>• Use cases: banking, inventory management, leader election<br>• Implementation: Raft, 2PC, synchronous replication |
| 8 | What is eventual consistency? | eventual, consistency | • If no new updates, all replicas will eventually converge to the same value<br>• Reads may return stale data temporarily (consistency window)<br>• Lower latency: writes acknowledged by single node, replicated async<br>• Use cases: social media likes, DNS propagation, activity feeds<br>• Conflict resolution needed: LWW, vector clocks, CRDTs |
| 9 | What is causal consistency? | causal, consistency | • Operations with a causal relationship are seen in the correct order by all nodes<br>• If A causes B, everyone sees A before B — but concurrent operations may appear in any order<br>• Weaker than strong consistency but stronger than eventual<br>• Example: reply to a comment always appears after the comment<br>• Implementation: vector clocks, Lamport timestamps |
| 10 | What is linearizability? | linearizability, strongest | • Strongest consistency model — operations appear to execute at a single point in time<br>• Once a write completes, ALL subsequent reads (from any client) see that write<br>• Real-time ordering: if operation A completes before B starts, A appears first<br>• Expensive to implement (requires coordination on every operation)<br>• Use cases: distributed locks, critical financial transactions |
| 11 | Explain quorum-based consistency. | quorum, N, W, R | • N = total replicas, W = write quorum, R = read quorum<br>• Strong consistency when: W + R > N (read overlaps with at least one up-to-date node)<br>• Common config for N=3: W=2, R=2 (strong) or W=1, R=1 (eventual, faster)<br>• Trade-off: higher W → slower writes but more durable; higher R → slower reads but more fresh<br>• Used by: DynamoDB, Cassandra, Riak |
| 12 | What is the Raft consensus algorithm? | Raft, consensus | • Designed for understandability — equivalent to Paxos but much simpler<br>• Three roles: Leader (handles all writes), Follower (replicates), Candidate (seeks election)<br>• Leader election: random timeout → candidate → majority vote → leader<br>• Log replication: leader appends to log → replicates to followers → commits when majority confirm<br>• Used by: etcd, CockroachDB, TiKV, Consul |
| 13 | Walk through a Raft leader election step by step. | Raft, election, steps | • 1. Follower's election timeout fires (no heartbeat from leader)<br>• 2. Follower increments term number, becomes Candidate<br>• 3. Candidate votes for itself, sends RequestVote RPCs to all peers<br>• 4. Each node votes for at most one candidate per term (first-come-first-served)<br>• 5. If Candidate receives majority → becomes Leader, sends heartbeats<br>• 6. If receives heartbeat from valid Leader → steps down to Follower<br>• 7. If election timeout with no winner → new election with incremented term |
| 14 | How does Raft handle log replication? | Raft, log replication | • Client sends write to Leader → Leader appends to its log<br>• Leader sends AppendEntries RPC to all Followers<br>• Followers append to their logs, respond with success<br>• When Leader receives majority acknowledgement → entry is committed<br>• Leader notifies client of success; committed entries are durable<br>• Followers apply committed entries to their state machines |
| 15 | What is Paxos and how does it differ from Raft? | Paxos, vs Raft | • Paxos: consensus protocol with Prepare/Accept/Learn phases<br>• Multi-proposer: any node can propose (vs Raft's single leader)<br>• More general but notoriously difficult to implement correctly<br>• Raft was designed to be the "understandable Paxos"<br>• Raft has strong leader requirement; Paxos can work without leader<br>• In practice, Raft is preferred for new systems; Paxos is legacy/theoretical |
| 16 | What is the split-brain problem? | split brain, partition | • Network partition creates two groups that can't communicate<br>• Each group might elect its own leader → two leaders accepting conflicting writes<br>• Results in data inconsistency, potential data loss when partition heals<br>• Solution: require majority quorum for leader election (only one group has majority)<br>• 3-node cluster: only the partition with 2+ nodes can elect a leader |
| 17 | How do you prevent split-brain? | split brain, prevention | • Quorum-based election: require N/2 + 1 votes (majority) to become leader<br>• Odd number of nodes: 3, 5, 7 — ensures exactly one partition can have majority<br>• Fencing tokens: old leader's writes rejected by storage after new leader elected<br>• STONITH (Shoot The Other Node In The Head): forcibly shut down the old leader<br>• Lease-based: leader holds a lease with TTL; must renew or step down |
| 18 | What is single-leader replication? | single leader, replication | • One node designated as primary (leader) — all writes go through it<br>• Multiple replicas (followers) — receive replication stream from primary<br>• Reads can go to any replica (eventual) or primary only (strong)<br>• Used by: PostgreSQL, MySQL, MongoDB (replica set)<br>• Advantages: simple, no write conflicts. Disadvantage: single write bottleneck |
| 19 | What is multi-leader replication? | multi leader, replication | • Multiple nodes accept writes independently, cross-replicate to each other<br>• Use case: multi-region deployments (each region has a leader for low latency)<br>• Challenge: write conflicts when two leaders modify same data simultaneously<br>• Conflict resolution: LWW, merge functions, or CRDT-based<br>• Examples: CockroachDB, active-active PostgreSQL (BDR), Google Spanner |
| 20 | What is leaderless replication? | leaderless, replication | • No designated leader — any node can accept reads and writes<br>• Quorum: W nodes must acknowledge write, R nodes read for consistency<br>• No failover needed — no leader to fail over from<br>• Used by: Cassandra, DynamoDB, Riak<br>• Challenge: conflict resolution for concurrent writes (vector clocks, LWW) |


| # | Question | Keywords | Answer |
|---|----------|----------|--------|
| 21 | What is last-write-wins (LWW) conflict resolution? | LWW, conflict | • When concurrent writes conflict, the one with the latest timestamp wins<br>• Simple to implement — just compare timestamps<br>• Problem: clock skew between nodes can cause "wrong" winner<br>• Data loss: earlier write is silently discarded<br>• Cassandra default strategy; acceptable for non-critical data like view counts |
| 22 | What are vector clocks? | vector clocks, causality | • Each node maintains a counter; together they form a vector [A:2, B:1, C:3]<br>• On write: increment own counter, attach vector to value<br>• Comparison: V1 < V2 if all V1 entries ≤ V2 entries (V2 is newer)<br>• If neither dominates → concurrent writes → conflict detected, app resolves<br>• Used by Amazon DynamoDB (internally) and Riak |
| 23 | What are CRDTs? | CRDT, conflict-free | • Conflict-free Replicated Data Types — merge without coordination<br>• Types: G-Counter (grow only), PN-Counter (add/subtract), G-Set, OR-Set, LWW-Register<br>• Property: any order of operations produces same result (commutative, associative, idempotent)<br>• No conflicts by design — mathematical guarantee<br>• Used by: Redis (CRDB), Riak, collaborative editing (Figma) |
| 24 | What is sharding/partitioning? | sharding, partitioning | • Splitting a dataset across multiple machines (shards/partitions)<br>• Each shard holds a subset of total data<br>• Purpose: scale storage, write throughput, and query parallelism beyond single machine<br>• Types: range-based, hash-based, geo-based, directory-based<br>• Challenge: cross-shard queries, rebalancing, join operations |
| 25 | Compare range-based vs hash-based sharding. | range vs hash, sharding | • Range-based: shard by key ranges (A-M → Shard 1) — efficient range queries but prone to hotspots<br>• Hash-based: hash(key) % N determines shard — even distribution but range queries hit all shards<br>• Range: good for time-series data (recent data on hot shard, old data archived)<br>• Hash: good for user data (even distribution by user_id)<br>• Many systems support both: MongoDB (range and hashed shard keys) |
| 26 | Explain consistent hashing. | consistent hashing, ring | • Keys and nodes mapped to a ring (0 to 2^32)<br>• Key assigned to first node clockwise from its hash position<br>• Adding a node: only keys between new node and predecessor migrate<br>• Removing a node: only that node's keys migrate to successor<br>• Much better than `hash % N` which remaps nearly all keys on change<br>• Used by: Cassandra, DynamoDB, memcached, CDNs |
| 27 | What are virtual nodes in consistent hashing? | virtual nodes, vnodes | • Each physical node gets multiple positions on the hash ring (e.g., 150 vnodes)<br>• Solves uneven distribution with few physical nodes<br>• When node is added: its vnodes spread across ring, taking small chunks from each existing node<br>• Result: much smoother data distribution and rebalancing<br>• Used by: Cassandra (default 256 vnodes per node) |
| 28 | What is a hotspot in sharding and how do you handle it? | hotspot, sharding | • Hotspot: one shard receives disproportionately more traffic than others<br>• Causes: celebrity user (millions of followers), time-based range shard (current month), viral content<br>• Solutions: add random suffix to hot key (spread across shards), dedicated shard for hot keys<br>• Monitoring: track per-shard request rates, alert on imbalance<br>• Example: Twitter's hot key problem with celebrity tweet distribution |
| 29 | What is the two-phase commit (2PC) protocol? | 2PC, distributed transaction | • Atomic commitment protocol for distributed transactions<br>• Phase 1 (Prepare): coordinator asks all participants to prepare (vote commit/abort)<br>• Phase 2 (Commit/Abort): if all voted commit → coordinator sends COMMIT; if any voted abort → ABORT<br>• Guarantees atomicity: either all commit or all abort<br>• Problem: blocking — if coordinator fails after prepare, participants hold locks and wait |
| 30 | Why is 2PC problematic in microservices? | 2PC, microservices, problems | • Blocking: participants hold locks until coordinator resolves (could be forever if coordinator dies)<br>• Tightly coupled: all participants must be available simultaneously<br>• Performance: synchronous, high latency, reduces throughput<br>• Not partition-tolerant: fails if network partition between coordinator and participants<br>• Alternative: Saga pattern — eventual consistency with compensating transactions |
| 31 | What is the Saga pattern? | Saga, pattern | • Sequence of local transactions across services, each with a compensating transaction for rollback<br>• If step N fails: execute compensating transactions for steps N-1, N-2, ..., 1<br>• Two styles: Choreography (event-driven) and Orchestration (central coordinator)<br>• Provides eventual consistency (not atomicity like 2PC)<br>• Example: Order → Payment → Inventory → Shipping; if inventory fails → refund payment → cancel order |
| 32 | What is the difference between choreography and orchestration sagas? | choreography vs orchestration | • Choreography: each service publishes events, next service subscribes and reacts<br>• Orchestration: central orchestrator tells each service what to do in sequence<br>• Choreography pros: loosely coupled, no SPOF. Cons: hard to track, circular events<br>• Orchestration pros: clear flow, easier debugging. Cons: orchestrator is coupling point<br>• Small flows → choreography; complex flows (5+ steps) → orchestration |
| 33 | What is idempotency and why is it important? | idempotency, definition | • Operation that produces same result whether executed once or multiple times<br>• Critical in distributed systems because: network retries, message duplication, at-least-once delivery<br>• Without idempotency: retrying a payment could charge the customer twice<br>• HTTP: GET, PUT, DELETE are idempotent by design; POST is not<br>• Pattern: idempotency key (unique request ID checked before processing) |
| 34 | How do you implement idempotency for a payment API? | idempotency, payment | • Client generates unique idempotency_key (UUID) and sends with request<br>• Server checks: has this key been processed? (lookup in Redis/DB)<br>• If YES → return cached result (don't process again)<br>• If NO → process payment → store result with key (TTL 24-48 hours)<br>• Database unique constraint on idempotency_key prevents race conditions<br>• Stripe and PayPal both use this exact pattern |
| 35 | What is a Lamport timestamp? | Lamport, timestamp, logical | • Logical clock for ordering events in distributed systems<br>• Rules: (1) increment counter before each event, (2) on send, attach counter, (3) on receive, set counter to max(local, received) + 1<br>• Guarantees: if A happened-before B, then L(A) < L(B)<br>• Does NOT guarantee: if L(A) < L(B), then A happened-before B (concurrent events may have any Lamport order)<br>• Simpler than vector clocks but less information |
| 36 | What is a distributed lock? | distributed lock, coordination | • Lock that works across multiple machines/processes<br>• Ensures mutual exclusion for critical sections in distributed systems<br>• Implementations: Zookeeper (ephemeral znodes), Redis (Redlock), etcd (lease-based)<br>• Challenges: lock holder dies without releasing, network partitions, clock skew<br>• Fencing token: monotonically increasing token prevents stale lock holder from writing |
| 37 | What is the Redlock algorithm? | Redlock, Redis, lock | • Distributed lock across N independent Redis instances (recommended N=5)<br>• Acquire: try to SET NX lock on all N instances with TTL<br>• Success if lock acquired on majority (N/2 + 1) within a time less than TTL<br>• Release: DELETE lock on all instances<br>• Controversial: Martin Kleppmann argued it's unsafe due to clock skew and GC pauses |
| 38 | What is a distributed counter? | counter, distributed | • Counter that can be incremented/decremented from multiple nodes<br>• Challenge: concurrent increments can lead to lost updates<br>• Solutions: (1) centralized counter with lock, (2) per-node counters summed on read, (3) CRDTs<br>• CRDT G-Counter: each node has own counter, total = sum of all nodes<br>• Used for: like counts, view counts, rate limiting across nodes |
| 39 | What is gossip protocol? | gossip, protocol | • Communication protocol where nodes spread information like a rumor<br>• Each node periodically picks random peer and exchanges state<br>• Information spreads exponentially: O(log N) rounds to reach all N nodes<br>• Used for: failure detection, membership, state dissemination<br>• Examples: Cassandra (failure detection), Amazon S3 (data replication metadata) |
| 40 | What is the Byzantine Generals Problem? | Byzantine, fault tolerance | • How to reach consensus when some nodes may be malicious (sending conflicting info)<br>• Requires 3f+1 nodes to tolerate f Byzantine faults (need 2/3 majority honest)<br>• More expensive than crash-fault tolerance (which only handles dead nodes)<br>• Solutions: PBFT (Practical Byzantine Fault Tolerance), blockchain consensus<br>• Most enterprise systems assume non-Byzantine faults (trusted internal network) |


| # | Question | Keywords | Answer |
|---|----------|----------|--------|
| 41 | What is the write-ahead log (WAL)? | WAL, durability | • Append-only log where changes are written BEFORE applying to the database<br>• Guarantees durability: if crash occurs, replay WAL to recover state<br>• Used by: PostgreSQL, MySQL (redo log), etcd, Kafka (commit log)<br>• WAL is append-only → fast sequential writes (much faster than random disk I/O)<br>• Replication: WAL shipped to replicas for synchronization |
| 42 | What is a distributed snapshot? | snapshot, distributed | • Consistent view of the state of all nodes at a logical point in time<br>• Chandy-Lamport algorithm: marker messages determine snapshot boundaries<br>• Use cases: backup, debugging, checkpoint/restart<br>• Challenge: no global clock → must use logical markers<br>• Example: Apache Flink's checkpointing mechanism uses distributed snapshots |
| 43 | What is a lease in distributed systems? | lease, TTL | • Time-limited grant of a resource (like a lock with built-in expiry)<br>• Holder must renew before TTL expires, otherwise resource released<br>• Advantage: no "forgotten locks" — lease expires automatically if holder dies<br>• Used by: etcd (leader leases), Zookeeper (session TTL), DHCP<br>• Risk: clock skew could cause two holders to think they have the lease simultaneously |
| 44 | What is data partitioning vs data replication? | partition vs replication | • Partitioning: split data across nodes (each node has DIFFERENT data) — for scalability<br>• Replication: copy data across nodes (each node has SAME data) — for availability<br>• Used together: each partition replicated to multiple nodes<br>• Example: Cassandra — data partitioned by hash, each partition replicated to 3 nodes<br>• Partitioning increases capacity; replication increases fault tolerance |
| 45 | What is fan-out in distributed systems? | fan-out, architecture | • Pattern where one message/request triggers many downstream actions<br>• Fan-out-on-write: when user posts, immediately write to all followers' timelines (Twitter approach)<br>• Fan-out-on-read: when user opens timeline, query all followees' posts on the fly<br>• Trade-off: write amplification vs read latency<br>• Hybrid: fan-out-on-write for most users, fan-out-on-read for celebrities |
| 46 | What is tail latency and why does it matter? | tail latency, P99 | • Tail latency: the response time at high percentiles (P99, P99.9)<br>• Matters because: a user request may fan out to 100 services; overall latency = slowest service<br>• With 100 services and P99 = 100ms each, 63% of requests hit at least one slow response<br>• Causes: GC pauses, context switches, disk I/O, network congestion<br>• Mitigation: hedged requests (send to multiple replicas, use first response) |
| 47 | What is a sidecar pattern? | sidecar, pattern | • Deploy a helper process alongside the main service in the same pod/VM<br>• Sidecar handles cross-cutting concerns: logging, monitoring, service mesh proxy<br>• Main service communicates with sidecar via localhost<br>• Used by: Envoy proxy (Istio), log collectors (Fluentd), config refreshers<br>• Benefits: language-agnostic infrastructure concerns, separation of concerns |
| 48 | What is a service mesh? | service mesh, Istio | • Infrastructure layer for service-to-service communication<br>• Handles: load balancing, encryption (mTLS), observability, retries, circuit breaking<br>• Data plane: sidecar proxies (Envoy) intercept all network traffic<br>• Control plane: manages proxy configuration (Istio, Linkerd)<br>• Benefits: consistent networking policies without modifying application code |
| 49 | What is a bloom filter? | bloom filter, probabilistic | • Space-efficient probabilistic data structure for set membership testing<br>• Returns: "definitely not in set" or "probably in set" (false positives possible, no false negatives)<br>• Used to avoid unnecessary disk reads: check bloom filter → if negative, skip disk lookup<br>• Used by: Cassandra (SSTable lookup), Google Bigtable, Chrome (malicious URL check)<br>• Trade-off: more hash functions and bits → lower false positive rate → more memory |
| 50 | What is the difference between horizontal and vertical partitioning? | horizontal vs vertical | • Horizontal partitioning (sharding): split ROWS across shards (each shard has all columns, subset of rows)<br>• Vertical partitioning: split COLUMNS across tables (each table has all rows, subset of columns)<br>• Horizontal: scale read/write throughput, distribute storage<br>• Vertical: separate hot columns (queried often) from cold columns (large, rarely accessed)<br>• Example vertical: move user profile blob (1MB) to separate table from user auth data (100 bytes) |
| 51 | What is clock skew and why is it dangerous? | clock skew, time | • Difference in time between clocks on different machines<br>• NTP synchronization typically achieves ±10ms accuracy (not perfect)<br>• Dangerous: LWW conflict resolution depends on accurate timestamps — wrong clock = wrong winner<br>• Dangerous: lease expiry — node thinks lease valid but actually expired<br>• Solution: logical clocks (vector clocks, Lamport timestamps) instead of wall clocks |
| 52 | What is a tombstone in distributed databases? | tombstone, delete | • Marker indicating a record has been deleted (not actually removing it immediately)<br>• Necessary because: in eventual consistency, a delete must be propagated to all replicas<br>• Without tombstone: deleted data could reappear when syncing with a replica that didn't get the delete<br>• Tombstones are garbage-collected after a grace period (e.g., 10 days in Cassandra)<br>• Cost: tombstones consume storage and slow reads until compacted |
| 53 | What is the difference between synchronous and asynchronous replication? | sync vs async, replication | • Synchronous: write waits for replica ACK before responding to client — strong consistency, higher latency<br>• Asynchronous: write responds immediately, replica updated later — eventual consistency, lower latency<br>• Semi-sync: wait for at least one replica ACK — balance of durability and latency<br>• Sync risk: if replica slow → write latency spikes for everyone<br>• Async risk: if primary fails → replicated data may be lost |
| 54 | How does follower catch-up work after recovery? | recovery, catch-up | • Follower reconnects after downtime<br>• Checks its last-known position in the replication stream (LSN/offset)<br>• Primary sends all WAL entries since that position (catch-up replication)<br>• If too far behind → full snapshot from primary + WAL from snapshot point<br>• Once caught up → resume normal streaming replication |
| 55 | What is read-your-writes consistency? | read-your-writes | • After a write, the SAME client always sees its own write on subsequent reads<br>• Other clients may not see the write immediately (eventually consistent for them)<br>• Implementation: route reads to the same replica that handled the write, or read from primary<br>• Common UX requirement: user updates profile → should see updated profile immediately<br>• Also called "read-after-write consistency" |
| 56 | What is monotonic reads consistency? | monotonic reads | • Once a client reads a value, it will never see an older value in subsequent reads<br>• Prevents "time travel" — reading a newer value then an older one<br>• Without it: user refreshes page and sees old data, then refreshes again and sees new data, then old again<br>• Implementation: sticky sessions (always read from same replica) or track read position<br>• Weaker than strong consistency but avoids confusing user experience |
| 57 | What are the challenges of distributed joins? | distributed join, challenge | • Data for both tables may live on different shards<br>• Broadcast join: send entire small table to all shards (expensive for large tables)<br>• Colocated join: shard both tables by join key (ensures matching rows on same shard)<br>• Cross-shard join: coordinator gathers data from multiple shards and joins locally<br>• Best practice: denormalize data to avoid distributed joins, or pre-compute join results |
| 58 | What is data denormalization and when is it used? | denormalization, distributed | • Storing redundant copies of data to avoid joins and cross-shard queries<br>• Example: store author_name in every blog post (instead of joining with users table)<br>• Trade-off: faster reads, more storage, more complex writes (must update all copies)<br>• Common in NoSQL databases where joins are not supported<br>• Decision: if reads >> writes and cross-service joins needed → denormalize |
| 59 | What is the outbox pattern? | outbox, pattern | • Reliably publish events when database state changes (transactional outbox)<br>• Write business data AND event to outbox table in SAME database transaction<br>• Separate process reads outbox table and publishes events to message broker<br>• Guarantees: event published if and only if database transaction committed<br>• Solves: dual-write problem (writing to DB and Kafka separately can lose either) |
| 60 | What is the dual-write problem? | dual write, problem | • Updating two systems (e.g., DB and Kafka) without distributed transaction<br>• If DB write succeeds but Kafka publish fails → inconsistent state<br>• If Kafka publish succeeds but DB write fails → event without matching data<br>• Solutions: outbox pattern (single DB transaction), change data capture (CDC), event sourcing<br>• Never write to two systems independently without a coordination mechanism |


| # | Question | Keywords | Answer |
|---|----------|----------|--------|
| 61 | What is change data capture (CDC)? | CDC, change data capture | • Capturing and streaming changes from a database's transaction log<br>• Tools: Debezium (open-source), AWS DMS, Oracle GoldenGate<br>• Reads PostgreSQL WAL / MySQL binlog → publishes as events to Kafka<br>• Benefits: no application code changes needed, captures all changes<br>• Use cases: sync databases, populate search indexes, feed event-driven systems |
| 62 | What is a distributed queue vs a distributed log? | queue vs log | • Queue: messages consumed and deleted (RabbitMQ, SQS) — each message processed once<br>• Log: messages appended and retained (Kafka) — consumers track offset, messages replayable<br>• Queue: best for task distribution (worker pools)<br>• Log: best for event streaming (multiple consumers, replay, audit)<br>• Key difference: log preserves history; queue doesn't |
| 63 | What is backpressure in distributed systems? | backpressure, flow control | • Mechanism to slow down producers when consumers can't keep up<br>• Without backpressure: memory overflow, dropped messages, system crash<br>• TCP backpressure: TCP window shrinks when receiver buffer full<br>• App level: reject new requests (429 Too Many Requests), queue with bounded size<br>• Reactive Streams: publisher notified of subscriber capacity |
| 64 | What are the failure modes in distributed systems? | failure modes, types | • Crash failure: node stops and doesn't recover (hardware death)<br>• Omission failure: node fails to send or receive messages (network)<br>• Timing failure: response comes too late (timeout)<br>• Byzantine failure: node sends incorrect or conflicting messages (malicious/buggy)<br>• Most systems handle crash + omission; Byzantine tolerance is expensive |
| 65 | What is a circuit breaker in distributed systems? | circuit breaker, resilience | • Pattern that prevents cascading failures by stopping requests to failing services<br>• States: CLOSED (normal) → OPEN (failing, reject all) → HALF-OPEN (test with limited traffic)<br>• Transition: after N failures in window → OPEN → after timeout → HALF-OPEN → if success → CLOSED<br>• Benefits: fail fast, preserve resources, allow failing service to recover<br>• Libraries: Hystrix (Netflix, deprecated), resilience4j, Polly (.NET) |
| 66 | What is the difference between fail-over and fail-back? | failover, failback | • Failover: switching from failed primary to standby (automated or manual)<br>• Failback: switching back to the original primary after it recovers<br>• Hot standby: always running, ready to take over immediately<br>• Cold standby: needs to be started and synced before taking over<br>• Risk: failback can be dangerous if primary database diverged during downtime |
| 67 | What is a distributed hash table (DHT)? | DHT, hash table | • Key-value store distributed across many nodes using consistent hashing<br>• Each node responsible for a range of keys on the hash ring<br>• Lookup: O(log N) hops to find responsible node (using finger table)<br>• Used by: BitTorrent (peer discovery), IPFS, Kademlia<br>• Difference from centralized: no single point of failure, scales to millions of nodes |
| 68 | What is the thundering herd problem? | thundering herd, cache | • Cache expires → many simultaneous requests hit the database at once<br>• Causes: popular cache key expires, all waiting requests bypass cache together<br>• Solutions: (1) lock on cache miss — only one request fetches, others wait<br>• (2) Probabilistic early expiration (jitter)<br>• (3) Never expire — background refresh before TTL |
| 69 | What is request coalescing? | coalescing, request | • Combining multiple identical requests into one<br>• When cache miss occurs: first request fetches from DB, subsequent identical requests wait for that result<br>• Also called "request collapsing" or "singleflight"<br>• Used by: Go's singleflight package, nginx proxy_cache with proxy_cache_lock<br>• Prevents thundering herd on same key |
| 70 | What is geo-replication? | geo-replication, multi-region | • Replicating data across geographically distributed data centers<br>• Benefits: lower latency for users worldwide, disaster recovery across regions<br>• Challenge: replication lag between regions (cross-continent: 50-200ms)<br>• Strategies: active-passive (one region primary, others read-only) or active-active (all regions accept writes)<br>• Active-active conflict resolution: CRDTs, LWW, or application-level merge |
| 71 | What is data locality and why does it matter? | data locality, performance | • Keeping related data physically close together (same shard, same disk page)<br>• Benefits: fewer network hops, fewer disk seeks, better cache utilization<br>• Example: shard by user_id → all of a user's data on one shard → no cross-shard queries<br>• Columnar storage: related columns stored together for analytical queries<br>• Anti-pattern: random sharding breaks locality |
| 72 | What is the split-brain resolver? | resolver, split brain | • Mechanism to resolve split-brain after partition heals<br>• Strategy 1: Accept data from the partition with more nodes (majority)<br>• Strategy 2: Accept data from the partition with more recent writes (LWW)<br>• Strategy 3: Merge data from both partitions (CRDTs, application-level merge)<br>• Strategy 4: Discard data from the minority partition (simplest, data loss risk) |
| 73 | What is exactly-once semantics? | exactly once, semantics | • Each message processed exactly one time — no duplicates, no losses<br>• Very hard to achieve in distributed systems (network failures cause retries)<br>• At-most-once: no retries → may lose messages<br>• At-least-once: retry on failure → may duplicate<br>• Exactly-once: at-least-once + idempotent consumer (deduplicate on receive)<br>• Kafka achieves it via idempotent producer + transactional consumer |
| 74 | What is anti-entropy in distributed systems? | anti-entropy, repair | • Background process that compares data across replicas and fixes differences<br>• Techniques: Merkle trees (hash trees) to efficiently find different data blocks<br>• Runs periodically: doesn't affect normal read/write performance significantly<br>• Used by: Cassandra (repair), DynamoDB, S3 (background verification)<br>• Complements read-repair (which fixes data during normal reads) |
| 75 | What is read repair? | read repair, consistency | • During a read, if different replicas return different values, update stale replicas<br>• Works with quorum reads: compare R responses, return most recent, send updates to stale replicas<br>• Piggybacks on normal reads — no extra background process needed<br>• Limitation: only repairs data that is actually read — unpopular data stays inconsistent<br>• Used by: Cassandra, DynamoDB |
| 76 | What is hinted handoff? | hinted handoff, availability | • When a write's target node is down, another node temporarily stores the write as a "hint"<br>• When the target recovers, the hints are delivered to it<br>• Benefits: writes still succeed even when some replicas are down<br>• Like leaving a note on someone's desk when they're out of office<br>• Used by: Cassandra, DynamoDB, Riak |
| 77 | What is a Merkle tree and how is it used? | Merkle tree, integrity | • Hash tree where each leaf is a hash of a data block; each parent is hash of its children<br>• Compare two trees: if root hashes differ, drill down to find exactly which blocks differ<br>• Efficient: comparing 1M blocks requires only O(log N) hash comparisons<br>• Used for: anti-entropy repair (Cassandra), blockchain integrity, Git, BitTorrent |
| 78 | What is the consensus number problem? | consensus number, theory | • How many processes can reach consensus using a given synchronization primitive?<br>• Atomic register: consensus number 1 (only one process)<br>• Test-and-Set: consensus number 2 (two processes)<br>• Compare-and-Swap (CAS): consensus number ∞ (unlimited processes)<br>• CAS is the most powerful primitive for lock-free concurrent data structures |
| 79 | What is logical partitioning vs physical partitioning? | logical vs physical | • Logical: data logically separated but may share physical infrastructure (multi-tenant schemas)<br>• Physical: data on completely separate hardware (dedicated servers per tenant)<br>• Logical: cost-effective, shared resources, but noisy neighbor risk<br>• Physical: expensive, strong isolation, guaranteed performance<br>• Hybrid: logical for data, physical for compute (most SaaS companies) |
| 80 | Scenario: Design a globally consistent counter that handles 1M increments/second. | scenario, counter, global | • Challenge: single counter is bottleneck at 1M/s<br>• Solution: per-region local counters + periodic aggregation<br>• Each region: Redis INCR on local counter (microsecond latency)<br>• Background job: every 5 seconds, flush local counters to centralized store<br>• Read: sum of all regional counters (eventually consistent, ~5s lag)<br>• For exact count: CRDT PN-Counter across regions (no aggregation needed)<br>• Trade-off: exact real-time count requires global coordination = high latency |


---
<a id="scalability"></a>
# Section 2: Scalability

Scalability is a system's ability to handle growing workloads by adding resources. A scalable system maintains performance as users, data, or traffic increase.


## 2.1 Vertical vs Horizontal Scaling

| Aspect | Vertical Scaling (Scale Up) | Horizontal Scaling (Scale Out) |
|--------|----------------------------|-------------------------------|
| **How** | Add more power to existing machine (CPU, RAM, SSD) | Add more machines to the pool |
| **Cost** | Expensive (enterprise hardware) | Cheaper (commodity servers) |
| **Limit** | Physical limit (can't add infinite CPU) | Virtually unlimited |
| **Downtime** | Usually requires restart | No downtime (add nodes live) |
| **Complexity** | Simple (no distributed logic) | Complex (distributed state, coordination) |
| **Data** | All data on one machine | Data sharded/replicated across machines |
| **SPOF** | Single machine = single point of failure | Node failure handled by other nodes |
| **Example** | Upgrade from 4 cores → 64 cores | Add servers behind a load balancer |

### When to Use Which?

```
Traffic < 1000 req/s → Single powerful server (vertical scaling)
  ↓ hitting limits?
Traffic < 10K req/s → Read replicas + caching (mild horizontal)
  ↓ still growing?
Traffic < 100K req/s → Load balancer + multiple app servers + sharded DB
  ↓ still growing?
Traffic > 100K req/s → Full horizontal: microservices, CDN, edge caching, auto-scaling
```


## 2.2 Load Balancing

A load balancer distributes incoming requests across multiple servers to ensure no single server is overwhelmed.

### Load Balancing Algorithms

| Algorithm | How It Works | Best For |
|-----------|-------------|----------|
| **Round Robin** | Rotate through servers sequentially: A → B → C → A → ... | Equal-capacity servers, stateless |
| **Weighted Round Robin** | High-capacity servers get more requests (A:3, B:1) | Mixed-capacity servers |
| **Least Connections** | Send to server with fewest active connections | Long-lived connections (WebSocket) |
| **Least Response Time** | Send to server with fastest recent response | Performance-sensitive applications |
| **IP Hash** | Hash client IP to determine server (sticky) | Session affinity (stateful apps) |
| **Random** | Randomly pick a server | Simple, surprisingly effective at scale |
| **Consistent Hashing** | Hash-ring based server selection | Cache servers (memcached, Redis cluster) |

### Load Balancer Types

| Type | Layer | Operates On | Example |
|------|-------|-------------|---------|
| **L4 (Transport)** | TCP/UDP | IP address + port (no content inspection) | AWS NLB, HAProxy (TCP mode) |
| **L7 (Application)** | HTTP/HTTPS | URL path, headers, cookies (content-aware) | AWS ALB, Nginx, HAProxy (HTTP) |
| **DNS** | DNS | DNS resolution to different IPs | Route53, Cloudflare DNS |
| **Global** | Anycast/DNS | Geographic routing to nearest data center | Cloudflare, AWS Global Accelerator |

### L4 vs L7 Load Balancer Comparison

| Aspect | L4 (Transport) | L7 (Application) |
|--------|----------------|-------------------|
| **Speed** | Very fast (no content inspection) | Slower (parses HTTP headers) |
| **SSL termination** | No (passes encrypted traffic) | Yes (decrypts, inspects, re-encrypts) |
| **Routing** | IP + port based only | URL path, headers, cookies, host |
| **Use case** | TCP services, databases, very high throughput | Web apps, API routing, A/B testing |
| **Health checks** | TCP connect check | HTTP endpoint health check |

### Health Check Patterns

```
Load Balancer → sends periodic health check requests

Active Health Check:
  LB → GET /health → Server responds 200 OK → Server marked healthy
  LB → GET /health → Server responds 503/timeout → Server marked unhealthy
  LB → removed from rotation until healthy

Passive Health Check:
  LB monitors actual traffic
  If 5 errors in 30 seconds → Server marked unhealthy
  After 60 seconds → LB retries with real traffic
```


## 2.3 Auto Scaling

Auto scaling automatically adjusts the number of server instances based on current demand.

### Auto Scaling Strategies

| Strategy | Trigger | Response | Example |
|----------|---------|----------|---------|
| **Target Tracking** | Maintain metric at target value | Add/remove instances to hit target | Keep CPU at 60% |
| **Step Scaling** | Metric crosses threshold | Add fixed count of instances | CPU > 80% → add 2 instances |
| **Scheduled** | Time-based (cron) | Pre-scale for known traffic patterns | Scale up at 9 AM, down at 10 PM |
| **Predictive** | ML-based traffic prediction | Pre-scale before predicted spike | AWS Predictive Scaling |

### Key Metrics for Auto Scaling

| Metric | What It Measures | Scale Up When | Scale Down When |
|--------|-----------------|---------------|-----------------|
| **CPU utilization** | Compute load | > 70% average | < 30% average |
| **Memory utilization** | RAM usage | > 80% | < 40% |
| **Request count** | Traffic volume | > 1000 req/s per instance | < 200 req/s per instance |
| **Queue depth** | Pending work items | > 100 messages | < 10 messages |
| **Response time** | P95 latency | > 500ms | < 100ms |

### Cooldown & Stability

```
Scale-Up Event → Add Instance → Cooldown (300s) → Evaluate again
  Why cooldown?
    - New instance needs time to start and register
    - Metrics need time to reflect the new capacity
    - Prevents oscillation (add → metrics drop → remove → metrics spike → add → ...)

Scale-In Protection:
  - Minimum instances: never go below 2 (availability)
  - Scale-in cooldown: longer than scale-out (be aggressive adding, conservative removing)
  - Instance protection: mark instances handling long tasks as non-removable
```


## 2.4 Caching Strategies

Caching stores frequently accessed data in a faster storage layer to reduce latency and database load.

### Cache Placement Hierarchy

| Level | Location | Latency | Capacity | Example |
|-------|----------|---------|----------|---------|
| **L1 CPU Cache** | On-chip | ~1ns | 64KB-1MB | CPU register |
| **L2/L3 CPU Cache** | On-chip | ~10ns | 2-64MB | Hardware |
| **In-process** | Application memory | ~100ns | 100MB-1GB | Python dict, Java HashMap |
| **Distributed Cache** | Separate server | ~1ms | 10GB-1TB | Redis, Memcached |
| **CDN** | Edge servers worldwide | ~10ms | PBs | CloudFront, Cloudflare |
| **Database Cache** | DB buffer pool | ~2ms | 1-128GB | PostgreSQL shared_buffers |
| **Disk Cache** | OS file system | ~5ms | Available RAM | Page cache |

### Caching Patterns

| Pattern | Read | Write | Consistency | Best For |
|---------|------|-------|-------------|----------|
| **Cache-Aside** | App checks cache; on miss, reads DB and populates cache | App writes to DB only; invalidates/updates cache | Eventual (stale until TTL or invalidation) | General purpose, read-heavy |
| **Read-Through** | Cache fetches from DB on miss automatically | Same as cache-aside | Eventual | Simpler app code |
| **Write-Through** | Same as cache-aside | App writes to cache; cache writes to DB synchronously | Strong (cache always current) | Read-heavy with consistent data |
| **Write-Behind** | Same as cache-aside | App writes to cache; cache writes to DB asynchronously | Eventual (risk of data loss) | Write-heavy workloads |
| **Write-Around** | Same as cache-aside | App writes directly to DB; skips cache | Eventual | Write-heavy, read-infrequent data |

### Cache-Aside Pattern (Most Common)

```
Read:
  1. App checks cache → Key found? → Return cached value (HIT)
  2. Cache miss → App reads from database
  3. App writes result to cache with TTL
  4. Return result to client

Write:
  1. App writes to database
  2. App invalidates cache key (DELETE, not SET)
  Why DELETE not SET?
    - Race condition: two concurrent writes could set stale value
    - DELETE: next read will populate fresh value from DB

Cache Invalidation Rule:
  "There are only two hard things in CS: cache invalidation and naming things."
```


### Redis In-Depth

| Feature | Description |
|---------|-------------|
| **Data Structures** | Strings, Lists, Sets, Sorted Sets, Hashes, Streams, HyperLogLog |
| **Persistence** | RDB (point-in-time snapshots) + AOF (append-only file for durability) |
| **Pub/Sub** | Built-in publish/subscribe messaging |
| **TTL** | Per-key expiration for automatic cache cleanup |
| **Transactions** | MULTI/EXEC for atomic command batches |
| **Clustering** | Redis Cluster for horizontal scaling (16384 hash slots) |
| **Replication** | Async master-replica for read scaling and failover |

### Redis Use Cases

| Use Case | Redis Data Structure | Why Redis |
|----------|---------------------|-----------|
| Session store | Hash (user_id → session data) | Sub-ms reads, TTL for expiry |
| Rate limiter | Sorted Set or String with INCR | Atomic increment, TTL-based window |
| Leaderboard | Sorted Set (ZADD, ZRANGE) | O(log N) insert, O(log N + M) range query |
| Cache | String (key → JSON value) | Sub-ms GET/SET, TTL eviction |
| Distributed lock | String with NX + TTL (SET NX EX) | Atomic set-if-not-exists |
| Message queue | List (LPUSH/BRPOP) or Stream | Blocking pop, consumer groups |
| Real-time analytics | HyperLogLog (unique counts) | Probabilistic, 12KB per counter |

### CDN (Content Delivery Network)

| Concept | Description |
|---------|-------------|
| **What** | Network of edge servers distributed globally that cache content close to users |
| **How** | User request → nearest edge server → if cached, return immediately; if not, fetch from origin |
| **Benefits** | Lower latency (~10ms vs ~200ms), reduced origin load, DDoS protection |
| **Static content** | HTML, CSS, JS, images, videos (perfect for CDN — rarely changes) |
| **Dynamic content** | API responses, personalized pages (harder to cache but possible with Edge Functions) |
| **Invalidation** | TTL-based (30-3600 seconds), manual purge API, or versioned URLs (/style.v2.css) |
| **Providers** | CloudFront (AWS), Cloudflare, Akamai, Fastly |

```
Without CDN:
  User (Tokyo) → Origin Server (US-East) → 200ms latency

With CDN:
  User (Tokyo) → CDN Edge (Tokyo) → 10ms latency
  CDN Edge → Origin (only on cache miss) → populates cache for subsequent requests
```


## 2.5 Database Indexing

An index is a data structure that speeds up data retrieval at the cost of slower writes and extra storage.

### Index Types

| Index Type | Structure | Best For | Example |
|------------|-----------|----------|---------|
| **B-Tree** | Balanced tree (sorted keys) | Range queries, equality, ORDER BY | PostgreSQL default, MySQL InnoDB |
| **Hash** | Hash table | Equality lookups only (O(1)) | Memcached, DynamoDB partition key |
| **Bitmap** | Bit array per distinct value | Low-cardinality columns (gender, status) | Oracle, data warehouses |
| **GiST** | Generalized Search Tree | Geometric, full-text, custom types | PostgreSQL PostGIS |
| **GIN** | Inverted index | Full-text search, JSONB, arrays | PostgreSQL JSONB, Elasticsearch |

### B-Tree Index — How It Works

```
B-Tree for indexed column "age":

              [30 | 60]                    ← Root node
             /    |    \
     [10|20]   [40|50]   [70|80]          ← Internal nodes
      /  |  \   /  |  \   /  |  \
   [3] [15] [25] [35] [45] [55] [65] [75] [85]  ← Leaf nodes (→ row pointers)

Query: SELECT * FROM users WHERE age = 45
  1. Root: 45 > 30, 45 < 60 → middle child
  2. Internal: 45 > 40, 45 < 50 → right child
  3. Leaf: found 45 → return row pointer
  Steps: O(log N) — 3 hops instead of scanning millions of rows
```

### Index Best Practices

| Practice | Explanation |
|----------|-------------|
| Index columns in WHERE, JOIN, ORDER BY | These are the columns searched/sorted — biggest impact |
| Composite index column order matters | `(a, b, c)` helps queries on `a`, `a+b`, `a+b+c` — NOT `b` or `c` alone |
| Don't over-index | Each index slows INSERT/UPDATE (index must be maintained) |
| Covering index | Index includes all columns a query needs → no table lookup needed |
| Partial index | Index only a subset of rows (e.g., WHERE status = 'active') — smaller, faster |
| Monitor with EXPLAIN | Use `EXPLAIN ANALYZE` to verify index usage |


## 2.6 Read Replicas

| Concept | Description |
|---------|-------------|
| **What** | Copies of the primary database that serve read-only queries |
| **How** | Primary streams WAL/binlog to replicas; replicas apply changes |
| **Benefits** | Scale read throughput, offload analytics from primary, geographic proximity |
| **Replication lag** | Delay between primary write and replica receiving it (usually < 1 second) |
| **Consistency** | Eventually consistent (reads from replica may be slightly stale) |
| **Failover** | Replica can be promoted to primary if primary fails |

### Read Replica Architecture

```
Write queries → Primary DB (single)
                   ├── WAL stream → Replica 1 (US-East)  ← Read queries
                   ├── WAL stream → Replica 2 (EU-West)  ← Read queries
                   └── WAL stream → Replica 3 (AP-South) ← Read queries

Read routing: application or proxy (PgBouncer, ProxySQL) routes
  - Write queries → Primary
  - Read queries → nearest replica
```

## 2.7 Rate Limiting

Rate limiting controls the number of requests a client can make in a given time window.

### Rate Limiting Algorithms

| Algorithm | How It Works | Pros | Cons |
|-----------|-------------|------|------|
| **Fixed Window** | Count requests per time window (e.g., 100/minute) | Simple, low memory | Burst at window boundary (2x rate) |
| **Sliding Window Log** | Store timestamp of each request; count in last N seconds | Accurate, no boundary burst | High memory (store all timestamps) |
| **Sliding Window Counter** | Weighted combo of current + previous window counts | Low memory, reasonable accuracy | Approximation, not exact |
| **Token Bucket** | Bucket fills with tokens at fixed rate; each request consumes a token | Allows controlled bursts, smooth rate | Slightly more complex |
| **Leaky Bucket** | Requests queue in bucket; processed at fixed rate | Constant output rate, no bursts | Bursts cause queueing/drops |

### Token Bucket — Step by Step

```
Bucket capacity: 10 tokens
Refill rate: 2 tokens/second

Time 0s:  Bucket = 10 tokens
Request → consume 1 → Bucket = 9 ✓ (allowed)
Request → consume 1 → Bucket = 8 ✓
... (burst of 8 more requests)
Bucket = 0 tokens → next request REJECTED (429 Too Many Requests)

Time 1s: 2 tokens added → Bucket = 2
Request → consume 1 → Bucket = 1 ✓
Request → consume 1 → Bucket = 0 ✓
Request → REJECTED

Steady state: 2 requests/second sustained, burst up to 10
```

### Rate Limiting Headers

| Header | Meaning | Example |
|--------|---------|---------|
| `X-RateLimit-Limit` | Maximum requests per window | `100` |
| `X-RateLimit-Remaining` | Remaining requests in current window | `42` |
| `X-RateLimit-Reset` | Unix timestamp when window resets | `1678901234` |
| `Retry-After` | Seconds to wait before retrying (on 429) | `30` |


---
## Section 2 — Interview Questions: Scalability (80 Questions)


| # | Question | Keywords | Answer |
|---|----------|----------|--------|
| 81 | What is scalability? | scalability, definition | • System's ability to handle growing workload by adding resources<br>• Two dimensions: scaling compute (more requests) and scaling data (more storage)<br>• A scalable system maintains performance characteristics as load increases<br>• Measured by: throughput increase per resource added, latency under load<br>• Not just "handles more traffic" — also cost-efficient at scale |
| 82 | Vertical vs horizontal scaling — when to use each? | vertical vs horizontal | • Vertical: add CPU/RAM to existing server — simple, limited by hardware max<br>• Horizontal: add more servers — unlimited, but requires distributed architecture<br>• Start vertical: works until ~10K req/s on powerful hardware<br>• Switch to horizontal: when single machine can't handle load or for HA<br>• Most systems use both: powerful individual servers in a horizontally scaled pool |
| 83 | What is a load balancer and why is it needed? | load balancer, purpose | • Distributes incoming requests across multiple backend servers<br>• Prevents any single server from being overwhelmed<br>• Also handles: health checking, SSL termination, sticky sessions<br>• Without LB: single server is SPOF and throughput bottleneck<br>• Types: L4 (TCP), L7 (HTTP), DNS-based, global/anycast |
| 84 | Compare L4 and L7 load balancers. | L4 vs L7, comparison | • L4: operates at TCP level, fast, no content inspection, routes by IP/port<br>• L7: operates at HTTP level, can route by URL path, headers, cookies<br>• L7 can do: A/B testing, canary routing, WebSocket upgrade, SSL termination<br>• L4 is faster (no parsing), L7 is smarter (content-aware)<br>• Common: L4 in front for connection distribution → L7 behind for smart routing |
| 85 | How does round-robin load balancing work? | round robin, algorithm | • Requests distributed sequentially: Server A → B → C → A → B → C<br>• Simple, no state needed, works well with identical servers<br>• Problem: doesn't account for server load — busy server gets same traffic as idle one<br>• Weighted variant: Server A (weight 3) gets 3x traffic of Server C (weight 1)<br>• Best for: stateless services with similar hardware |
| 86 | How does least-connections load balancing work? | least connections, algorithm | • Routes new request to server with fewest active connections<br>• Better than round-robin when request processing times vary<br>• Accounts for: slow requests keeping connections open, servers with different capacity<br>• Requires: LB tracks connection count per server (slightly more overhead)<br>• Best for: long-lived connections (WebSocket, streaming) |
| 87 | What is sticky sessions and when is it needed? | sticky sessions, affinity | • Same client always routed to same server (session affinity)<br>• Implementation: cookie-based (LB sets cookie with server ID) or IP-hash<br>• Needed when: server stores session state in memory (not shared)<br>• Problem: if that server dies → user loses session<br>• Better approach: externalize session to Redis → no sticky sessions needed |
| 88 | What is SSL/TLS termination at the load balancer? | SSL, termination | • LB decrypts HTTPS traffic and forwards plain HTTP to backend servers<br>• Benefits: backend servers don't manage certificates, reduced CPU on backends<br>• Security concern: internal traffic is unencrypted (acceptable in trusted networks)<br>• Alternative: SSL passthrough (LB forwards encrypted traffic, backend decrypts)<br>• Re-encryption: LB decrypts → inspects → re-encrypts to backend (maximum security) |
| 89 | What is auto scaling and what triggers it? | auto scaling, triggers | • Automatically adding/removing server instances based on demand<br>• Scale-out triggers: CPU > 70%, request count > threshold, queue depth > limit<br>• Scale-in triggers: CPU < 30%, low request count (with cooldown period)<br>• Cooldown: prevents rapid add/remove oscillation (wait 5 min between actions)<br>• Types: reactive (respond to metrics), predictive (ML-based anticipation), scheduled |
| 90 | What is the difference between scaling compute and scaling storage? | compute vs storage | • Compute scaling: more servers to handle more requests (stateless, easy to scale)<br>• Storage scaling: more disk/database capacity for more data (stateful, harder to scale)<br>• Compute: add app servers behind LB, containers with orchestrator<br>• Storage: sharding, read replicas, distributed databases, object storage (S3)<br>• The hard part is always stateful scaling — design for stateless where possible |


| # | Question | Keywords | Answer |
|---|----------|----------|--------|
| 91 | What is caching and why is it the #1 performance optimization? | caching, importance | • Stores frequently accessed data in faster storage layer<br>• Cache hit: ~1ms (Redis) vs database query: ~10-50ms → 10-50x improvement<br>• Reduces database load: 80% cache hit rate means 80% fewer DB queries<br>• Use everywhere: browser cache, CDN, application cache, database bufferpool<br>• "The fastest query is the one you don't make" |
| 92 | Explain the cache-aside pattern. | cache-aside, pattern | • Read: check cache → miss → read DB → populate cache → return<br>• Write: write DB → invalidate cache (DELETE key, NOT update)<br>• Why invalidate not update: prevents race condition where concurrent writes set stale value<br>• Most common caching pattern — application controls cache logic<br>• Drawback: first request always misses cache (cold start) |
| 93 | What is write-through caching? | write-through, pattern | • Write goes to cache AND database synchronously<br>• Cache always has latest data → reads always consistent<br>• Higher write latency (must wait for both cache and DB)<br>• Good for: read-heavy workloads where consistency matters<br>• Cache populates on writes → no cold-start problem for written data |
| 94 | What is write-behind (write-back) caching? | write-behind, pattern | • Write goes to cache only; database updated asynchronously later<br>• Very fast writes (only cache latency)<br>• Risk: data loss if cache crashes before DB write<br>• Good for: write-heavy workloads where some data loss is acceptable<br>• Example: gaming leaderboard updates, analytics events |
| 95 | What is cache invalidation and why is it hard? | invalidation, cache, challenges | • Removing stale data from cache when underlying data changes<br>• "Two hard things in CS: cache invalidation and naming things"<br>• Strategies: TTL (time-based), event-based (invalidate on write), versioned keys<br>• Challenge: race conditions (write + read simultaneously), distributed invalidation (multiple cache nodes)<br>• Over-invalidation: too often → poor hit rate. Under-invalidation: stale data. |
| 96 | What is the thundering herd problem and how to prevent it? | thundering herd, cache | • Popular cache key expires → many concurrent requests hit database simultaneously<br>• DB overwhelmed → slow responses → cascade failure<br>• Solutions: jitter TTL (random variation prevents simultaneous expiry)<br>• Lock on cache miss: first request acquires lock, others wait for result<br>• Never-expire: background refresh before TTL |
| 97 | What is cache warming? | cache warming, preload | • Pre-populating cache with expected hot data before traffic arrives<br>• Run on: application startup, server restart, deployment<br>• Method: load top-N most accessed items from DB into cache<br>• Why: avoids cold-start cache misses and thundering herd on fresh cache<br>• Example: pre-cache product catalog, user profiles for active users |
| 98 | What is Redis and why is it so popular? | Redis, in-memory | • In-memory key-value store with rich data structures<br>• Sub-millisecond latency for most operations<br>• Data structures: strings, lists, sets, sorted sets, hashes, streams, HyperLogLog<br>• Persistence: RDB snapshots + AOF (doesn't lose data on restart)<br>• Use cases: caching, session store, rate limiter, leaderboard, pub/sub, queue |
| 99 | How does Redis Cluster work? | Redis Cluster, sharding | • 16384 hash slots distributed across master nodes<br>• Key assigned to slot: `CRC16(key) % 16384`<br>• Each master handles a range of slots; each has 1+ replica for failover<br>• Client-side routing: client knows which node handles which slot<br>• Adding a node: reassign slots from existing nodes to new node (live resharding) |
| 100 | What is a CDN and how does it work? | CDN, content delivery | • Network of edge servers worldwide that cache content close to users<br>• User request → CDN edge → if cached, return directly; if not, fetch from origin<br>• Best for: static assets (images, CSS, JS), video streaming<br>• Configuration: TTL, cache key, compression, versioned URLs for invalidation<br>• Providers: CloudFront, Cloudflare, Akamai, Fastly |
| 101 | What is database indexing? | indexing, database | • Data structure (usually B-tree) that speeds up read queries<br>• Without index: full table scan O(N) — scans every row<br>• With index: B-tree lookup O(log N) — skips directly to matching rows<br>• Trade-off: faster reads, slower writes (index must be updated on INSERT/UPDATE)<br>• Rule of thumb: index columns in WHERE, JOIN, ORDER BY clauses |
| 102 | What is a composite index and how does column order matter? | composite index, order | • Index on multiple columns: CREATE INDEX idx ON orders(user_id, created_at)<br>• Leftmost prefix rule: index (A,B,C) supports queries on A; A+B; A+B+C — NOT B or C alone<br>• Column order: put most selective (highest cardinality) first<br>• Equality before range: (status, date) supports WHERE status='active' AND date > '2024-01-01'<br>• This ordering uses both columns efficiently |
| 103 | What is a covering index? | covering index, query | • Index that contains ALL columns a query needs — no table lookup required<br>• PostgreSQL: `CREATE INDEX idx ON orders(user_id) INCLUDE (total, status)`<br>• Query: `SELECT total, status FROM orders WHERE user_id = 123` → reads ONLY from index<br>• Much faster: index is smaller than full table, fewer disk reads<br>• Use for: high-frequency queries with specific column access patterns |
| 104 | What are read replicas? | read replicas, scaling | • Copies of primary database that serve read-only queries<br>• Primary handles writes; replicas handle reads → distributes read load<br>• Replication: WAL streaming (PostgreSQL), binlog (MySQL)<br>• Eventually consistent: replica may be 1-100ms behind primary<br>• Common: 1 primary + 2-3 read replicas for typical workloads |
| 105 | What is connection pooling? | connection pooling, database | • Reuse existing database connections instead of creating new ones per request<br>• New connection overhead: TCP handshake + TLS + authentication = ~50-100ms<br>• Pool: pre-established connections checked out/returned by application<br>• Tools: PgBouncer (PostgreSQL), ProxySQL (MySQL), HikariCP (Java)<br>• Configuration: pool_size (min/max connections), timeout, max lifetime |
| 106 | What is rate limiting and why is it needed? | rate limiting, purpose | • Controls number of requests a client can make in a time window<br>• Prevents: abuse, DoS attacks, resource exhaustion, cost overruns<br>• Applied at: API gateway, reverse proxy, application layer<br>• HTTP response on limit: 429 Too Many Requests<br>• Headers: X-RateLimit-Limit, X-RateLimit-Remaining, Retry-After |
| 107 | Compare token bucket and sliding window algorithms. | token bucket vs sliding window | • Token bucket: steady refill rate (e.g., 10 tokens/sec), allows burst up to bucket capacity<br>• Sliding window: strict count in rolling time window (no burst allowed)<br>• Token bucket: better for APIs that allow short bursts (user experience)<br>• Sliding window: better for strict rate enforcement (security, billing)<br>• Token bucket used by: AWS API Gateway; sliding window: rate-limiting middleware |
| 108 | How do you rate limit in a distributed system? | distributed rate limiting | • Challenge: multiple app servers → each only sees partial traffic<br>• Solution 1: centralized counter in Redis (INCR with TTL)<br>• Solution 2: sticky sessions (each user always hits same server → local counting)<br>• Solution 3: distributed rate limiter (each server gets quota = total limit / num servers)<br>• Redis approach is most common (accurate, sub-ms, scales with Redis Cluster) |
| 109 | What is database query optimization? | query optimization, performance | • EXPLAIN ANALYZE: shows query execution plan (index scan vs sequential scan)<br>• Avoid SELECT *: SELECT only needed columns (less data transfer)<br>• Avoid N+1 queries: use JOIN or batch loading instead of per-row queries<br>• Pagination: LIMIT/OFFSET or keyset pagination for large result sets<br>• Connection pooling: reuse connections instead of creating per request |
| 110 | What is the N+1 query problem? | N+1, query, problem | • Fetching list of N items, then making 1 query per item to load related data<br>• Example: 100 orders, then 100 queries to get each customer = 101 queries<br>• Fix: JOIN query (1 query), eager loading (ORM-level), batch loading (WHERE IN)<br>• ORM specific: Django select_related(), SQLAlchemy joinedload()<br>• Impact: 101 queries × 5ms = 500ms vs 1 query × 10ms = 10ms |


| # | Question | Keywords | Answer |
|---|----------|----------|--------|
| 111 | What is database connection pooling? | connection pooling, performance | • Pre-established pool of database connections reused across requests<br>• Without pool: each request opens new TCP connection → ~50ms overhead per request<br>• With pool: borrow connection from pool → ~0ms overhead → return when done<br>• Configuration: min_pool_size (always ready), max_pool_size (limit), timeout<br>• Tools: PgBouncer, ProxySQL, HikariCP |
| 112 | What is the difference between caching and buffering? | caching vs buffering | • Caching: store result for REUSE (same data accessed multiple times)<br>• Buffering: temporarily hold data being TRANSFERRED (smooths out speed differences)<br>• Cache hit: data already computed, return immediately<br>• Buffer: batch small writes into one large write (reduce write amplification)<br>• Both improve performance but for different reasons |
| 113 | What is materialized view and how does it help performance? | materialized view, database | • Pre-computed query result stored as a table (not recomputed on each read)<br>• Regular view: runs query every time → slow for complex aggregations<br>• Materialized view: cached result → fast reads, but must be refreshed periodically<br>• Refresh: REFRESH MATERIALIZED VIEW (full) or CONCURRENTLY (incremental, no lock)<br>• Use case: dashboard aggregations, reporting queries, denormalized read models |
| 114 | What is CQRS and how does it help scalability? | CQRS, scalability | • Command Query Responsibility Segregation — separate read and write models<br>• Write model: optimized for data integrity (normalized, transactional)<br>• Read model: optimized for query performance (denormalized, pre-aggregated)<br>• Scale independently: more read replicas for queries, single primary for writes<br>• Commonly paired with event sourcing and materialized views |
| 115 | What is horizontal partitioning vs vertical partitioning? | horizontal vs vertical partition | • Horizontal: split ROWS across tables/shards (each has all columns, subset of rows)<br>• Vertical: split COLUMNS across tables (each has all rows, subset of columns)<br>• Horizontal: scale beyond single machine capacity (sharding)<br>• Vertical: separate hot data from cold data (profile image blob on separate storage)<br>• Can combine both for maximum scalability |
| 116 | What is data denormalization for performance? | denormalization, performance | • Storing redundant data to eliminate expensive JOIN queries<br>• Example: store author_name in blog post (avoid user table JOIN on every read)<br>• Trade-off: faster reads, complicated writes (must update all copies)<br>• When: read >> write, and JOIN is slow at scale<br>• NoSQL databases often require denormalization (no JOIN support) |
| 117 | What is connection draining? | connection draining, graceful | • Allowing in-flight requests to complete before removing a server from the pool<br>• When: auto scaling down, deployment, maintenance<br>• Without draining: active requests terminated → errors for users<br>• With draining: LB stops sending NEW requests, waits for existing to finish<br>• Timeout: if connections don't close in X seconds, force terminate |
| 118 | What is the C10K problem? | C10K, concurrent connections | • Challenge of handling 10,000 concurrent connections on a single server<br>• Traditional: one thread per connection → 10K threads = memory exhaustion<br>• Solution: event-driven I/O (epoll, kqueue, IOCP) — single thread handles thousands of connections<br>• Modern frameworks: Node.js, Nginx, Go, AsyncIO solve C10K easily<br>• Modern challenge: C10M (10 million connections) — kernel bypass, DPDK |
| 119 | What is a reverse proxy? | reverse proxy, architecture | • Server that sits between clients and backend servers<br>• Client thinks it's talking to origin, but proxy forwards requests<br>• Functions: load balancing, SSL termination, caching, compression, rate limiting<br>• Forward proxy: client-side (client uses proxy to access internet) — VPNs, corporate proxies<br>• Reverse proxy: server-side (protects and routes to backends) — Nginx, HAProxy, Traefik |
| 120 | How do you handle database connection exhaustion? | connection exhaustion, database | • Symptom: new requests timeout waiting for database connection<br>• Cause: too many concurrent requests, slow queries holding connections, connection leaks<br>• Fix 1: increase connection pool size (but DB has max connections too)<br>• Fix 2: optimize slow queries (reduce connection hold time)<br>• Fix 3: PgBouncer for connection multiplexing (fewer DB connections, more app connections)<br>• Fix 4: add read replicas (distribute load) |
| 121 | What is graceful degradation vs graceful shutdown? | degradation vs shutdown | • Graceful degradation: when component fails, system continues with reduced functionality<br>  Example: cache down → serve from DB (slower but works)<br>• Graceful shutdown: when stopping a server, finish existing requests before terminating<br>  Example: SIGTERM → stop accepting new requests → complete in-flight → exit<br>• Both are essential for production systems |
| 122 | How do you scale a WebSocket server? | WebSocket, scaling | • Challenge: WebSocket connections are stateful (not like stateless HTTP)<br>• Sticky sessions: route same user to same server (simple, but single server failure loses connections)<br>• Pub/Sub backplane: Redis Pub/Sub or Kafka so any server can broadcast to connected clients<br>• Reconnection: client auto-reconnects to any server → re-subscribes to channels<br>• Services: use managed WebSocket services (AWS API Gateway WebSocket, Pusher) |
| 123 | What is request collapsing/coalescing? | request collapsing, thundering herd | • Multiple identical requests merged into one backend call<br>• First request triggers DB fetch; identical concurrent requests wait for that result<br>• All receive the same response — one DB query instead of hundreds<br>• Go: singleflight package; Nginx: proxy_cache_lock<br>• Eliminates thundering herd for the same key |
| 124 | What is edge computing? | edge computing, latency | • Processing data at the network edge (close to users) instead of central cloud<br>• CDN with compute: CloudFront Functions, Cloudflare Workers, Vercel Edge Functions<br>• Use cases: auth validation, A/B testing, header manipulation, personalization<br>• Latency: ~5ms (edge) vs ~100ms (central cloud)<br>• Limitation: limited compute power, no persistent storage at edge |
| 125 | How do you design for stateless services? | stateless, design | • No server-side session state — every request contains all needed info<br>• Authentication: JWT (token contains user info) instead of server-side sessions<br>• Session data: externalize to Redis (all servers can access)<br>• File uploads: store in S3 (not local disk)<br>• Benefits: any server can handle any request → easy horizontal scaling |
| 126 | What are the key metrics for measuring scalability? | metrics, scalability | • Throughput: requests per second (RPS) at acceptable latency<br>• Latency: P50, P95, P99 response times under load<br>• Error rate: percentage of failed requests under load<br>• Resource utilization: CPU, memory, disk I/O, network bandwidth<br>• Cost efficiency: cost per request or cost per user (linear growth = good, exponential = bad) |
| 127 | What is the difference between latency and throughput? | latency vs throughput | • Latency: time for a single request (measured in ms)<br>• Throughput: total requests handled per second (measured in RPS)<br>• Improving latency: faster processing, caching, reducing hops<br>• Improving throughput: more servers, parallel processing, batching<br>• Often in tension: optimizing for throughput (batching) increases individual latency |
| 128 | What is Amdahl's Law? | Amdahl, parallel, scaling | • Speedup limited by the serial portion of the workload<br>• Formula: Speedup = 1 / (S + P/N) where S = serial fraction, P = parallel fraction, N = processors<br>• If 10% of work is serial → max speedup = 10x regardless of how many processors you add<br>• Implication: identify and optimize serial bottlenecks before adding more servers<br>• Example: if DB is the serial bottleneck, adding more app servers won't help |
| 129 | What is prefetching? | prefetching, optimization | • Loading data before it's actually requested (prediction-based)<br>• Example: user viewing page 1 → prefetch page 2 content<br>• Browser: `<link rel="prefetch" href="/next-page">`<br>• Database: preload user's recent data on login<br>• Risk: wasted resources if prediction is wrong (bandwidth, memory) |
| 130 | Scenario: Your API handles 100 req/s but needs to handle 100K req/s. Design the scaling strategy. | scenario, scaling, 1000x | • Layer 1: CDN for all static assets (removes 60% of traffic)<br>• Layer 2: API response caching in Redis (removes 30% of remaining)<br>• Layer 3: horizontal scaling — 50+ app servers behind load balancer<br>• Layer 4: database read replicas (5-10) for read-heavy queries<br>• Layer 5: database sharding for write scaling<br>• Layer 6: async processing — queue heavy work (email, reports) to Kafka/SQS<br>• Layer 7: connection pooling (PgBouncer) to manage DB connections<br>• Result: ~1000x throughput via caching + horizontal scaling |


---
<a id="microservices"></a>
# Section 3: Microservices Architecture

Microservices architecture decomposes a large application into small, independently deployable services, each owning its own data and business logic.


## 3.1 Monolith vs Microservices

### Monolithic Architecture

```
┌──────────────────────────────────────────┐
│              Monolith Application          │
│  ┌──────┐ ┌──────┐ ┌──────┐ ┌──────────┐ │
│  │ Auth │ │Users │ │Orders│ │ Payments │ │
│  └──┬───┘ └──┬───┘ └──┬───┘ └────┬─────┘ │
│     └────────┴────────┴──────────┘        │
│              Shared Database               │
└──────────────────────────────────────────┘
```

### Microservices Architecture

```
┌──────────┐  ┌──────────┐  ┌──────────┐  ┌──────────┐
│Auth Svc  │  │User Svc  │  │Order Svc │  │Payment   │
│  (own DB)│  │  (own DB)│  │  (own DB)│  │  (own DB)│
└────┬─────┘  └────┬─────┘  └────┬─────┘  └────┬─────┘
     └──────────┬──┴──────────┬──┴──────────────┘
           API Gateway / Message Broker
```

| Aspect | Monolith | Microservices |
|--------|----------|---------------|
| **Deployment** | Deploy entire app for any change | Deploy individual services independently |
| **Scaling** | Scale entire app (even unused parts) | Scale only the service that needs it |
| **Technology** | Single tech stack | Polyglot (different lang/DB per service) |
| **Team** | One team, one codebase | Small teams own individual services |
| **Data** | Single shared database | Each service owns its data (Database per Service) |
| **Complexity** | Simple at first, complex at scale | Complex from start, manageable at scale |
| **Testing** | Easy integration testing | Complex integration testing across services |
| **Latency** | In-process function calls (~ns) | Network calls between services (~ms) |
| **Transactions** | ACID across entire app | Distributed transactions (Saga pattern) |
| **Failure** | Single failure can crash everything | Failure isolated to one service |

### When NOT to Use Microservices

| Scenario | Why Monolith is Better |
|----------|----------------------|
| Small team (< 5 engineers) | Distributed overhead not justified |
| Early-stage startup | Need to iterate fast, not manage infra |
| Simple domain | Doesn't benefit from decomposition |
| No clear service boundaries | Will create a "distributed monolith" |
| No DevOps maturity | Need CI/CD, monitoring, container orchestration first |


## 3.2 Service Discovery

In microservices, service instances are dynamic (auto-scaling, deployments, failures). Service discovery helps services **find each other**.

### Service Discovery Patterns

| Pattern | How It Works | Example |
|---------|-------------|---------|
| **Client-Side Discovery** | Client queries registry, picks an instance, calls directly | Netflix Eureka + Ribbon |
| **Server-Side Discovery** | Client calls load balancer, LB queries registry and forwards | AWS ALB, Kubernetes Services |
| **DNS-Based** | Service registers DNS name, DNS resolves to instance IPs | Consul DNS, AWS Cloud Map |
| **Sidecar Proxy** | Proxy (sidecar) handles discovery transparently | Envoy (Istio service mesh) |

### Service Registry

```
Service Registry (Consul/Eureka/etcd):
  ┌─────────────────────────────────────────────────┐
  │  Service Name     │  Instance        │ Health    │
  │───────────────────┼──────────────────┼───────────│
  │  user-service     │  10.0.1.5:8080   │ ✅ healthy │
  │  user-service     │  10.0.1.6:8080   │ ✅ healthy │
  │  user-service     │  10.0.1.7:8080   │ ❌ unhealthy│
  │  order-service    │  10.0.2.3:9090   │ ✅ healthy │
  │  payment-service  │  10.0.3.1:7070   │ ✅ healthy │
  └─────────────────────────────────────────────────┘

Registration: on startup, service registers itself
Health check: periodic heartbeat (TTL) or HTTP check
Deregistration: on shutdown, or when health check fails
```


## 3.3 API Gateway

An API gateway is the single entry point for all client requests, routing them to appropriate microservices.

### API Gateway Responsibilities

| Function | Description |
|----------|-------------|
| **Request routing** | Route `/users/*` to User Service, `/orders/*` to Order Service |
| **Authentication** | Validate JWT/API key before forwarding to service |
| **Rate limiting** | Enforce per-client or per-endpoint rate limits |
| **SSL termination** | Handle HTTPS at gateway level |
| **Request/Response transformation** | Aggregate responses from multiple services |
| **Caching** | Cache frequent responses at gateway level |
| **Circuit breaking** | Prevent cascading failures |
| **Logging/Monitoring** | Centralized request logging and metrics |

### API Gateway Products

| Gateway | Type | Key Feature |
|---------|------|-------------|
| **Kong** | Open-source / Enterprise | Plugin ecosystem, Lua scripting |
| **AWS API Gateway** | Managed | Serverless, Lambda integration |
| **Nginx** | Open-source | High performance, configuration-based |
| **Traefik** | Open-source | Auto-discovery, Docker/K8s native |
| **Ambassador** | Open-source | Kubernetes-native, Envoy-based |

### BFF (Backend For Frontend)

```
Mobile App → Mobile BFF Gateway → optimized for mobile (smaller payloads)
Web App   → Web BFF Gateway    → optimized for web (richer payloads)
3rd Party → Public API Gateway → versioned, rate-limited, documented

Each BFF:
  - Aggregates multiple microservice calls into one
  - Transforms response format for specific client needs
  - Handles client-specific authentication
```


## 3.4 Inter-Service Communication

### Communication Patterns Comparison

| Pattern | Type | Protocol | When To Use |
|---------|------|----------|-------------|
| **REST** | Sync | HTTP/1.1, HTTP/2 | CRUD operations, simple services |
| **gRPC** | Sync | HTTP/2 + Protobuf | High performance, low latency, strong typing |
| **GraphQL** | Sync | HTTP | Flexible queries, BFF aggregation |
| **Message Queue** | Async | AMQP, custom | Background tasks, decoupled services |
| **Event Streaming** | Async | Kafka protocol | Event sourcing, real-time data pipelines |

### REST vs gRPC

| Aspect | REST | gRPC |
|--------|------|------|
| **Protocol** | HTTP/1.1 (or HTTP/2) | HTTP/2 only |
| **Payload** | JSON (text, human-readable) | Protocol Buffers (binary, compact) |
| **Speed** | Slower (text parsing) | 7-10x faster (binary serialization) |
| **Streaming** | SSE or WebSocket (separate) | Native bidirectional streaming |
| **Contract** | OpenAPI/Swagger (optional) | .proto file (required, strongly typed) |
| **Browser support** | Full native | gRPC-Web required (extra layer) |
| **Debugging** | Easy (curl, Postman) | Harder (need gRPC tools) |
| **Code generation** | Optional (OpenAPI codegen) | Built-in (protoc generates client/server) |

### gRPC Communication Types

| Type | Flow | Use Case |
|------|------|----------|
| **Unary** | Client sends one request → Server sends one response | Simple request-response (like REST) |
| **Server Streaming** | Client sends one → Server sends stream of responses | Real-time updates, log streaming |
| **Client Streaming** | Client sends stream → Server sends one response | File upload, IoT sensor data |
| **Bidirectional Streaming** | Both send streams simultaneously | Chat, real-time collaboration |


## 3.5 Circuit Breaker Pattern

Prevents cascading failures by stopping requests to a failing downstream service.

### Circuit Breaker States

```
                    ┌──────────┐
        success     │  CLOSED  │  normal operation
        ←──────────│ (allow)  │────────────→
                    └────┬─────┘    N failures
                         │          in window
                         ▼
                    ┌──────────┐
     timeout        │   OPEN   │  all requests fail-fast
     (30 sec)  ────│ (reject) │
                    └────┬─────┘
                         │
                         ▼
                    ┌──────────┐
                    │HALF-OPEN │  allow limited requests
                    │  (test)  │
                    └────┬─────┘
                    /           \
              success          failure
                ↓                ↓
            CLOSED            OPEN
```

### Circuit Breaker Configuration

| Parameter | Typical Value | Purpose |
|-----------|--------------|---------|
| **Failure threshold** | 5 failures | Number of failures before opening circuit |
| **Window size** | 60 seconds | Time window to count failures |
| **Open duration** | 30 seconds | How long to stay OPEN before testing |
| **Half-open requests** | 1-3 | How many test requests in HALF-OPEN |
| **Success threshold** | 3 successes | Number of successes in HALF-OPEN to close |

## 3.6 Saga Pattern (Detailed)

### Orchestration vs Choreography

| Aspect | Choreography | Orchestration |
|--------|-------------|---------------|
| **Coordination** | No coordinator — events trigger next step | Central orchestrator controls sequence |
| **Coupling** | Loosely coupled services | Orchestrator coupled to all participants |
| **Visibility** | Hard to track saga status | Easy — orchestrator knows full state |
| **Complexity** | Simple for 2-3 steps; messy for 5+ | Scales well to complex multi-step sagas |
| **Failure** | Each service must know compensating actions | Orchestrator manages compensations |

### Saga Example: Order Processing

```
Forward transactions:
  1. OrderService → CreateOrder (status: PENDING)
  2. PaymentService → ChargeCard ($100)
  3. InventoryService → ReserveStock (item_id, qty)
  4. ShippingService → CreateShipment

Compensating transactions (if InventoryService fails):
  3c. InventoryService → (no-op, it failed)
  2c. PaymentService → RefundCard ($100)
  1c. OrderService → CancelOrder (status: CANCELLED)
```


## 3.7 Observability (Logs, Metrics, Tracing)

### Three Pillars of Observability

| Pillar | What | Why | Tools |
|--------|------|-----|-------|
| **Logs** | Timestamped text records of events | Debug specific errors, audit trail | ELK Stack, Loki, Splunk |
| **Metrics** | Numerical measurements over time | Detect trends, alerting, SLOs | Prometheus, Datadog, Grafana |
| **Traces** | Request flow across services | Find latency bottlenecks, dependencies | Jaeger, Zipkin, OpenTelemetry |

### Distributed Tracing Concepts

| Concept | Description |
|---------|-------------|
| **Trace** | End-to-end journey of a request through all services |
| **Span** | A single operation within a trace (one service call) |
| **Trace ID** | Unique identifier propagated across all services (via HTTP headers) |
| **Span ID** | Unique identifier for each span within a trace |
| **Parent Span** | The calling span (establishes parent-child hierarchy) |

### Trace Propagation

```
Client → API Gateway (Span 1, Trace: abc-123)
            → User Service (Span 2, Parent: Span 1)
            → Order Service (Span 3, Parent: Span 1)
                → Payment Service (Span 4, Parent: Span 3)
                → Inventory Service (Span 5, Parent: Span 3)

HTTP Header: traceparent: 00-abc123-span1-01
  Each service extracts trace ID, creates child span, forwards header

Waterfall View:
  ├── API Gateway ────────────────────────── 250ms
  │   ├── User Service ─────── 30ms
  │   └── Order Service ─────────────── 200ms
  │       ├── Payment Service ──── 50ms
  │       └── Inventory Service ── 80ms
```

### Key Metrics to Monitor

| Metric | Type | Example |
|--------|------|---------|
| **RED** | Rate, Errors, Duration | Request rate, error %, P99 latency |
| **USE** | Utilization, Saturation, Errors | CPU %, queue depth, error count |
| **SLI** | Service Level Indicator | 99% of requests < 200ms |
| **SLO** | Service Level Objective | 99.9% availability per month |
| **SLA** | Service Level Agreement | Contractual guarantee (uptime, penalties) |


---
## Section 3 — Interview Questions: Microservices Architecture (70 Questions)


| # | Question | Keywords | Answer |
|---|----------|----------|--------|
| 131 | What is a microservice? | microservice, definition | • Small, independently deployable service that does one thing well<br>• Owns its own data (Database per Service pattern)<br>• Communicates via APIs (REST, gRPC, events)<br>• Can be developed, deployed, and scaled independently<br>• Organized around business capability, not technical layer |
| 132 | What are the key benefits of microservices? | benefits, microservices | • Independent deployment: change one service without redeploying all<br>• Technology freedom: use best language/DB per service<br>• Team autonomy: small teams own services end-to-end<br>• Scalability: scale only services that need it<br>• Fault isolation: failure in one service doesn't crash others |
| 133 | What are the main challenges of microservices? | challenges, drawbacks | • Distributed system complexity: network failures, latency, partial failures<br>• Data consistency: no ACID across services (need Saga, eventual consistency)<br>• Operational overhead: monitoring, logging, tracing across many services<br>• Testing complexity: integration tests must cover cross-service interactions<br>• Network overhead: every call is an HTTP/gRPC request (not in-process function call) |
| 134 | When should you NOT use microservices? | anti-pattern, wrong use | • Small team (< 5 people) — overhead exceeds benefits<br>• Simple domain — doesn't justify decomposition<br>• Early startup — need to iterate quickly, not manage distributed infra<br>• No clear boundaries — will create "distributed monolith"<br>• No DevOps maturity — need CI/CD, monitoring, containers first |
| 135 | What is a distributed monolith? | distributed monolith, anti-pattern | • Microservices that are tightly coupled — can't deploy independently<br>• Shared database: services access each other's tables directly<br>• Synchronous chain: Service A → B → C → D (all must be up)<br>• Coordinated deployment: change in A requires redeploying B and C<br>• Worst of both worlds: complexity of distributed + rigidity of monolith |
| 136 | What is the Database per Service pattern? | database per service | • Each microservice has its own database (not shared with others)<br>• Ensures loose coupling: changing one service's schema doesn't affect others<br>• Cross-service data access: via APIs, not direct database queries<br>• Trade-off: can't do simple JOINs across services<br>• Alternatives: shared database (simpler but coupled), data duplication (via events) |
| 137 | What is the Strangler Fig pattern? | strangler fig, migration | • Gradually migrate from monolith to microservices<br>• Route: new feature requests go to new microservice; old features stay in monolith<br>• Over time: migrate more features until monolith is empty<br>• Named after: strangler fig tree that grows around and replaces host tree<br>• Benefits: incremental migration, no "big bang" rewrite, low risk |
| 138 | What is service discovery? | service discovery | • Mechanism for services to find network locations of other services<br>• Needed because: instances are dynamic (auto-scaling, deployments, failures)<br>• Types: client-side (client queries registry), server-side (LB queries registry)<br>• Implementations: Consul, Eureka, etcd, Kubernetes DNS<br>• Health checks: registry removes unhealthy instances |
| 139 | What is an API gateway? | API gateway, entry | • Single entry point for all client requests to microservices<br>• Handles: routing, auth, rate limiting, SSL, response aggregation<br>• Avoids: client knowing about individual service locations<br>• Products: Kong, AWS API Gateway, Nginx, Traefik<br>• Variant: BFF (Backend For Frontend) — separate gateway per client type |
| 140 | What is the BFF (Backend For Frontend) pattern? | BFF, frontend backend | • Separate API gateway/backend for each client type (mobile, web, third-party)<br>• Each BFF optimized for its client: mobile gets smaller payloads, web gets richer data<br>• BFF aggregates multiple microservice calls into one client-friendly response<br>• Reduces coupling: changes to mobile API don't affect web API<br>• Trade-off: code duplication across BFFs for shared logic |
| 141 | Compare REST and gRPC for inter-service communication. | REST vs gRPC | • REST: JSON over HTTP — human-readable, wide tooling, browser-native<br>• gRPC: Protocol Buffers over HTTP/2 — binary, 7-10x faster, strongly typed<br>• REST: better for public APIs and simple CRUD<br>• gRPC: better for internal high-frequency service-to-service calls<br>• gRPC: native streaming support (server, client, bidirectional) |
| 142 | What is a circuit breaker and why use it? | circuit breaker, pattern | • Stops making requests to a failing service (prevent cascading failure)<br>• States: CLOSED (normal) → OPEN (failing, reject all) → HALF-OPEN (test recovery)<br>• Without circuit breaker: slow failing service causes all callers to time out → domino effect<br>• With circuit breaker: fail-fast (immediate error) → callers don't waste resources waiting<br>• Libraries: resilience4j (Java), Polly (.NET), pybreaker (Python) |
| 143 | What is the Saga pattern and when do you use it? | Saga, distributed transaction | • Manages distributed transactions across microservices using compensating transactions<br>• Instead of 2PC (which blocks): each service has forward + compensation actions<br>• If step N fails: compensating transactions undo steps N-1, N-2, ..., 1<br>• Two styles: choreography (event-driven) or orchestration (central coordinator)<br>• Use when: multi-service transaction needed (order → payment → inventory) |
| 144 | What is the difference between choreography and orchestration? | choreography vs orchestration | • Choreography: each service emits events; no central coordination<br>• Orchestration: central service controls the saga execution sequence<br>• Choreography: loosely coupled but hard to track the overall flow<br>• Orchestration: clear visibility but adds coupling to the orchestrator<br>• Small sagas (2-3 steps): choreography. Complex sagas (5+): orchestration |
| 145 | What are the three pillars of observability? | observability, pillars | • Logs: timestamped text records of events — debug specific issues<br>• Metrics: numerical measurements over time — detect trends, alerting<br>• Traces: request flow across services — find latency bottlenecks<br>• Together: metrics spot the problem, traces find the service, logs explain the root cause<br>• Tools: ELK/Loki (logs), Prometheus/Grafana (metrics), Jaeger/Zipkin (traces) |
| 146 | What is distributed tracing? | distributed tracing | • Following a single request's journey across multiple microservices<br>• Trace ID propagated via HTTP headers to all services<br>• Each service creates a span (timed operation) within the trace<br>• Visualization: waterfall diagram showing duration per service<br>• Critical for: finding latency bottlenecks, understanding service dependencies |
| 147 | What is the sidecar pattern? | sidecar, pattern | • Deploy a helper container alongside each service (same pod in Kubernetes)<br>• Sidecar handles cross-cutting concerns: logging, monitoring, mTLS, service mesh proxy<br>• Main service communicates with sidecar via localhost<br>• Benefits: language-agnostic infra, consistent behavior across services<br>• Example: Envoy sidecar proxy in Istio service mesh |
| 148 | What is a service mesh? | service mesh | • Infrastructure layer that handles all service-to-service communication<br>• Data plane: sidecar proxies (Envoy) intercept traffic between services<br>• Control plane: manages proxy configuration (Istio, Linkerd)<br>• Features: mTLS, load balancing, retries, circuit breaking, traffic shaping, observability<br>• When to use: large microservices deployments (50+ services) |
| 149 | What is eventual consistency in microservices? | eventual consistency, micro | • After a change in one service, other services will eventually reflect that change<br>• Propagation via events: Service A writes → publishes event → Service B consumes → updates<br>• Trade-off: faster writes, temporary inconsistency between services<br>• Example: order placed in Order Service → inventory updated 100ms later in Inventory Service<br>• Compensating actions handle edge cases (order placed but inventory already sold) |
| 150 | What is the Outbox Pattern in microservices? | outbox, pattern, microservices | • Reliably publish events when database state changes<br>• Approach: write data + event to same database transaction (outbox table)<br>• Separate process polls outbox → publishes to message broker (Kafka)<br>• Guarantees: event published iff transaction committed (no dual-write problem)<br>• Alternative: Change Data Capture (Debezium reads DB transaction log directly) |


| # | Question | Keywords | Answer |
|---|----------|----------|--------|
| 151 | What is the difference between SLI, SLO, and SLA? | SLI, SLO, SLA | • SLI (Service Level Indicator): actual measurement (e.g., 99.85% availability this month)<br>• SLO (Service Level Objective): target goal (e.g., 99.9% availability target)<br>• SLA (Service Level Agreement): contract with penalties (e.g., refund if SLO breached)<br>• SLI is what you measure, SLO is what you aim for, SLA is what you promise customers<br>• SLA usually less strict than SLO (buffer for error) |
| 152 | What is the Bulkhead pattern? | bulkhead, isolation | • Isolate resources for different services/features into separate pools<br>• Named after ship bulkheads (compartments that contain flooding)<br>• Example: separate thread pools for payment calls vs user calls<br>• If payment service slows down: only payment thread pool is exhausted, user calls unaffected<br>• Implementation: separate connection pools, thread pools, or dedicated instances per dependency |
| 153 | What is the Retry pattern and when is it dangerous? | retry, pattern | • Automatically retry failed requests (useful for transient failures)<br>• Exponential backoff: wait 1s, 2s, 4s, 8s between retries (avoid thundering herd)<br>• Jitter: add random delay to backoff (prevents synchronized retries)<br>• Dangerous when: downstream is overloaded (retries add more load = amplify failure)<br>• Must pair with: circuit breaker, retry budget, idempotency |
| 154 | What is a retry budget? | retry budget, rate | • Limits total retries across all clients to prevent cascading failure<br>• Example: allow at most 10% of requests to be retries<br>• Without budget: 100 requests fail → 100 retries → 100 more retries → 10x traffic to failing service<br>• Implementation: track retry rate; when budget exhausted, fail immediately (don't retry)<br>• First adopted by Google SRE |
| 155 | What is the Ambassador pattern? | ambassador, pattern | • Proxy running alongside a service that handles connection complexity<br>• Example: ambassador handles connection pooling, TLS, circuit breaking to a database<br>• Similar to sidecar but specifically for outbound connections<br>• Service calls ambassador on localhost → ambassador handles external connection details<br>• Benefits: service code is simpler, connection logic centralized |
| 156 | What is data replication between microservices? | data replication, micro | • Copying data from one service's database to another (via events)<br>• Example: User Service publishes "UserCreated" → Order Service stores user name locally<br>• Why: Order Service can query user name without calling User Service API (faster, decoupled)<br>• Trade-off: data duplication, eventual consistency, more complex writes<br>• CQRS read model: dedicated read projection populated by events |
| 157 | What is a health check endpoint? | health check, endpoint | • HTTP endpoint (GET /health or /healthz) that reports service health<br>• Shallow check: service is running (returns 200 OK)<br>• Deep check: service and ALL dependencies healthy (DB, cache, downstream services)<br>• Used by: load balancers (routing), orchestrators (restart unhealthy), monitoring (alerting)<br>• Best practice: separate liveness (am I alive?) from readiness (can I serve traffic?) |
| 158 | What is the difference between liveness and readiness probes? | liveness vs readiness, Kubernetes | • Liveness probe: is the service alive? If no → Kubernetes restarts the pod<br>• Readiness probe: can the service accept traffic? If no → removed from Service endpoints<br>• Example: service is loading model (alive but not ready) → pass liveness, fail readiness<br>• Startup probe: for services with slow startup (prevents premature liveness failure)<br>• Critical: wrong probe config can cause restart loops or serving traffic to unready pods |
| 159 | What is the Backends for Frontends (BFF) pattern in detail? | BFF, detail | • Each client type (iOS, Android, Web, TV) gets its own dedicated backend<br>• BFF aggregates multiple microservice calls into optimized client payloads<br>• Mobile BFF: smaller payloads, fewer fields, optimized for bandwidth<br>• Web BFF: richer data, more fields, server-side rendering<br>• Each BFF can evolve independently with its client team |
| 160 | How do you handle cross-cutting concerns in microservices? | cross-cutting, concerns | • Options: (1) shared library, (2) sidecar proxy, (3) API gateway, (4) service mesh<br>• Shared library: auth, logging, tracing code in a library used by all services<br>  Pro: simple. Con: must deploy all services when library updates<br>• Sidecar: Envoy proxy handles mTLS, retries, metrics transparently<br>  Pro: language-agnostic. Con: extra memory/CPU per pod<br>• Service mesh: universal networking policies without code changes |
| 161 | What is the difference between synchronous and asynchronous communication? | sync vs async, communication | • Synchronous: caller waits for response (REST, gRPC) — real-time result needed<br>• Asynchronous: caller sends message and moves on (Kafka, SQS) — eventual processing<br>• Sync: simpler, but creates temporal coupling (both services must be up)<br>• Async: decoupled, resilient, but harder to debug and reason about<br>• Rule of thumb: queries → sync; commands with side effects → async |
| 162 | What is an idempotent consumer? | idempotent consumer | • Consumer that handles duplicate messages safely (same result on reprocessing)<br>• Needed because: at-least-once delivery guarantees may deliver messages multiple times<br>• Implementation: track processed message IDs in database, check before processing<br>• Example: payment consumer checks if order already paid before charging again<br>• Storage: message ID → result mapping with TTL |
| 163 | What is the Decomposition pattern for microservices? | decomposition, boundaries | • How to split a monolith into services<br>• By Business Capability: User Management, Order Management, Payment Processing<br>• By Subdomain (DDD): Bounded Contexts define service boundaries<br>• Guidelines: single responsibility, high cohesion within, loose coupling between<br>• Anti-pattern: splitting by technical layer (DB team, API team) — creates cross-service dependencies |
| 164 | What is an anti-corruption layer (ACL)? | ACL, anti-corruption | • Translation layer between a new service and a legacy system<br>• Prevents legacy domain concepts from "corrupting" the new service's domain model<br>• New service speaks its own language → ACL translates to/from legacy format<br>• When to use: integrating with legacy systems during migration<br>• DDD concept: keeps bounded contexts clean and independent |
| 165 | What is blue-green deployment? | blue-green, deployment | • Two identical environments: Blue (current) and Green (new version)<br>• Deploy new version to Green → test → switch router to Green → Blue becomes standby<br>• Rollback: switch router back to Blue (instant)<br>• Advantage: zero-downtime deployment, instant rollback<br>• Disadvantage: need double the infrastructure (temporarily) |
| 166 | What is canary deployment? | canary, deployment | • Deploy new version to a small percentage of traffic (e.g., 5%)<br>• Monitor: error rate, latency, user feedback<br>• If metrics good → gradually increase (10%, 25%, 50%, 100%)<br>• If metrics bad → rollback by routing all traffic to old version<br>• Named after canary in coal mine — early warning of problems |
| 167 | What is feature flagging? | feature flags, toggling | • Deploy new code to production behind an off flag → enable gradually<br>• Decouple deployment from release: code is deployed but feature not visible<br>• Use for: A/B testing, gradual rollout, emergency kill switch<br>• Tools: LaunchDarkly, Unleash, environment variables<br>• Clean up: remove old flags to avoid "flag debt" |
| 168 | What is the correlation ID pattern? | correlation ID, tracing | • Unique ID attached to every request, propagated to all downstream services<br>• Enables: tracing a request's path through the entire system<br>• HTTP header: `X-Correlation-ID` or `X-Request-ID`<br>• Added at: API gateway (first entry point) if not already present<br>• Logging: every service log line includes correlation ID → easy cross-service debugging |
| 169 | What are compensating transactions? | compensating, transactions | • Actions that undo the effect of a previously completed step in a saga<br>• Example: if shipping fails → compensate by: cancel shipment → refund payment → cancel order<br>• Not always exact reversal: "delete user" compensation might be "disable user" (soft delete)<br>• Must be idempotent: compensating action might be retried on failure<br>• Design consideration: some actions are NOT compensatable (sent email, physical shipment) |
| 170 | What is the Two-Pizza Rule? | two-pizza, team size | • Jeff Bezos: a team should be small enough to feed with two pizzas (6-8 people)<br>• Microservices context: each small team owns 1-3 services<br>• Communication overhead: grows as N² with team size → small teams communicate faster<br>• Conway's Law: organizations design systems that mirror their communication structure<br>• Result: microservice boundaries often align with team boundaries |


---
<a id="api-design"></a>
# Section 4: API Design

APIs (Application Programming Interfaces) define how clients communicate with services. Designing clean, consistent APIs is critical for developer experience and system maintainability.


## 4.1 REST Principles

REST (Representational State Transfer) is an architectural style for designing networked applications.

### REST Constraints

| Constraint | Meaning | Practical Implication |
|------------|---------|----------------------|
| **Client-Server** | Separation of concerns between client (UI) and server (data) | Client and server evolve independently |
| **Stateless** | Each request contains all info needed — no server-side session | Scalable: any server can handle any request |
| **Cacheable** | Responses explicitly indicate if they're cacheable | Cache-Control headers, ETags, CDN-friendly |
| **Uniform Interface** | Consistent resource representation (URLs, methods, status codes) | Predictable, self-documenting APIs |
| **Layered System** | Client can't tell if directly connected to server or intermediary | Load balancers, caches, gateways transparent |
| **Code on Demand (optional)** | Server can send executable code (JS) | Rarely used in APIs |

### RESTful Resource Design

| Principle | ✅ Good Example | ❌ Bad Example |
|-----------|----------------|---------------|
| Use nouns, not verbs | `/users` | `/getUsers` |
| Use plural nouns | `/users/123` | `/user/123` |
| Nest related resources | `/users/123/orders` | `/getUserOrders?id=123` |
| Use query params for filtering | `/users?status=active&role=admin` | `/activeAdminUsers` |
| Use HTTP methods for actions | `DELETE /users/123` | `POST /deleteUser` |
| Consistent naming (kebab-case) | `/user-profiles` | `/userProfiles` or `/user_profiles` |


## 4.2 HTTP Methods & Status Codes

### HTTP Methods (Verbs)

| Method | Purpose | Idempotent? | Safe? | Request Body | Example |
|--------|---------|:-----------:|:-----:|:------------:|---------|
| **GET** | Read/retrieve resource | ✅ | ✅ | No | `GET /users/123` |
| **POST** | Create new resource | ❌ | ❌ | Yes | `POST /users` |
| **PUT** | Replace entire resource | ✅ | ❌ | Yes | `PUT /users/123` |
| **PATCH** | Partial update | ❌* | ❌ | Yes | `PATCH /users/123` |
| **DELETE** | Remove resource | ✅ | ❌ | No | `DELETE /users/123` |
| **HEAD** | Same as GET but no body | ✅ | ✅ | No | `HEAD /users/123` |
| **OPTIONS** | Get allowed methods | ✅ | ✅ | No | `OPTIONS /users` |

### HTTP Status Codes — Complete Reference

| Range | Category | Common Codes |
|-------|----------|-------------|
| **1xx** | Informational | 100 Continue, 101 Switching Protocols |
| **2xx** | Success | 200 OK, 201 Created, 202 Accepted, 204 No Content |
| **3xx** | Redirection | 301 Moved Permanently, 302 Found, 304 Not Modified |
| **4xx** | Client Error | 400 Bad Request, 401 Unauthorized, 403 Forbidden, 404 Not Found, 409 Conflict, 422 Unprocessable Entity, 429 Too Many Requests |
| **5xx** | Server Error | 500 Internal Server Error, 502 Bad Gateway, 503 Service Unavailable, 504 Gateway Timeout |

### When to Use Each Status Code

| Scenario | Status Code | When |
|----------|-------------|------|
| Successful GET | `200 OK` | Resource found and returned |
| Successful POST (created) | `201 Created` | New resource created (include Location header) |
| Async processing accepted | `202 Accepted` | Request received, will be processed later |
| Successful DELETE | `204 No Content` | Resource deleted, no body returned |
| Validation error | `400 Bad Request` | Invalid JSON, missing required field |
| Not authenticated | `401 Unauthorized` | Missing or invalid JWT/API key |
| Not authorized | `403 Forbidden` | Valid auth but insufficient permissions |
| Resource not found | `404 Not Found` | ID doesn't exist |
| Conflict | `409 Conflict` | Duplicate email, version mismatch |
| Rate limited | `429 Too Many Requests` | Client exceeded rate limit |
| Server bug | `500 Internal Server Error` | Unhandled exception |


## 4.3 API Versioning

### Versioning Strategies

| Strategy | Example | Pros | Cons |
|----------|---------|------|------|
| **URL Path** | `/api/v1/users` | Clear, easy to implement, cacheable | URL changes on version bump |
| **Query Parameter** | `/api/users?version=1` | Optional, default version | Easy to miss, harder to route |
| **Header** | `Accept: application/vnd.api+json;version=1` | Clean URL, semantic | Hidden, harder to test (no browser URL) |
| **Content Negotiation** | `Accept: application/vnd.company.v2+json` | True REST, media type versioning | Complex, rarely used |

### Versioning Best Practices

| Practice | Explanation |
|----------|-------------|
| Version from day 1 | Adding versioning later is much harder |
| Support max 2-3 versions | Old versions = maintenance burden |
| Deprecation notice | HTTP header `Sunset: Sat, 01 Jan 2025 00:00:00 GMT` |
| Backward-compatible changes | Add fields (don't remove), new optional params |
| Breaking changes = new version | Remove field, change type, change behavior |

## 4.4 Authentication (JWT & OAuth)

### JWT (JSON Web Token)

| Component | Content | Example |
|-----------|---------|---------|
| **Header** | Algorithm + token type | `{"alg": "HS256", "typ": "JWT"}` |
| **Payload** | Claims (user data) | `{"user_id": 123, "role": "admin", "exp": 1700000000}` |
| **Signature** | HMAC of header + payload | `HMACSHA256(base64(header) + "." + base64(payload), secret)` |

```
JWT Structure:
  xxxxx.yyyyy.zzzzz
  ─────  ─────  ─────
  Header Payload Signature
  (base64)(base64)(base64)
```

### JWT vs Session-Based Auth

| Aspect | JWT (Token-Based) | Session-Based |
|--------|-------------------|---------------|
| **Storage** | Client stores token (localStorage, cookie) | Server stores session in memory/Redis |
| **Stateless** | ✅ Server doesn't store anything | ❌ Server must look up session |
| **Scalability** | Easy — any server validates token | Need shared session store (Redis) |
| **Revocation** | Hard — token valid until expiry | Easy — delete session from store |
| **Size** | Larger (contains claims) | Small (just session ID) |
| **Best for** | Microservices, SPAs, mobile | Traditional web apps, server-rendered |

### OAuth 2.0 Flow (Authorization Code)

```
1. User clicks "Login with Google"
2. App redirects to Google's auth page:
   GET https://accounts.google.com/authorize
     ?client_id=APP_ID
     &redirect_uri=https://app.com/callback
     &response_type=code
     &scope=openid email profile
3. User logs in to Google and consents
4. Google redirects back to app with auth code:
   GET https://app.com/callback?code=AUTH_CODE
5. App exchanges code for tokens (server-to-server):
   POST https://oauth2.googleapis.com/token
     grant_type=authorization_code&code=AUTH_CODE&client_secret=SECRET
6. Google returns: access_token + refresh_token + id_token
7. App uses access_token to call Google APIs on behalf of user
```

### OAuth 2.0 Grant Types

| Grant Type | Use Case | Flow |
|------------|----------|------|
| **Authorization Code** | Web apps with server backend | Redirect → code → exchange → token |
| **Authorization Code + PKCE** | SPAs and mobile apps | Same + code verifier (no client secret) |
| **Client Credentials** | Machine-to-machine (no user) | Service sends client_id + secret → token |
| **Refresh Token** | Extend session without re-login | Send refresh_token → get new access_token |


## 4.5 Authorization

| Model | How It Works | Best For |
|-------|-------------|----------|
| **RBAC** (Role-Based) | Users assigned roles; roles have permissions | Most applications (admin, editor, viewer) |
| **ABAC** (Attribute-Based) | Policies based on user/resource/environment attributes | Complex rules (time-based, location-based) |
| **ReBAC** (Relationship-Based) | Access based on resource relationships | Google Zanzibar, social networks, file sharing |
| **ACL** (Access Control List) | Per-resource list of who can do what | File systems, simple permissions |

### RBAC vs ABAC

| Aspect | RBAC | ABAC |
|--------|------|------|
| **Rule** | User has role → role has permission | If (user.department == resource.department AND time < 5PM) → allow |
| **Flexibility** | Limited (static roles) | Very flexible (dynamic policies) |
| **Complexity** | Simple to implement | Complex policy engine needed |
| **Example** | Admin can delete any post | Manager can delete posts in their department during business hours |

## 4.6 Pagination & Filtering

### Pagination Strategies

| Strategy | How | Pros | Cons |
|----------|-----|------|------|
| **Offset-Based** | `?page=3&limit=20` → OFFSET 40 LIMIT 20 | Simple, jump to any page | Slow on large offsets, inconsistent if data changes |
| **Cursor-Based** | `?cursor=abc123&limit=20` → WHERE id > cursor | Consistent, fast on any position | Can't jump to arbitrary page |
| **Keyset** | `?after_id=500&limit=20` → WHERE id > 500 ORDER BY id | Efficient with index, stable pagination | Only works with sortable unique column |

### Offset vs Cursor Comparison

| Aspect | Offset | Cursor |
|--------|--------|--------|
| **Query** | `SELECT * FROM items OFFSET 10000 LIMIT 20` | `SELECT * FROM items WHERE id > 'last_seen_id' LIMIT 20` |
| **Performance** | O(offset) — DB scans and discards offset rows | O(1) — index seek directly to cursor position |
| **Stability** | Rows shift if data inserted/deleted between pages | Stable — always picks up where left off |
| **Random access** | ✅ Can jump to page 50 | ❌ Must traverse sequentially |
| **Use case** | Admin dashboards, small datasets | Infinite scroll, mobile feeds, large datasets |

### Filtering & Sorting

```
GET /api/products?
  category=electronics&         # Filter by category
  price_min=100&price_max=500&  # Range filter
  brand=apple,samsung&          # Multiple values (OR)
  sort=price&                   # Sort field
  order=asc&                    # Sort direction
  fields=id,name,price&         # Field selection (sparse fieldsets)
  page=2&limit=20               # Pagination
```

## 4.7 OpenAPI / Swagger

| Concept | Description |
|---------|-------------|
| **OpenAPI Spec** | Standard format (YAML/JSON) for describing REST APIs |
| **Swagger UI** | Interactive documentation generated from OpenAPI spec |
| **Swagger Editor** | Visual editor for writing OpenAPI specs |
| **Code Generation** | Generate client SDKs and server stubs from spec |
| **Contract-First** | Write spec first → then implement (recommended) |
| **Code-First** | Write code → generate spec from annotations (faster but less intentional) |


---
## Section 4 — Interview Questions: API Design (60 Questions)


| # | Question | Keywords | Answer |
|---|----------|----------|--------|
| 171 | What is REST? | REST, definition | • Representational State Transfer — architectural style for web APIs<br>• Resources identified by URLs, manipulated via HTTP methods (GET, POST, PUT, DELETE)<br>• Stateless: each request contains all info needed (no server-side session)<br>• Cacheable: responses indicate cacheability via headers<br>• Uniform interface: consistent URL patterns, HTTP semantics, standard status codes |
| 172 | What are the key REST constraints? | REST, constraints | • Client-server separation (frontend/backend independent)<br>• Stateless (no server-side session per client)<br>• Cacheable (responses declare cacheability)<br>• Uniform interface (resources, HTTP methods, status codes)<br>• Layered system (proxies, LBs transparent to client)<br>• Code on demand (optional — server can send executable code) |
| 173 | What is the difference between PUT and PATCH? | PUT vs PATCH | • PUT: replaces ENTIRE resource — send complete object<br>• PATCH: updates PART of resource — send only changed fields<br>• PUT is idempotent: same PUT twice = same result<br>• PATCH may not be idempotent: `{"views": "+1"}` increments each time<br>• Use PUT when client has full resource; PATCH when updating one field |
| 174 | What is the difference between 401 and 403? | 401 vs 403 | • 401 Unauthorized: client not authenticated (missing/invalid credentials)<br>• 403 Forbidden: client authenticated but not authorized (wrong permissions)<br>• 401: "Who are you?" — needs login/token<br>• 403: "I know who you are, but you can't do this" — has valid token but insufficient role<br>• Fix 401 by providing credentials; 403 by requesting higher permissions |
| 175 | How do you design a RESTful URL? | URL design, REST | • Use nouns: `/users`, `/orders`, `/products` (not `/getUsers`)<br>• Use plural: `/users/123` (not `/user/123`)<br>• Nest related resources: `/users/123/orders` (orders belonging to user 123)<br>• Use query params for filtering: `/users?status=active&sort=name`<br>• Use kebab-case: `/user-profiles` (not camelCase or snake_case) |
| 176 | What is HATEOAS? | HATEOAS, REST | • Hypermedia As The Engine Of Application State<br>• Response includes links to related actions/resources<br>• Client discovers available actions dynamically (not hardcoded)<br>• Example: GET /orders/123 response includes links: `{"cancel": "/orders/123/cancel", "pay": "/orders/123/pay"}`<br>• Rarely implemented in practice (adds complexity, most APIs skip it) |
| 177 | What is API versioning and which strategy is best? | versioning, strategy | • Maintaining backward compatibility while evolving the API<br>• URL path versioning: `/api/v1/users` — most popular, clear, cacheable<br>• Header versioning: `Accept: application/vnd.api.v2+json` — clean URL but hidden<br>• Best practice: URL path for simplicity; support max 2-3 versions simultaneously<br>• Deprecation: announce via headers + docs, give 6+ months migration window |
| 178 | What is a JWT and how does it work? | JWT, token | • JSON Web Token — self-contained signed token for authentication<br>• Structure: header.payload.signature (Base64 encoded, dot-separated)<br>• Header: algorithm (HS256/RS256). Payload: claims (user_id, role, expiry). Signature: HMAC or RSA<br>• Stateless: server validates by checking signature, no session lookup needed<br>• Drawback: can't be revoked before expiry (need blacklist or short TTL + refresh) |
| 179 | What is the difference between JWT and session-based auth? | JWT vs session | • JWT: token stored on client, server is stateless (no server-side storage)<br>• Session: session ID stored on client, session data in server memory or Redis<br>• JWT: scales easily (any server validates), harder to revoke<br>• Session: easy to revoke (delete from store), but needs shared store for horizontal scaling<br>• Modern SPAs and microservices prefer JWT; traditional web apps use sessions |
| 180 | What is OAuth 2.0? | OAuth, authorization | • Authorization framework — lets third-party apps access user's resources<br>• Key roles: Resource Owner (user), Client (app), Authorization Server (Google), Resource Server (API)<br>• Authorization Code flow: user consents → app gets auth code → exchanges for tokens<br>• Access token: short-lived, used for API calls. Refresh token: long-lived, used to get new access token<br>• "Login with Google/GitHub/Facebook" uses OAuth 2.0 |
| 181 | What is the Authorization Code + PKCE flow? | PKCE, OAuth, SPA | • PKCE: Proof Key for Code Exchange — secures OAuth for public clients (SPA, mobile)<br>• Problem: SPAs can't securely store client_secret (code is visible in browser)<br>• Solution: client generates random code_verifier → sends hash (code_challenge) in auth request<br>• Exchange: sends original code_verifier with auth code → server verifies hash matches<br>• Prevents: authorization code interception attacks (interceptor doesn't have verifier) |
| 182 | What is RBAC? | RBAC, authorization | • Role-Based Access Control: users → roles → permissions<br>• Example: student role has read; teacher has read+write; admin has read+write+delete<br>• Implementation: store role in JWT claims or database lookup<br>• Middleware: check `user.role` against required role for each endpoint<br>• Simple, widely used, sufficient for most applications |
| 183 | What is the difference between RBAC and ABAC? | RBAC vs ABAC | • RBAC: permission based on role (admin can delete any post)<br>• ABAC: permission based on attributes (manager can delete posts in their department during business hours)<br>• RBAC: simple, easy to audit, covers 90% of use cases<br>• ABAC: flexible, complex, needed for nuanced policies<br>• Many systems start with RBAC and add ABAC for specific edge cases |
| 184 | What is offset vs cursor pagination? | pagination, offset vs cursor | • Offset: `?page=100&limit=20` → OFFSET 2000 LIMIT 20 — DB scans 2000 rows to skip<br>• Cursor: `?cursor=last_id&limit=20` → WHERE id > last_id LIMIT 20 — index seek, O(1)<br>• Offset: supports random page access but slow for large offsets<br>• Cursor: fast at any position but must traverse sequentially (no "jump to page 50")<br>• Use cursor for: infinite scroll, feeds, mobile apps. Use offset for: admin dashboards |
| 185 | How do you design error responses? | error response, design | • Consistent structure: `{"error": {"code": "VALIDATION_ERROR", "message": "Email is required", "details": [...]}}`<br>• Include: machine-readable error code + human-readable message<br>• Validation: list each field error in details array<br>• Don't expose: stack traces, internal IDs, database schema in production<br>• HTTP status code matches error type (400 for validation, 500 for server error) |
| 186 | What is content negotiation? | content negotiation, Accept | • Client specifies preferred response format via Accept header<br>• `Accept: application/json` → JSON response<br>• `Accept: application/xml` → XML response<br>• Server responds with Content-Type matching accepted format<br>• 406 Not Acceptable if server can't provide requested format |
| 187 | What is an idempotency key? | idempotency key, API | • Client-generated unique ID sent with non-idempotent requests (POST)<br>• Server checks if key was already processed before executing<br>• If already processed → return cached result (no duplicate creation)<br>• If new → process request → cache result with key (TTL 24-48 hours)<br>• Used by: Stripe, PayPal, Square for payment deduplication |
| 188 | What is rate limiting at the API level? | rate limiting, API | • Limiting requests per client per time window (e.g., 100 req/min)<br>• Response headers: X-RateLimit-Limit, X-RateLimit-Remaining, Retry-After<br>• HTTP 429 Too Many Requests when limit exceeded<br>• Per-client: by API key, user ID, or IP address<br>• Per-endpoint: stricter limits on expensive operations (search: 10/min vs list: 100/min) |
| 189 | What is GraphQL and when to use it over REST? | GraphQL, vs REST | • Query language where client specifies exactly what data to return<br>• Single endpoint: POST /graphql with query in body<br>• Solves: over-fetching (REST returns too many fields) and under-fetching (need multiple REST calls)<br>• Best for: BFF pattern, mobile apps (bandwidth-sensitive), complex nested data<br>• Trade-offs: caching harder, complexity higher, N+1 query risk on server side |
| 190 | What is API throttling vs rate limiting? | throttling vs rate limiting | • Rate limiting: hard limit — reject requests above threshold (429 error)<br>• Throttling: soft limit — slow down requests (queue them, delay response)<br>• Rate limiting: protects server from abuse (security-focused)<br>• Throttling: smooths traffic spikes (performance-focused)<br>• Some APIs use both: throttle first, rate-limit if still overwhelmed |


| # | Question | Keywords | Answer |
|---|----------|----------|--------|
| 191 | What is CORS and why does it matter? | CORS, cross-origin | • Cross-Origin Resource Sharing — browser security mechanism<br>• Blocks: JavaScript on domain A from calling API on domain B (by default)<br>• Server must explicitly allow: `Access-Control-Allow-Origin: https://app.com`<br>• Preflight: browser sends OPTIONS request before actual request (for non-simple requests)<br>• Common issue: forgetting to add CORS headers → API works in Postman but fails in browser |
| 192 | What is an API gateway vs a reverse proxy? | API gateway vs reverse proxy | • Reverse proxy: forwards requests, handles SSL, caching, load balancing (Nginx)<br>• API gateway: reverse proxy + API-specific features (auth, rate limiting, transformation, analytics)<br>• Reverse proxy: infrastructure concern (doesn't understand API logic)<br>• API gateway: application concern (understands API routes, versions, consumers)<br>• Many API gateways are built on reverse proxies (Kong uses Nginx) |
| 193 | What is ETags and conditional requests? | ETag, conditional request | • ETag: hash of response content — represents current version of resource<br>• Client stores ETag, sends `If-None-Match: "etag"` on subsequent request<br>• If resource unchanged → 304 Not Modified (no body, saves bandwidth)<br>• If changed → 200 OK with new content and new ETag<br>• Great for caching: reduces bandwidth without serving stale data |
| 194 | What is webhook vs polling? | webhook vs polling | • Polling: client repeatedly asks server "any updates?" (every 5 seconds)<br>• Webhook: server calls client's URL when something happens (push-based)<br>• Polling: simple, but wastes resources (99% of polls return "no updates")<br>• Webhook: efficient, real-time, but harder to implement (client must expose URL, handle retries)<br>• Webhook retry: if client URL fails, server retries with exponential backoff |
| 195 | How do you handle large file uploads via API? | file upload, large | • Multipart upload: split file into chunks, upload each chunk separately<br>• Presigned URL: server generates signed S3 URL → client uploads directly to S3 (bypasses server)<br>• Chunked upload: client sends chunk 1 → server acks → send chunk 2 (resumable)<br>• Size limit: enforce MAX_CONTENT_LENGTH (reject oversized uploads early)<br>• Progress: chunk-based upload enables progress tracking |
| 196 | What is API-first development? | API-first, design | • Design the API specification before writing implementation code<br>• Write OpenAPI spec → review with team → agree on contract → then implement<br>• Benefits: frontend and backend can work in parallel (mock server from spec)<br>• Catches: design issues early (before coding), consistent API design<br>• Alternative: code-first (write code → generate spec) — faster but less intentional |
| 197 | What is sparse fieldsets? | sparse fieldsets, optimization | • Client specifies which fields to return: `GET /users?fields=id,name,email`<br>• Server returns only requested fields (less data transfer)<br>• Reduces: bandwidth usage, serialization time, especially for mobile clients<br>• Implemented by: GraphQL (natively), JSON:API (sparse fieldsets), custom query param<br>• Good for: large objects where client only needs a few fields |
| 198 | What are best practices for API error handling? | error handling, API best practices | • Use standard HTTP status codes (don't return 200 with error body)<br>• Consistent error format: `{"error": {"code": "...", "message": "..."}}`<br>• Include machine-readable error codes for client logic<br>• Provide actionable error messages for developers<br>• Log errors server-side with request ID for debugging<br>• Don't expose internal details (stack traces, queries) in production |
| 199 | How do you handle API backwards compatibility? | backward compatibility | • Additive changes only: add new fields, new endpoints, new optional parameters<br>• Never: remove fields, rename fields, change field types, change behavior<br>• New fields: make optional with defaults (don't break existing clients)<br>• Breaking changes: announce deprecation → support old version 6+ months → sunset<br>• Testing: run old client test suite against new API version |
| 200 | What is the Richardson Maturity Model? | Richardson, maturity, REST | • Level 0: One URL, one method (RPC over HTTP) — e.g., `POST /api` with action in body<br>• Level 1: Resources — different URLs per resource (`/users`, `/orders`)<br>• Level 2: HTTP Methods — use correct HTTP verbs (GET for read, POST for create)<br>• Level 3: HATEOAS — responses include hypermedia links to related actions<br>• Most APIs are Level 2; Level 3 is rare in practice |


---
<a id="event-driven"></a>
# Section 5: Event-Driven Architecture

Event-driven architecture (EDA) is a design pattern where services communicate by producing and consuming **events** — notifications that something happened.


## 5.1 Pub/Sub Model

### Publish-Subscribe Pattern

```
Producer(s)                    Consumer(s)
┌──────────┐    Event Bus     ┌──────────┐
│ Order Svc│ ──"OrderCreated"──│Payment   │
│          │ ──"OrderCreated"──│Inventory │
│          │ ──"OrderCreated"──│Analytics │
└──────────┘                  └──────────┘

Key: ONE event → MULTIPLE consumers (fan-out)
     Producers don't know who consumes
     Consumers subscribe to topics of interest
```

### Pub/Sub vs Point-to-Point

| Aspect | Pub/Sub | Point-to-Point (Queue) |
|--------|---------|----------------------|
| **Consumers** | Multiple consumers per message | One consumer per message |
| **Delivery** | Fan-out (all subscribers get message) | Competing consumers (only one gets it) |
| **Use case** | Event notification, broadcasting | Task distribution, work queue |
| **Example** | "OrderPlaced" → notify email, inventory, analytics | "ProcessImage" → one of many workers |
| **System** | Kafka topics, Redis Pub/Sub, SNS | RabbitMQ, SQS, Celery |

## 5.2 Message Brokers — Kafka Deep Dive

### Kafka Architecture

| Component | Description |
|-----------|-------------|
| **Broker** | Kafka server that stores and serves messages |
| **Topic** | Named category/feed of messages (like a table name) |
| **Partition** | Ordered, immutable sequence within a topic (enables parallelism) |
| **Producer** | Service that publishes messages to topics |
| **Consumer** | Service that reads messages from topics |
| **Consumer Group** | Group of consumers sharing the work of reading a topic |
| **Offset** | Position of a consumer in a partition (how far it has read) |
| **Replication Factor** | Number of copies of each partition (usually 3 for durability) |

### Kafka Partitioning

```
Topic: "orders" with 3 partitions

Partition 0: [order-1] [order-4] [order-7] [order-10] ...
Partition 1: [order-2] [order-5] [order-8] [order-11] ...
Partition 2: [order-3] [order-6] [order-9] [order-12] ...

Assignment: hash(message_key) % num_partitions
Key = user_id → all orders for same user go to same partition → ordering guaranteed per user

Consumer Group "payment-service" (3 consumers):
  Consumer-A reads Partition 0
  Consumer-B reads Partition 1
  Consumer-C reads Partition 2
  → parallelism = num_partitions
```

### Kafka vs RabbitMQ vs SQS

| Feature | Kafka | RabbitMQ | AWS SQS |
|---------|-------|----------|---------|
| **Type** | Distributed log | Message queue | Managed queue |
| **Message retention** | Configurable (days/forever) | Until consumed | 14 days max |
| **Ordering** | Per-partition guaranteed | Per-queue (with single consumer) | Best-effort (FIFO option available) |
| **Throughput** | Very high (millions/sec) | Moderate (tens of thousands/sec) | High (managed, auto-scales) |
| **Replay** | ✅ Re-read from any offset | ❌ Once consumed, gone | ❌ Consumed = deleted |
| **Consumer model** | Pull (consumer polls) | Push (broker delivers) | Pull |
| **Best for** | Event streaming, log aggregation | Task queues, RPC | Serverless, simple queues |


## 5.3 Event Sourcing

Instead of storing current state, store the **complete history of events** that led to the current state.

### Traditional vs Event Sourcing

| Aspect | Traditional (State Store) | Event Sourcing |
|--------|--------------------------|----------------|
| **Storage** | Current state only (e.g., balance = $500) | All events: [+$1000, -$200, -$300] |
| **History** | Lost — only current state available | Complete audit trail |
| **Rebuilding** | Not possible | Replay all events → derive any past state |
| **Schema change** | Complex migrations | Add new event types, replay to migrate |
| **Debugging** | "What happened?" → don't know | Replay events → see exactly what happened |
| **Complexity** | Simpler | More complex (event versioning, snapshots needed) |

### Event Sourcing Flow

```
Command: "Withdraw $200 from account A"
  
1. Load events for account A:
   [AccountCreated($1000), Deposited($500), Withdrawn($200)]

2. Build current state from events:
   Balance = $1000 + $500 - $200 = $1300

3. Validate: $1300 >= $200? YES

4. Append new event:
   [AccountCreated($1000), Deposited($500), Withdrawn($200), Withdrawn($200)]

5. Project new state:
   Balance = $1000 + $500 - $200 - $200 = $1100

Snapshots:
  - Periodically save current state as snapshot
  - To rebuild: load last snapshot + replay events after snapshot
  - Avoids replaying millions of events from beginning
```

## 5.4 CQRS Pattern

### Command Query Responsibility Segregation

```
                    ┌──────────────────┐
                    │   API Gateway     │
                    └───────┬──────────┘
                   ┌────────┼────────┐
                   ▼                 ▼
         ┌──────────────┐  ┌──────────────┐
         │  Command Side │  │  Query Side   │
         │  (Write Model)│  │ (Read Model)  │
         │  Normalized   │  │ Denormalized  │
         │  Primary DB   │  │ Read Replica  │
         └───────┬───────┘  └──────────────┘
                 │                    ▲
                 │  Events (async)    │
                 └────────────────────┘
```

| Component | Write Model | Read Model |
|-----------|-------------|------------|
| **Optimized for** | Data integrity, validation | Query performance, UI |
| **Schema** | Normalized (3NF, no duplication) | Denormalized (pre-joined, pre-aggregated) |
| **Database** | PostgreSQL (ACID) | Elasticsearch, Redis, materialized views |
| **Scale** | Single primary (write bottleneck) | Multiple read replicas (horizontally scalable) |
| **Consistency** | Immediate | Eventually consistent (slight lag from event propagation) |


## 5.5 Delivery Guarantees

| Guarantee | Description | How | Trade-off |
|-----------|-------------|-----|-----------|
| **At-Most-Once** | Message delivered 0 or 1 times | Fire and forget — no retry | May lose messages |
| **At-Least-Once** | Message delivered 1 or more times | Retry on failure until ack | May produce duplicates |
| **Exactly-Once** | Message delivered exactly 1 time | At-least-once + idempotent consumer | Complex, performance overhead |

### How Kafka Achieves Exactly-Once

```
Producer:
  - enable.idempotence = true
  - Each message gets sequence number per partition
  - Broker deduplicates by (producer_id, partition, sequence_number)
  - Result: producer retries don't cause duplicate messages

Consumer + Transactions:
  - Consumer reads message → processes → writes result to DB → commits offset
  - All three in a Kafka transaction (read + process + commit atomically)
  - If consumer crashes: uncommitted offset → message reprocessed
  - If committed: message won't be reprocessed

End-to-end:
  Idempotent Producer + Transactional Consumer = Exactly-Once Semantics
```

### Dead Letter Queue (DLQ)

| Concept | Description |
|---------|-------------|
| **What** | Special queue/topic for messages that failed processing after max retries |
| **Why** | Don't block good messages behind a poison pill (bad message) |
| **Flow** | Process message → fails 3 times → move to DLQ → continue processing next message |
| **Action** | Engineers investigate DLQ messages, fix bug, replay them |
| **Important** | DLQ must be monitored — messages sitting there means lost processing |


---
## Section 5 — Interview Questions: Event-Driven Architecture (60 Questions)


| # | Question | Keywords | Answer |
|---|----------|----------|--------|
| 201 | What is event-driven architecture? | EDA, definition | • Architecture where services communicate by emitting and consuming events<br>• Event: notification that something happened ("OrderCreated", "PaymentProcessed")<br>• Producers don't know who consumes; consumers subscribe to topics of interest<br>• Benefits: loose coupling, scalability, async processing<br>• Components: producers, event bus/broker, consumers |
| 202 | What is the difference between a command and an event? | command vs event | • Command: imperative, tells a service what to do ("CreateOrder") — directed at one receiver<br>• Event: notification, announces what happened ("OrderCreated") — broadcast to any interested consumer<br>• Command: expects a response/confirmation. Event: fire-and-forget<br>• Command: tight coupling with receiver. Event: loose coupling with unknown consumers<br>• Best practice: commands for synchronous operations, events for async fan-out |
| 203 | What is the pub/sub pattern? | pub/sub, pattern | • Publish-Subscribe: producers publish events to topics; consumers subscribe to topics<br>• Decoupling: producer doesn't know consumers; consumers don't know producers<br>• Fan-out: one event delivered to ALL subscribers (vs queue where one consumer gets it)<br>• Use cases: event notification, broadcasting, log aggregation<br>• Implementations: Kafka topics, Redis Pub/Sub, AWS SNS, Google Pub/Sub |
| 204 | What is Kafka and how does it work? | Kafka, architecture | • Distributed streaming platform — stores ordered, immutable event logs<br>• Topics: named feeds of events. Partitions: enable parallel processing within a topic<br>• Producers write to topics; consumers read from partitions by tracking offsets<br>• Retention: messages kept for configurable duration (or forever) — enables replay<br>• Consumer groups: group of consumers sharing the work (one partition per consumer max) |
| 205 | How does Kafka ensure message ordering? | Kafka, ordering | • Ordering guaranteed WITHIN a partition (not across partitions)<br>• Producer uses message key to determine partition: hash(key) % num_partitions<br>• Example: key = user_id → all events for same user in same partition → ordered<br>• If ordering needed globally: use single partition (but limits throughput)<br>• Trade-off: more partitions = more parallelism but no cross-partition ordering |
| 206 | What is a Kafka consumer group? | consumer group, Kafka | • Group of consumers sharing the work of reading from a topic<br>• Each partition assigned to exactly one consumer in the group<br>• Adding consumers: redistribute partitions (rebalancing)<br>• Max consumers in group = number of partitions (extras sit idle)<br>• Multiple groups: each group gets ALL messages (independent consumption) |
| 207 | What is event sourcing? | event sourcing, definition | • Storing every state change as an immutable event (not just current state)<br>• Current state derived by replaying events from the beginning<br>• Advantages: complete audit trail, time travel debugging, no data loss<br>• Challenges: event versioning, performance (snapshots needed), complexity<br>• Used by: banking, accounting, any domain requiring full history |
| 208 | What is CQRS? | CQRS, pattern | • Command Query Responsibility Segregation — separate write and read models<br>• Write model: normalized, transactional, handles commands<br>• Read model: denormalized, optimized for queries, eventually consistent<br>• Benefits: scale reads and writes independently, optimize each for its workload<br>• Often paired with event sourcing (events propagate changes to read model) |
| 209 | What is at-least-once vs exactly-once delivery? | delivery semantics | • At-least-once: message delivered one or more times (retries may cause duplicates)<br>• Exactly-once: message processed exactly one time (no duplicates, no losses)<br>• At-least-once: simple, default for most systems (producer retries on failure)<br>• Exactly-once: complex — requires idempotent producer + transactional consumer<br>• Practical: use at-least-once + idempotent consumer (simpler than true exactly-once) |
| 210 | What is a dead letter queue (DLQ)? | DLQ, dead letter | • Queue/topic for messages that failed processing after maximum retry attempts<br>• Prevents: "poison pill" messages from blocking the entire queue<br>• Flow: process → fail → retry 3x → move to DLQ → process next message<br>• DLQ must be monitored: messages there represent lost/failed processing<br>• Action: investigate, fix bug, replay messages from DLQ |
| 211 | What is the difference between Kafka and RabbitMQ? | Kafka vs RabbitMQ | • Kafka: distributed log — messages retained, replayable, high throughput<br>• RabbitMQ: message queue — messages deleted after consumption, lower latency per message<br>• Kafka: pull-based (consumer polls), ordered per partition, millions/sec throughput<br>• RabbitMQ: push-based (broker delivers), complex routing (exchanges, bindings), lower throughput<br>• Kafka for: event streaming, log aggregation. RabbitMQ for: task queues, RPC |
| 212 | What is an event schema and why is it important? | schema, event | • Formal definition of event structure (field names, types, required fields)<br>• Schema registry: central store of event schemas (Confluent Schema Registry)<br>• Benefits: producers and consumers agree on format → no surprises<br>• Evolution: add fields (backward compatible), never remove/rename (breaking)<br>• Formats: Avro (compact, schema evolution), Protobuf (strong typing), JSON Schema |
| 213 | What is the outbox pattern in event-driven systems? | outbox, event pattern | • Write database change AND event to same database transaction (outbox table)<br>• Separate process polls outbox → publishes to Kafka/broker<br>• Solves: dual-write problem (DB write succeeds but event publish fails)<br>• Guarantees: event published if and only if transaction committed<br>• Alternative: CDC (Change Data Capture) — Debezium reads DB transaction log directly |
| 214 | What is the Choreography vs Orchestration saga? | saga, choreography, orchestration | • Choreography: each service publishes events, next service reacts (no central coordinator)<br>• Orchestration: central service (orchestrator) tells each service what to do in sequence<br>• Choreography: loosely coupled but hard to trace; good for simple flows<br>• Orchestration: centralized control, easy debugging; good for complex flows<br>• Choose based on saga complexity: 2-3 steps → choreography; 5+ steps → orchestration |
| 215 | What is event replay and why is it useful? | replay, event | • Re-processing events from Kafka by resetting consumer offset to earlier position<br>• Use cases: fix bug in consumer logic → replay to reprocess correctly<br>• Rebuild read models: new CQRS projection → replay all events to populate<br>• Audit: replay events to verify processing was correct<br>• Requirements: events must be immutable, consumers must be idempotent |
| 216 | What is backpressure in event-driven systems? | backpressure, flow | • Consumer can't keep up with producer rate → messages pile up<br>• Without backpressure: memory overflow, system crash, message loss<br>• Kafka handles it naturally: messages persist on disk, consumer reads at own pace<br>• Application level: consumer pauses consumption (stop polling), resumes when ready<br>• Monitoring: consumer lag metric (how far behind consumer is from latest offset) |
| 217 | What is a message key in Kafka? | message key, Kafka | • Optional key attached to each message — determines partition assignment<br>• Hash(key) % partitions → same key always goes to same partition<br>• Use case: user_id as key → all user events in same partition → ordering guaranteed<br>• Null key: round-robin distribution across partitions (no ordering guarantee)<br>• Important: changing partition count changes key-to-partition mapping |
| 218 | What is a Kafka offset? | offset, Kafka | • Sequential ID assigned to each message within a partition<br>• Consumer tracks its offset — "I've processed up to offset 42"<br>• Auto-commit: consumer automatically commits offset periodically (risk of reprocessing on crash)<br>• Manual commit: consumer commits after successful processing (at-least-once guarantee)<br>• Reset: set offset to earliest (replay all) or latest (skip to current) |
| 219 | How does Kafka handle fault tolerance? | Kafka, fault tolerance | • Replication: each partition replicated to N brokers (replication factor, usually 3)<br>• Leader: one replica is leader (handles reads/writes); others are followers<br>• ISR (In-Sync Replicas): followers that are caught up with leader<br>• Leader failure: ISR member automatically elected as new leader<br>• acks=all: producer waits for all ISR replicas to acknowledge (strongest durability) |
| 220 | What is the difference between event notification and event-carried state transfer? | event notification vs state transfer | • Event notification: just signal that something happened, no data — consumer calls back for details<br>  Example: `{"event": "OrderCreated", "order_id": 123}` → consumer calls GET /orders/123<br>• Event-carried state transfer: event contains all necessary data — no callback needed<br>  Example: `{"event": "OrderCreated", "order_id": 123, "items": [...], "total": 500}`<br>• Notification: simpler events, tighter coupling (callback). State transfer: bigger events, full decoupling |


| # | Question | Keywords | Answer |
|---|----------|----------|--------|
| 221 | What is Change Data Capture (CDC)? | CDC, change data capture | • Capturing changes from a database and streaming them as events<br>• Reads database transaction log (WAL/binlog) — no application code changes<br>• Tools: Debezium (open-source), AWS DMS, Oracle GoldenGate<br>• Publishes: INSERT/UPDATE/DELETE events to Kafka topics<br>• Use cases: sync databases, populate search indexes, event sourcing from legacy systems |
| 222 | What is a Kafka Connect? | Kafka Connect, integration | • Framework for connecting Kafka with external systems (databases, S3, Elasticsearch)<br>• Source connector: pulls data FROM external system INTO Kafka topic<br>• Sink connector: pushes data FROM Kafka topic INTO external system<br>• No custom code needed — configuration-only (JSON config file)<br>• Examples: Debezium (source from DB), Elasticsearch sink, S3 sink |
| 223 | What is event versioning? | versioning, event | • Evolving event schemas over time while maintaining backward compatibility<br>• Rule 1: never remove fields (old consumers depend on them)<br>• Rule 2: new fields must be optional with default values<br>• Rule 3: breaking changes require new event type (OrderCreatedV2)<br>• Schema registry validates compatibility before allowing schema changes |
| 224 | How do you handle out-of-order events? | out of order, events | • Problem: Event A should be processed before Event B, but B arrives first<br>• Solution 1: timestamp-based reordering (buffer events, sort by timestamp)<br>• Solution 2: sequence numbers (reject if sequence gap detected, request retry)<br>• Solution 3: design for idempotency (processing order doesn't matter)<br>• Kafka: ordered within partition — use same key for related events |
| 225 | What is the event store pattern? | event store, pattern | • Append-only log that stores all domain events<br>• Write: append new events. Read: load all events for an aggregate, replay to get state<br>• Database: purpose-built (EventStoreDB) or regular DB (append-only table)<br>• Snapshots: periodically save current state to avoid replaying millions of events<br>• Used with: event sourcing, CQRS, audit logging |
| 226 | What is stream processing vs batch processing? | stream vs batch | • Batch: process accumulated data periodically (hourly, daily). Tools: Spark, Hadoop<br>• Stream: process events in real-time as they arrive. Tools: Kafka Streams, Flink, Spark Streaming<br>• Batch: higher throughput, higher latency (wait for batch window)<br>• Stream: lower throughput per event, lower latency (immediate processing)<br>• Modern trend: stream processing replacing batch for real-time analytics |
| 227 | What is a Kafka Stream vs Kafka Consumer? | Kafka Streams, consumer | • Kafka Consumer: basic API — poll messages, process, commit offset<br>• Kafka Streams: stream processing library — filter, map, join, aggregate, window<br>• Consumer: good for simple consume-and-process workflows<br>• Streams: good for complex event transformations, joins between topics, windowed aggregations<br>• Streams: library (no separate cluster), lightweight, runs in your application |
| 228 | What is the Competing Consumers pattern? | competing consumers, pattern | • Multiple consumers reading from same queue — each message processed by exactly ONE consumer<br>• Purpose: parallel processing for high throughput<br>• In Kafka: consumers in same consumer group → each partition assigned to one consumer<br>• In RabbitMQ: multiple consumers on same queue → round-robin distribution<br>• Scaling: add more consumers (up to number of partitions in Kafka) |
| 229 | How do you monitor event-driven systems? | monitoring, events | • Consumer lag: how far behind consumer is from latest offset (critical metric)<br>• Throughput: messages/sec produced and consumed per topic<br>• Error rate: messages sent to DLQ / total messages processed<br>• End-to-end latency: time from event published to event processed<br>• Tools: Kafka Manager, Burrow (consumer lag), Prometheus + Grafana |
| 230 | What is the Transactional Outbox pattern in detail? | transactional outbox, detail | • Database table "outbox" stores events alongside business data<br>• Within same transaction: INSERT business row + INSERT outbox event<br>• Polling publisher: separate process reads outbox → publishes to Kafka → marks as published<br>• Or: log-based (CDC) reads DB log → publishes events (no polling needed)<br>• Guarantees: atomicity between data write and event publication |


---
<a id="high-availability"></a>
# Section 6: High Availability & Reliability

High availability (HA) means a system remains operational and accessible for a very high percentage of time. Reliability means it continues to work correctly even when things go wrong.


## 6.1 Availability Numbers (The "Nines")

| Availability | Downtime/Year | Downtime/Month | Downtime/Week |
|-------------|---------------|----------------|---------------|
| **99%** (two nines) | 3.65 days | 7.31 hours | 1.68 hours |
| **99.9%** (three nines) | 8.77 hours | 43.83 minutes | 10.08 minutes |
| **99.95%** | 4.38 hours | 21.92 minutes | 5.04 minutes |
| **99.99%** (four nines) | 52.60 minutes | 4.38 minutes | 1.01 minutes |
| **99.999%** (five nines) | 5.26 minutes | 26.30 seconds | 6.05 seconds |

### Availability in Series vs Parallel

```
Series (ALL must work):
  Availability = A1 × A2 × A3
  Example: 99.9% × 99.9% × 99.9% = 99.7% (three dependent services)

Parallel (ANY can serve):
  Availability = 1 - (1-A1)(1-A2)
  Example: 1 - (1-0.999)(1-0.999) = 1 - 0.000001 = 99.9999%

Implication: redundancy dramatically improves availability
  One server at 99.9% = 8.77 hours downtime/year
  Two servers at 99.9% each = 31.5 seconds downtime/year
```

## 6.2 Fault Tolerance

| Concept | Description |
|---------|-------------|
| **Fault** | Something goes wrong (disk dies, network drops, bug triggered) |
| **Error** | Fault manifests as incorrect behavior (wrong response, timeout) |
| **Failure** | System stops providing required service (complete outage) |

### Fault Tolerance Strategies

| Strategy | How It Works | Example |
|----------|-------------|---------|
| **Redundancy** | Multiple copies of everything | 3 app servers, database replicas, multi-AZ |
| **Replication** | Copy data across nodes | PostgreSQL streaming replication, Kafka partition replicas |
| **Failover** | Detect failure, switch to standby | Primary DB fails → promote replica to primary |
| **Graceful Degradation** | Reduce functionality instead of complete failure | Cache fails → serve from DB (slower but works) |
| **Retry with Backoff** | Retry failed operations with increasing delay | API call fails → retry after 1s, 2s, 4s |
| **Circuit Breaker** | Stop calling failing service | Payment service down → return "try later" instead of hanging |
| **Timeout** | Don't wait forever for response | DB query timeout after 5s → return error |
| **Bulkhead** | Isolate resources per dependency | Separate thread pools for each downstream service |


## 6.3 Redundancy & Failover

### Redundancy Levels

| Level | What's Redundant | Example |
|-------|-----------------|---------|
| **Server** | Multiple app servers behind load balancer | 3 web servers, any can serve requests |
| **Database** | Primary + replicas | PostgreSQL primary + 2 replicas |
| **Data Center** | Multiple availability zones | AWS: us-east-1a, us-east-1b, us-east-1c |
| **Region** | Multiple geographic regions | US-East + EU-West (disaster recovery) |
| **Network** | Multiple network paths | Dual ISP connections, BGP failover |

### Failover Types

| Type | Description | RPO | RTO |
|------|-------------|-----|-----|
| **Hot Standby** | Standby is running, data synchronized, ready to take over instantly | ~0 (real-time sync) | Seconds |
| **Warm Standby** | Standby running but needs to catch up on latest data | Minutes | Minutes |
| **Cold Standby** | Standby exists but not running — needs to be started and data restored | Hours | Hours |

> **RPO** = Recovery Point Objective — how much data can you afford to lose?
> **RTO** = Recovery Time Objective — how long can the system be down?

### Automatic Failover Flow

```
Normal:  Client → LB → Primary DB    Replica DB (warm standby)
                        ↓ WAL stream →     ↓
                   [writes + reads]    [replication]

Primary dies:
  1. Health check fails (no response for 30 seconds)
  2. LB marks primary as unhealthy
  3. Replica promoted to primary (auto or manual)
  4. Application config updated to point to new primary
  5. Old primary removed from rotation

Risk: "split brain" if old primary comes back — use fencing (old primary rejects writes)
```


## 6.4 Health Checks

### Health Check Types

| Type | Checks | Response Time | When to Use |
|------|--------|---------------|-------------|
| **Liveness** | Is the process alive? | < 1 second | Always — detects hung/crashed processes |
| **Readiness** | Can the service handle traffic? | < 2 seconds | After startup, during maintenance |
| **Startup** | Has the service finished initialization? | < 60 seconds | Services with slow startup (loading ML models) |
| **Deep/Dependency** | Are all dependencies healthy? | < 5 seconds | Monitoring dashboards, debugging |

```python
# Example health check endpoint
@app.get("/health")
def health():
    return {"status": "ok"}  # Liveness only

@app.get("/health/ready")
def ready():
    checks = {
        "database": check_db_connection(),      # Can connect to PostgreSQL?
        "cache": check_redis_connection(),       # Can connect to Redis?
        "embedding_model": model_loaded(),       # Is ML model loaded?
    }
    all_healthy = all(checks.values())
    return {"status": "ready" if all_healthy else "not_ready", "checks": checks}
```

## 6.5 Backup & Disaster Recovery

### Backup Strategies

| Strategy | Description | RPO | Storage Cost |
|----------|-------------|-----|-------------|
| **Full Backup** | Copy entire database | Time since last backup | Highest |
| **Incremental** | Only changes since last backup | Lower (more frequent) | Lower |
| **Differential** | Changes since last FULL backup | Medium | Medium |
| **Continuous (WAL)** | Stream every write to backup | Near-zero | Medium-High |
| **Snapshot** | Point-in-time copy (filesystem or VM level) | Time since last snapshot | Depends |

### Disaster Recovery Tiers

| Tier | Strategy | RPO | RTO | Cost |
|------|----------|-----|-----|------|
| **0** | No DR | Total loss | N/A | $0 |
| **1** | Offsite backup | Hours-Days | Hours-Days | Low |
| **2** | Warm standby | Minutes | Minutes | Medium |
| **3** | Hot standby (multi-region active-passive) | Near-zero | Seconds | High |
| **4** | Active-active multi-region | Zero | Zero | Very High |

### 3-2-1 Backup Rule

```
3 copies of your data
2 different storage types (disk + cloud)
1 offsite copy (different geographic location)
```


## 6.6 Chaos Engineering

Intentionally introducing failures to test system resilience.

### Chaos Engineering Principles

| Principle | Description |
|-----------|-------------|
| **Build a hypothesis** | "If we kill one app server, latency stays under 200ms" |
| **Run in production** | Staging doesn't catch everything — real traffic reveals real issues |
| **Minimize blast radius** | Start small (one server, one user), gradually increase scope |
| **Automate experiments** | Run chaos tests continuously, not just once |
| **Stop on real damage** | If experiment causes user-visible outage, stop immediately |

### Chaos Experiments

| Experiment | What It Tests | Tool |
|------------|--------------|------|
| **Kill a server** | Failover, load balancing, auto-scaling | Chaos Monkey |
| **Inject network latency** | Timeout handling, circuit breakers | Toxiproxy, tc (traffic control) |
| **Fill disk** | Disk full handling, alerting | Custom script |
| **Corrupt DNS** | DNS failover, caching | Custom |
| **Kill a database** | Failover, data integrity, reconnection | Litmus Chaos, Gremlin |
| **Simulate region outage** | Multi-region failover | GameDay (AWS) |

### Netflix Simian Army

| Tool | What It Does |
|------|-------------|
| **Chaos Monkey** | Randomly kills instances in production |
| **Chaos Gorilla** | Simulates entire AZ failure |
| **Chaos Kong** | Simulates entire region failure |
| **Latency Monkey** | Adds artificial network latency |
| **Janitor Monkey** | Cleans up unused resources |


---
## Section 6 — Interview Questions: High Availability & Reliability (60 Questions)


| # | Question | Keywords | Answer |
|---|----------|----------|--------|
| 231 | What is high availability? | HA, definition | • System remains operational for a very high percentage of time (99.9%+)<br>• Measured in "nines": 99.99% = 52 minutes downtime per year<br>• Achieved through: redundancy, failover, load balancing, monitoring<br>• Not the same as fault tolerance (HA = minimize downtime; FT = continue working despite faults)<br>• Business impact: every hour of downtime costs revenue and trust |
| 232 | What is the difference between availability and reliability? | availability vs reliability | • Availability: percentage of time the system is operational (uptime/total time)<br>• Reliability: probability the system works correctly over a period (no errors, correct results)<br>• A system can be available but unreliable (returns wrong data 5% of the time)<br>• A system can be reliable but unavailable (works perfectly when up but has frequent outages)<br>• Both needed: available AND reliable |
| 233 | What do "nines" of availability mean? | nines, availability | • 99% = 3.65 days downtime/year (two nines — unacceptable for most services)<br>• 99.9% = 8.77 hours/year (three nines — acceptable for internal tools)<br>• 99.99% = 52 minutes/year (four nines — standard for most SaaS)<br>• 99.999% = 5.26 minutes/year (five nines — mission critical, very expensive)<br>• Each additional nine costs exponentially more to achieve |
| 234 | How does redundancy improve availability? | redundancy, improve | • Multiple copies: if one fails, others continue serving<br>• Series availability of two 99.9% services: 99.9% × 99.9% = 99.8%<br>• Parallel availability: 1 - (0.001 × 0.001) = 99.9999%<br>• Apply at every level: servers, databases, data centers, regions<br>• Cost: double/triple infrastructure cost for redundancy |
| 235 | What is failover and what are the types? | failover, types | • Switching from failed component to backup/standby<br>• Hot standby: backup is running, synchronized, takes over in seconds<br>• Warm standby: backup is running but needs to catch up (minutes)<br>• Cold standby: backup exists but not running — needs manual start and data restore (hours)<br>• Auto vs manual: automated failover is faster but risks false positives (unnecessary failover) |
| 236 | What is RPO and RTO? | RPO, RTO | • RPO (Recovery Point Objective): maximum acceptable data loss (15 minutes = can lose 15 min of data)<br>• RTO (Recovery Time Objective): maximum acceptable downtime (1 hour = must recover within 1 hour)<br>• Lower RPO: needs continuous backup / synchronous replication (expensive)<br>• Lower RTO: needs hot standby / automated failover (expensive)<br>• Business determines acceptable RPO/RTO based on cost vs risk analysis |
| 237 | What is graceful degradation? | graceful degradation | • Reducing functionality instead of complete failure when component fails<br>• Cache down → serve from database (slower but works)<br>• Recommendation engine down → show popular items instead<br>• Search service down → show categories for manual browsing<br>• Principle: core functionality always works; non-essential features degrade first |
| 238 | What is a health check? | health check, endpoint | • Endpoint that reports whether a service is operational<br>• Liveness: "am I alive?" (detect hung processes → restart)<br>• Readiness: "can I serve traffic?" (detect startup in progress → don't route traffic)<br>• Deep health: checks all dependencies (DB, cache, downstream services)<br>• Used by: load balancers (routing), Kubernetes (pod management), monitoring (alerting) |
| 239 | What is the 3-2-1 backup rule? | backup, 3-2-1 | • 3 copies of your data<br>• 2 different storage media (local disk + cloud storage)<br>• 1 copy offsite (different geographic location)<br>• Protects against: hardware failure (local), site failure (offsite), media corruption (different types)<br>• Test restores regularly — untested backups might not work |
| 240 | What is chaos engineering? | chaos engineering, definition | • Discipline of intentionally introducing failures to test system resilience<br>• Hypothesis: "System handles server failure without user impact"<br>• Experiment: kill a random server in production<br>• Observe: did latency increase? Did errors spike? Did failover work?<br>• Pioneered by Netflix: Chaos Monkey kills random instances in production |
| 241 | What is the blast radius in chaos engineering? | blast radius, chaos | • The scope of impact from a chaos experiment<br>• Start small: kill one instance → affect a few requests<br>• Expand gradually: kill an entire AZ → affect an entire region's worth of traffic<br>• Always have a kill switch: stop the experiment if real user impact detected<br>• Goal: find weaknesses without causing real outages |
| 242 | What is a circuit breaker and how does it prevent cascading failures? | circuit breaker, cascading | • Monitors failure rate of calls to a downstream service<br>• When failures exceed threshold → circuit "opens" → reject all calls immediately (fail-fast)<br>• After timeout → "half-open" → allow one test request → if success, close circuit<br>• Without circuit breaker: slow service → caller waits → caller's callers wait → entire system slows<br>• With circuit breaker: immediate rejection → resources freed → system stays responsive |
| 243 | What is the retry with exponential backoff pattern? | retry, exponential backoff | • Retry failed requests with increasing delay between attempts<br>• Schedule: 1s → 2s → 4s → 8s → max (capped at 30-60 seconds)<br>• Jitter: add random delay (0-50%) to prevent thundering herd of retries<br>• Max retries: cap at 3-5 attempts then give up<br>• Without backoff: rapid retries overwhelm already-struggling service |
| 244 | What is the bulkhead pattern? | bulkhead, pattern | • Isolate resources into separate compartments (like ship bulkheads)<br>• Example: distinct thread pools for calls to payment vs user vs inventory services<br>• If payment service slow → only payment thread pool exhausted<br>• User and inventory calls continue normally (not blocked by payment)<br>• Implementation: per-dependency connection pools, thread pools, or separate instances |
| 245 | What is a timeout and why is it critical? | timeout, importance | • Maximum time to wait for a response before giving up<br>• Without timeout: hung connection holds resources forever (thread, connection, memory)<br>• Too short: legitimate slow responses treated as failures (false positives)<br>• Too long: resources held unnecessarily during real failures<br>• Rule of thumb: set timeout to P99 latency × 2-3 for that endpoint |
| 246 | What is a single point of failure (SPOF)? | SPOF, single point | • Any component whose failure brings down the entire system<br>• Examples: single database, single load balancer, single network path, single DNS server<br>• Eliminate: add redundancy at every level (replicas, multi-AZ, failover)<br>• Identify: ask "what if THIS fails?" for every component in the architecture<br>• Even redundant systems can have SPOF (single config server, single certificate) |
| 247 | What are the common causes of system outages? | outages, causes | • Configuration changes: wrong config deployed (most common — ~30%)<br>• Software bugs: unhandled edge cases in new deployments<br>• Hardware failure: disk, network, server failure<br>• Dependency failure: external API or database goes down<br>• Traffic spikes: viral event, DDoS, marketing campaign<br>• Human error: wrong command, accidental deletion |
| 248 | What is the difference between active-active and active-passive? | active-active vs passive | • Active-passive: one primary handles traffic; standby takes over on failure<br>• Active-active: all instances handle traffic simultaneously; any can handle any request<br>• Active-passive: simpler, but standby may have replication lag (data risk)<br>• Active-active: better throughput, but conflict resolution needed (write conflicts)<br>• Database: active-passive is easier; active-active needs multi-leader replication |
| 249 | How do you test disaster recovery? | DR testing, plan | • Scheduled DR drills: simulate regional outage, verify failover works<br>• Backup restore test: regularly restore from backup, verify data integrity<br>• Failover test: promote replica to primary, verify application works<br>• Runbook: documented step-by-step recovery procedures<br>• After test: document issues found, update runbook, fix gaps |
| 250 | What is the difference between high availability and disaster recovery? | HA vs DR | • HA: prevent downtime from common failures (server crash, network blip)<br>• DR: recover from catastrophic failures (data center fire, regional outage)<br>• HA: automatic failover, redundancy within same region<br>• DR: may involve manual steps, cross-region failover, data restoration<br>• Both needed: HA for everyday reliability, DR for worst-case scenarios |


| # | Question | Keywords | Answer |
|---|----------|----------|--------|
| 251 | What is a runbook? | runbook, operations | • Step-by-step documented procedure for handling operational tasks or incidents<br>• Contents: symptom → diagnosis → resolution steps → verification → escalation path<br>• Example: "Database Disk Full" runbook → check disk usage → identify large tables → archive old data<br>• Should be: specific, actionable, regularly tested, version-controlled<br>• Automated runbooks: scripts that execute steps automatically (Ansible, Terraform) |
| 252 | What is a postmortem/incident review? | postmortem, incident | • Written analysis after an incident: what happened, why, how to prevent recurrence<br>• Blameless: focus on process/system failures, not individual blame<br>• Sections: summary, impact, timeline, root cause, contributing factors, action items<br>• Action items: concrete, assigned, deadlined (not "be more careful")<br>• Share widely: team learns from every incident |
| 253 | What is the concept of error budgets? | error budget, SRE | • Allowed amount of system failure before SLO is breached<br>• Example: 99.9% SLO = 43 minutes downtime allowed per month<br>• If error budget consumed: freeze deployments, focus on reliability<br>• If error budget remaining: safe to deploy new features, take risks<br>• Helps balance: innovation speed vs system reliability |
| 254 | What is canary analysis in deployment safety? | canary analysis, deployment | • Deploy to small percentage of traffic → compare metrics with baseline<br>• Automated analysis: compare error rate, latency, resource usage of canary vs baseline<br>• If canary significantly worse → auto-rollback (no human intervention)<br>• If canary comparable or better → gradually increase traffic to new version<br>• Tools: Spinnaker (Netflix), Argo Rollouts (Kubernetes), AWS CodeDeploy |
| 255 | How do you design for zero-downtime deployments? | zero downtime, deployment | • Pattern 1: rolling update — replace instances one at a time (always some healthy instances)<br>• Pattern 2: blue-green — deploy to Green, switch traffic, Blue becomes standby<br>• Pattern 3: canary — send small % to new version, gradually increase<br>• Database: backward-compatible migrations only (add column, not remove)<br>• Key: new and old code must coexist during deployment transition |
| 256 | What is the thundering herd problem in failover? | thundering herd, failover | • Primary fails → all connections redirect to replica simultaneously<br>• Replica overwhelmed by sudden traffic spike (was handling read-only, now gets everything)<br>• Solutions: connection queuing, gradual traffic shift, pre-warmed replica<br>• Similar to cache expiry thundering herd but at infrastructure level<br>• Mitigation: replicas should be sized to handle full write+read load |
| 257 | What is data consistency after failover? | consistency, failover | • Async replication: replica may be behind primary → data written just before failure might be lost<br>• Sync replication: no data loss but higher write latency and lower availability<br>• Semi-sync: at least one replica has all data (balance of safety and speed)<br>• After failover: check for data gaps, replay WAL if possible<br>• Application: design for idempotent operations to handle re-processing |
| 258 | What is geographic redundancy? | geo redundancy, multi-region | • Deploying across multiple geographic regions for DR and low latency<br>• Active-passive: one region serves traffic, other is standby for DR<br>• Active-active: both regions serve traffic (nearest to user), write conflicts resolved<br>• Data replication: cross-region replication with higher latency (~50-200ms)<br>• DNS-based routing: Route53 health checks → failover to healthy region |
| 259 | What is immutable infrastructure? | immutable, infrastructure | • Servers are never modified after deployment — replaced entirely for updates<br>• Deploy new version → create new servers → switch traffic → destroy old servers<br>• Benefits: no configuration drift, reproducible deployments, easy rollback<br>• Tools: Docker (immutable images), Terraform (infrastructure as code), Packer (machine images)<br>• Contrast: mutable infrastructure — SSH in, update packages, restart (configuration drift risk) |
| 260 | Scenario: Design a system with 99.99% availability. What components are needed? | scenario, four nines | • 99.99% = 52 minutes downtime per year — very demanding<br>• Load balancers: at least 2, in active-passive or active-active<br>• App servers: minimum 3 across 2+ availability zones, auto-scaling<br>• Database: primary + sync replica (RPO=0), auto-failover<br>• Caching: Redis cluster with replicas (tolerate node failures)<br>• Monitoring: real-time alerting, automated remediation<br>• Deployment: blue-green or canary (zero-downtime deploys)<br>• Testing: regular chaos engineering, DR drills quarterly<br>• On-call: 24/7 engineering rotation with escalation policy |


---
<a id="high-availability"></a>
# Section 6: High Availability & Reliability

High availability (HA) means a system remains operational and accessible for a very high percentage of time. Reliability means it continues to work correctly even when things go wrong.


## 6.1 Availability Numbers (The "Nines")

| Availability | Downtime/Year | Downtime/Month | Downtime/Week |
|-------------|---------------|----------------|---------------|
| **99%** (two nines) | 3.65 days | 7.31 hours | 1.68 hours |
| **99.9%** (three nines) | 8.77 hours | 43.83 minutes | 10.08 minutes |
| **99.95%** | 4.38 hours | 21.92 minutes | 5.04 minutes |
| **99.99%** (four nines) | 52.60 minutes | 4.38 minutes | 1.01 minutes |
| **99.999%** (five nines) | 5.26 minutes | 26.30 seconds | 6.05 seconds |

### Availability in Series vs Parallel

```
Series (ALL must work):
  Availability = A1 × A2 × A3
  Example: 99.9% × 99.9% × 99.9% = 99.7% (three dependent services)

Parallel (ANY can serve):
  Availability = 1 - (1-A1)(1-A2)
  Example: 1 - (1-0.999)(1-0.999) = 1 - 0.000001 = 99.9999%

Implication: redundancy dramatically improves availability
  One server at 99.9% = 8.77 hours downtime/year
  Two servers at 99.9% each = 31.5 seconds downtime/year
```

## 6.2 Fault Tolerance

| Concept | Description |
|---------|-------------|
| **Fault** | Something goes wrong (disk dies, network drops, bug triggered) |
| **Error** | Fault manifests as incorrect behavior (wrong response, timeout) |
| **Failure** | System stops providing required service (complete outage) |

### Fault Tolerance Strategies

| Strategy | How It Works | Example |
|----------|-------------|---------|
| **Redundancy** | Multiple copies of everything | 3 app servers, database replicas, multi-AZ |
| **Replication** | Copy data across nodes | PostgreSQL streaming replication, Kafka partition replicas |
| **Failover** | Detect failure, switch to standby | Primary DB fails → promote replica to primary |
| **Graceful Degradation** | Reduce functionality instead of complete failure | Cache fails → serve from DB (slower but works) |
| **Retry with Backoff** | Retry failed operations with increasing delay | API call fails → retry after 1s, 2s, 4s |
| **Circuit Breaker** | Stop calling failing service | Payment service down → return "try later" instead of hanging |
| **Timeout** | Don't wait forever for response | DB query timeout after 5s → return error |
| **Bulkhead** | Isolate resources per dependency | Separate thread pools for each downstream service |


## 6.3 Redundancy & Failover

### Redundancy Levels

| Level | What's Redundant | Example |
|-------|-----------------|---------|
| **Server** | Multiple app servers behind load balancer | 3 web servers, any can serve requests |
| **Database** | Primary + replicas | PostgreSQL primary + 2 replicas |
| **Data Center** | Multiple availability zones | AWS: us-east-1a, us-east-1b, us-east-1c |
| **Region** | Multiple geographic regions | US-East + EU-West (disaster recovery) |
| **Network** | Multiple network paths | Dual ISP connections, BGP failover |

### Failover Types

| Type | Description | RPO | RTO |
|------|-------------|-----|-----|
| **Hot Standby** | Standby is running, data synchronized, ready to take over instantly | ~0 (real-time sync) | Seconds |
| **Warm Standby** | Standby running but needs to catch up on latest data | Minutes | Minutes |
| **Cold Standby** | Standby exists but not running — needs to be started and data restored | Hours | Hours |

> **RPO** = Recovery Point Objective — how much data can you afford to lose?
> **RTO** = Recovery Time Objective — how long can the system be down?

### Automatic Failover Flow

```
Normal:  Client → LB → Primary DB    Replica DB (warm standby)
                        ↓ WAL stream →     ↓
                   [writes + reads]    [replication]

Primary dies:
  1. Health check fails (no response for 30 seconds)
  2. LB marks primary as unhealthy
  3. Replica promoted to primary (auto or manual)
  4. Application config updated to point to new primary
  5. Old primary removed from rotation

Risk: "split brain" if old primary comes back — use fencing (old primary rejects writes)
```


## 6.4 Health Checks

### Health Check Types

| Type | Checks | Response Time | When to Use |
|------|--------|---------------|-------------|
| **Liveness** | Is the process alive? | < 1 second | Always — detects hung/crashed processes |
| **Readiness** | Can the service handle traffic? | < 2 seconds | After startup, during maintenance |
| **Startup** | Has the service finished initialization? | < 60 seconds | Services with slow startup (loading ML models) |
| **Deep/Dependency** | Are all dependencies healthy? | < 5 seconds | Monitoring dashboards, debugging |

```python
# Example health check endpoint
@app.get("/health")
def health():
    return {"status": "ok"}  # Liveness only

@app.get("/health/ready")
def ready():
    checks = {
        "database": check_db_connection(),      # Can connect to PostgreSQL?
        "cache": check_redis_connection(),       # Can connect to Redis?
        "embedding_model": model_loaded(),       # Is ML model loaded?
    }
    all_healthy = all(checks.values())
    return {"status": "ready" if all_healthy else "not_ready", "checks": checks}
```

## 6.5 Backup & Disaster Recovery

### Backup Strategies

| Strategy | Description | RPO | Storage Cost |
|----------|-------------|-----|-------------|
| **Full Backup** | Copy entire database | Time since last backup | Highest |
| **Incremental** | Only changes since last backup | Lower (more frequent) | Lower |
| **Differential** | Changes since last FULL backup | Medium | Medium |
| **Continuous (WAL)** | Stream every write to backup | Near-zero | Medium-High |
| **Snapshot** | Point-in-time copy (filesystem or VM level) | Time since last snapshot | Depends |

### Disaster Recovery Tiers

| Tier | Strategy | RPO | RTO | Cost |
|------|----------|-----|-----|------|
| **0** | No DR | Total loss | N/A | $0 |
| **1** | Offsite backup | Hours-Days | Hours-Days | Low |
| **2** | Warm standby | Minutes | Minutes | Medium |
| **3** | Hot standby (multi-region active-passive) | Near-zero | Seconds | High |
| **4** | Active-active multi-region | Zero | Zero | Very High |

### 3-2-1 Backup Rule

```
3 copies of your data
2 different storage types (disk + cloud)
1 offsite copy (different geographic location)
```


## 6.6 Chaos Engineering

Intentionally introducing failures to test system resilience.

### Chaos Engineering Principles

| Principle | Description |
|-----------|-------------|
| **Build a hypothesis** | "If we kill one app server, latency stays under 200ms" |
| **Run in production** | Staging doesn't catch everything — real traffic reveals real issues |
| **Minimize blast radius** | Start small (one server, one user), gradually increase scope |
| **Automate experiments** | Run chaos tests continuously, not just once |
| **Stop on real damage** | If experiment causes user-visible outage, stop immediately |

### Chaos Experiments

| Experiment | What It Tests | Tool |
|------------|--------------|------|
| **Kill a server** | Failover, load balancing, auto-scaling | Chaos Monkey |
| **Inject network latency** | Timeout handling, circuit breakers | Toxiproxy, tc (traffic control) |
| **Fill disk** | Disk full handling, alerting | Custom script |
| **Corrupt DNS** | DNS failover, caching | Custom |
| **Kill a database** | Failover, data integrity, reconnection | Litmus Chaos, Gremlin |
| **Simulate region outage** | Multi-region failover | GameDay (AWS) |

### Netflix Simian Army

| Tool | What It Does |
|------|-------------|
| **Chaos Monkey** | Randomly kills instances in production |
| **Chaos Gorilla** | Simulates entire AZ failure |
| **Chaos Kong** | Simulates entire region failure |
| **Latency Monkey** | Adds artificial network latency |
| **Janitor Monkey** | Cleans up unused resources |


---
## Section 6 — Interview Questions: High Availability & Reliability (60 Questions)


| # | Question | Keywords | Answer |
|---|----------|----------|--------|
| 231 | What is high availability? | HA, definition | • System remains operational for a very high percentage of time (99.9%+)<br>• Measured in "nines": 99.99% = 52 minutes downtime per year<br>• Achieved through: redundancy, failover, load balancing, monitoring<br>• Not the same as fault tolerance (HA = minimize downtime; FT = continue working despite faults)<br>• Business impact: every hour of downtime costs revenue and trust |
| 232 | What is the difference between availability and reliability? | availability vs reliability | • Availability: percentage of time the system is operational (uptime/total time)<br>• Reliability: probability the system works correctly over a period (no errors, correct results)<br>• A system can be available but unreliable (returns wrong data 5% of the time)<br>• A system can be reliable but unavailable (works perfectly when up but has frequent outages)<br>• Both needed: available AND reliable |
| 233 | What do "nines" of availability mean? | nines, availability | • 99% = 3.65 days downtime/year (two nines — unacceptable for most services)<br>• 99.9% = 8.77 hours/year (three nines — acceptable for internal tools)<br>• 99.99% = 52 minutes/year (four nines — standard for most SaaS)<br>• 99.999% = 5.26 minutes/year (five nines — mission critical, very expensive)<br>• Each additional nine costs exponentially more to achieve |
| 234 | How does redundancy improve availability? | redundancy, improve | • Multiple copies: if one fails, others continue serving<br>• Series availability of two 99.9% services: 99.9% × 99.9% = 99.8%<br>• Parallel availability: 1 - (0.001 × 0.001) = 99.9999%<br>• Apply at every level: servers, databases, data centers, regions<br>• Cost: double/triple infrastructure cost for redundancy |
| 235 | What is failover and what are the types? | failover, types | • Switching from failed component to backup/standby<br>• Hot standby: backup is running, synchronized, takes over in seconds<br>• Warm standby: backup is running but needs to catch up (minutes)<br>• Cold standby: backup exists but not running — needs manual start and data restore (hours)<br>• Auto vs manual: automated failover is faster but risks false positives (unnecessary failover) |
| 236 | What is RPO and RTO? | RPO, RTO | • RPO (Recovery Point Objective): maximum acceptable data loss (15 minutes = can lose 15 min of data)<br>• RTO (Recovery Time Objective): maximum acceptable downtime (1 hour = must recover within 1 hour)<br>• Lower RPO: needs continuous backup / synchronous replication (expensive)<br>• Lower RTO: needs hot standby / automated failover (expensive)<br>• Business determines acceptable RPO/RTO based on cost vs risk analysis |
| 237 | What is graceful degradation? | graceful degradation | • Reducing functionality instead of complete failure when component fails<br>• Cache down → serve from database (slower but works)<br>• Recommendation engine down → show popular items instead<br>• Search service down → show categories for manual browsing<br>• Principle: core functionality always works; non-essential features degrade first |
| 238 | What is a health check? | health check, endpoint | • Endpoint that reports whether a service is operational<br>• Liveness: "am I alive?" (detect hung processes → restart)<br>• Readiness: "can I serve traffic?" (detect startup in progress → don't route traffic)<br>• Deep health: checks all dependencies (DB, cache, downstream services)<br>• Used by: load balancers (routing), Kubernetes (pod management), monitoring (alerting) |
| 239 | What is the 3-2-1 backup rule? | backup, 3-2-1 | • 3 copies of your data<br>• 2 different storage media (local disk + cloud storage)<br>• 1 copy offsite (different geographic location)<br>• Protects against: hardware failure (local), site failure (offsite), media corruption (different types)<br>• Test restores regularly — untested backups might not work |
| 240 | What is chaos engineering? | chaos engineering, definition | • Discipline of intentionally introducing failures to test system resilience<br>• Hypothesis: "System handles server failure without user impact"<br>• Experiment: kill a random server in production<br>• Observe: did latency increase? Did errors spike? Did failover work?<br>• Pioneered by Netflix: Chaos Monkey kills random instances in production |
| 241 | What is the blast radius in chaos engineering? | blast radius, chaos | • The scope of impact from a chaos experiment<br>• Start small: kill one instance → affect a few requests<br>• Expand gradually: kill an entire AZ → affect an entire region's worth of traffic<br>• Always have a kill switch: stop the experiment if real user impact detected<br>• Goal: find weaknesses without causing real outages |
| 242 | What is a circuit breaker and how does it prevent cascading failures? | circuit breaker, cascading | • Monitors failure rate of calls to a downstream service<br>• When failures exceed threshold → circuit "opens" → reject all calls immediately (fail-fast)<br>• After timeout → "half-open" → allow one test request → if success, close circuit<br>• Without circuit breaker: slow service → caller waits → caller's callers wait → entire system slows<br>• With circuit breaker: immediate rejection → resources freed → system stays responsive |
| 243 | What is the retry with exponential backoff pattern? | retry, exponential backoff | • Retry failed requests with increasing delay between attempts<br>• Schedule: 1s → 2s → 4s → 8s → max (capped at 30-60 seconds)<br>• Jitter: add random delay (0-50%) to prevent thundering herd of retries<br>• Max retries: cap at 3-5 attempts then give up<br>• Without backoff: rapid retries overwhelm already-struggling service |
| 244 | What is the bulkhead pattern? | bulkhead, pattern | • Isolate resources into separate compartments (like ship bulkheads)<br>• Example: distinct thread pools for calls to payment vs user vs inventory services<br>• If payment service slow → only payment thread pool exhausted<br>• User and inventory calls continue normally (not blocked by payment)<br>• Implementation: per-dependency connection pools, thread pools, or separate instances |
| 245 | What is a timeout and why is it critical? | timeout, importance | • Maximum time to wait for a response before giving up<br>• Without timeout: hung connection holds resources forever (thread, connection, memory)<br>• Too short: legitimate slow responses treated as failures (false positives)<br>• Too long: resources held unnecessarily during real failures<br>• Rule of thumb: set timeout to P99 latency × 2-3 for that endpoint |
| 246 | What is a single point of failure (SPOF)? | SPOF, single point | • Any component whose failure brings down the entire system<br>• Examples: single database, single load balancer, single network path, single DNS server<br>• Eliminate: add redundancy at every level (replicas, multi-AZ, failover)<br>• Identify: ask "what if THIS fails?" for every component in the architecture<br>• Even redundant systems can have SPOF (single config server, single certificate) |
| 247 | What are the common causes of system outages? | outages, causes | • Configuration changes: wrong config deployed (most common — ~30%)<br>• Software bugs: unhandled edge cases in new deployments<br>• Hardware failure: disk, network, server failure<br>• Dependency failure: external API or database goes down<br>• Traffic spikes: viral event, DDoS, marketing campaign<br>• Human error: wrong command, accidental deletion |
| 248 | What is the difference between active-active and active-passive? | active-active vs passive | • Active-passive: one primary handles traffic; standby takes over on failure<br>• Active-active: all instances handle traffic simultaneously; any can handle any request<br>• Active-passive: simpler, but standby may have replication lag (data risk)<br>• Active-active: better throughput, but conflict resolution needed (write conflicts)<br>• Database: active-passive is easier; active-active needs multi-leader replication |
| 249 | How do you test disaster recovery? | DR testing, plan | • Scheduled DR drills: simulate regional outage, verify failover works<br>• Backup restore test: regularly restore from backup, verify data integrity<br>• Failover test: promote replica to primary, verify application works<br>• Runbook: documented step-by-step recovery procedures<br>• After test: document issues found, update runbook, fix gaps |
| 250 | What is the difference between high availability and disaster recovery? | HA vs DR | • HA: prevent downtime from common failures (server crash, network blip)<br>• DR: recover from catastrophic failures (data center fire, regional outage)<br>• HA: automatic failover, redundancy within same region<br>• DR: may involve manual steps, cross-region failover, data restoration<br>• Both needed: HA for everyday reliability, DR for worst-case scenarios |


| # | Question | Keywords | Answer |
|---|----------|----------|--------|
| 251 | What is a runbook? | runbook, operations | • Step-by-step documented procedure for handling operational tasks or incidents<br>• Contents: symptom → diagnosis → resolution steps → verification → escalation path<br>• Example: "Database Disk Full" runbook → check disk usage → identify large tables → archive old data<br>• Should be: specific, actionable, regularly tested, version-controlled<br>• Automated runbooks: scripts that execute steps automatically (Ansible, Terraform) |
| 252 | What is a postmortem/incident review? | postmortem, incident | • Written analysis after an incident: what happened, why, how to prevent recurrence<br>• Blameless: focus on process/system failures, not individual blame<br>• Sections: summary, impact, timeline, root cause, contributing factors, action items<br>• Action items: concrete, assigned, deadlined (not "be more careful")<br>• Share widely: team learns from every incident |
| 253 | What is the concept of error budgets? | error budget, SRE | • Allowed amount of system failure before SLO is breached<br>• Example: 99.9% SLO = 43 minutes downtime allowed per month<br>• If error budget consumed: freeze deployments, focus on reliability<br>• If error budget remaining: safe to deploy new features, take risks<br>• Helps balance: innovation speed vs system reliability |
| 254 | What is canary analysis in deployment safety? | canary analysis, deployment | • Deploy to small percentage of traffic → compare metrics with baseline<br>• Automated analysis: compare error rate, latency, resource usage of canary vs baseline<br>• If canary significantly worse → auto-rollback (no human intervention)<br>• If canary comparable or better → gradually increase traffic to new version<br>• Tools: Spinnaker (Netflix), Argo Rollouts (Kubernetes), AWS CodeDeploy |
| 255 | How do you design for zero-downtime deployments? | zero downtime, deployment | • Pattern 1: rolling update — replace instances one at a time (always some healthy instances)<br>• Pattern 2: blue-green — deploy to Green, switch traffic, Blue becomes standby<br>• Pattern 3: canary — send small % to new version, gradually increase<br>• Database: backward-compatible migrations only (add column, not remove)<br>• Key: new and old code must coexist during deployment transition |
| 256 | What is the thundering herd problem in failover? | thundering herd, failover | • Primary fails → all connections redirect to replica simultaneously<br>• Replica overwhelmed by sudden traffic spike (was handling read-only, now gets everything)<br>• Solutions: connection queuing, gradual traffic shift, pre-warmed replica<br>• Similar to cache expiry thundering herd but at infrastructure level<br>• Mitigation: replicas should be sized to handle full write+read load |
| 257 | What is data consistency after failover? | consistency, failover | • Async replication: replica may be behind primary → data written just before failure might be lost<br>• Sync replication: no data loss but higher write latency and lower availability<br>• Semi-sync: at least one replica has all data (balance of safety and speed)<br>• After failover: check for data gaps, replay WAL if possible<br>• Application: design for idempotent operations to handle re-processing |
| 258 | What is geographic redundancy? | geo redundancy, multi-region | • Deploying across multiple geographic regions for DR and low latency<br>• Active-passive: one region serves traffic, other is standby for DR<br>• Active-active: both regions serve traffic (nearest to user), write conflicts resolved<br>• Data replication: cross-region replication with higher latency (~50-200ms)<br>• DNS-based routing: Route53 health checks → failover to healthy region |
| 259 | What is immutable infrastructure? | immutable, infrastructure | • Servers are never modified after deployment — replaced entirely for updates<br>• Deploy new version → create new servers → switch traffic → destroy old servers<br>• Benefits: no configuration drift, reproducible deployments, easy rollback<br>• Tools: Docker (immutable images), Terraform (infrastructure as code), Packer (machine images)<br>• Contrast: mutable infrastructure — SSH in, update packages, restart (configuration drift risk) |
| 260 | Scenario: Design a system with 99.99% availability. What components are needed? | scenario, four nines | • 99.99% = 52 minutes downtime per year — very demanding<br>• Load balancers: at least 2, in active-passive or active-active<br>• App servers: minimum 3 across 2+ availability zones, auto-scaling<br>• Database: primary + sync replica (RPO=0), auto-failover<br>• Caching: Redis cluster with replicas (tolerate node failures)<br>• Monitoring: real-time alerting, automated remediation<br>• Deployment: blue-green or canary (zero-downtime deploys)<br>• Testing: regular chaos engineering, DR drills quarterly<br>• On-call: 24/7 engineering rotation with escalation policy |


---
<a id="design-patterns"></a>
# Section 7: Architectural Design Patterns

Architecture patterns define how software components are organized and interact. Choosing the right pattern impacts scalability, maintainability, and testability.


## 7.1 Common Architecture Patterns

### Pattern Comparison

| Pattern | Structure | Pros | Cons | Best For |
|---------|-----------|------|------|----------|
| **Layered (N-tier)** | Presentation → Business → Persistence → Database | Simple, well-understood | Tight coupling, changes cascade | Traditional web apps |
| **Clean Architecture** | Entities → Use Cases → Adapters → Frameworks | Testable, framework-independent | Boilerplate, over-engineering for small projects | Complex business logic |
| **Hexagonal (Ports & Adapters)** | Core domain ← Ports → Adapters | Swappable adapters, testable | More upfront design | Services with multiple integrations |
| **Event-Driven** | Producers → Event Bus → Consumers | Loose coupling, scalable | Hard to debug, eventual consistency | Real-time systems, microservices |
| **CQRS** | Separate read/write models | Optimized read/write, scalable | Complexity, eventual consistency | Read-heavy with complex queries |
| **Microkernel (Plugin)** | Core + plugins | Extensible, modular | Plugin interface design | IDEs, browsers, extensible platforms |

### Layered Architecture

```
┌────────────────────────────────┐
│   Presentation Layer (UI/API)  │  Controller, Views, API handlers
├────────────────────────────────┤
│   Business Logic Layer         │  Services, use cases, validation
├────────────────────────────────┤
│   Persistence Layer (DAL)      │  Repositories, ORM, queries
├────────────────────────────────┤
│   Database Layer               │  PostgreSQL, Redis, S3
└────────────────────────────────┘

Rules:
  - Each layer only communicates with the layer directly below it
  - No "skipping" layers (UI should NOT directly access database)
  - Changes in one layer shouldn't affect other layers
```

### Clean Architecture (Uncle Bob)

```
                    ┌──────────────────────┐
                    │   Frameworks (outer)  │  Express, Django, PostgreSQL driver
                    │  ┌──────────────────┐ │
                    │  │ Interface Adapters│ │  Controllers, Presenters, Gateways
                    │  │ ┌──────────────┐ │ │
                    │  │ │  Use Cases    │ │ │  Application business rules
                    │  │ │ ┌──────────┐ │ │ │
                    │  │ │ │ Entities  │ │ │ │  Enterprise business rules (core domain)
                    │  │ │ └──────────┘ │ │ │
                    │  │ └──────────────┘ │ │
                    │  └──────────────────┘ │
                    └──────────────────────┘

Dependency Rule: Dependencies point INWARD only
  - Entities know nothing about use cases, frameworks, or DB
  - Use cases know about entities but not about frameworks
  - Frameworks depend on everything but nothing depends on them
```

### Hexagonal Architecture (Ports & Adapters)

```
          ┌─────────────────────────────────┐
REST API ─│ Port: UserAPI                    │
          │   ┌───────────────────────┐      │
gRPC ─────│   │  Domain Core          │      │─── PostgreSQL Adapter
          │   │  (Business Logic)     │      │
GraphQL ──│   │  Pure, no dependencies│      │─── Redis Adapter
          │   └───────────────────────┘      │
Message ──│ Port: EventConsumer              │─── Kafka Adapter
          └─────────────────────────────────┘

Ports: Interfaces that define how external world interacts with core
Adapters: Implementations of ports (REST adapter, DB adapter)
Core: Pure business logic — no frameworks, no I/O
Swap: Switch PostgreSQL to MongoDB by changing adapter only
```


## 7.2 Domain-Driven Design (DDD)

### DDD Key Concepts

| Concept | Description | Example |
|---------|-------------|---------|
| **Domain** | The subject area the software is built for | E-commerce, healthcare, finance |
| **Ubiquitous Language** | Shared vocabulary between devs and domain experts | "Order", "Shipment", "Invoice" (not "data record") |
| **Bounded Context** | Clear boundary around a domain model | "Ordering" context vs "Shipping" context |
| **Entity** | Object with unique identity that persists over time | User (identified by user_id), Order (order_id) |
| **Value Object** | Object defined by its attributes, no identity | Money (amount+currency), Address (street+city+zip) |
| **Aggregate** | Cluster of entities treated as a single unit | Order aggregate (Order + OrderLineItems + ShippingInfo) |
| **Aggregate Root** | Entry point to the aggregate (all access goes through it) | Order is the root; can't access OrderLineItem directly |
| **Domain Event** | Something significant that happened in the domain | OrderPlaced, PaymentReceived, ShipmentDelivered |
| **Repository** | Abstraction for data access (hides DB details from domain) | OrderRepository.find_by_id(), OrderRepository.save() |

### Bounded Context Mapping

```
┌──────────────────┐     ┌──────────────────┐
│  Ordering Context │     │ Shipping Context  │
│  - Order          │     │ - Shipment        │
│  - OrderItem      │     │ - TrackingNumber  │
│  - Customer       │──→──│ - DeliveryAddress  │
│  (Customer = full │     │ (Customer = just   │
│   profile, prefs) │     │  name + address)   │
└──────────────────┘     └──────────────────┘

Same real-world concept "Customer" has different models in different contexts
  - Ordering: full profile, preferences, payment info
  - Shipping: just delivery name and address
  - Marketing: email, engagement history, segments

Anti-Corruption Layer: translates between contexts
```

### DDD Strategic vs Tactical Patterns

| Level | Patterns | Focus |
|-------|----------|-------|
| **Strategic** | Bounded Contexts, Context Maps, Ubiquitous Language | High-level architecture, team boundaries |
| **Tactical** | Entities, Value Objects, Aggregates, Repositories, Domain Events | Code-level design within a bounded context |


---
## Section 7 — Interview Questions: Design Patterns & Architecture (40 Questions)


| # | Question | Keywords | Answer |
|---|----------|----------|--------|
| 261 | What is layered architecture? | layered, n-tier | • Organizes code into horizontal layers: Presentation → Business → Persistence → Database<br>• Each layer only communicates with the layer directly below it<br>• Benefits: separation of concerns, easy to understand, widely known<br>• Drawbacks: tight coupling between layers, changes cascade down, monolithic tendency<br>• Best for: traditional CRUD applications, small-medium projects |
| 262 | What is clean architecture? | clean architecture, Uncle Bob | • Concentric circles: Entities (center) → Use Cases → Adapters → Frameworks (outer)<br>• Dependency rule: dependencies point INWARD only (inner layers know nothing about outer layers)<br>• Entities: core business rules. Use Cases: application-specific logic<br>• Frameworks and databases are details (outermost layer, easily swappable)<br>• Benefits: testable without frameworks, technology-agnostic core |
| 263 | What is hexagonal architecture? | hexagonal, ports adapters | • Core domain logic surrounded by ports (interfaces) and adapters (implementations)<br>• Ports: define how external world interacts with core (input/output interfaces)<br>• Adapters: implementations of ports (REST adapter, PostgreSQL adapter, Kafka adapter)<br>• Core has zero dependencies on external frameworks<br>• Swap technology by changing adapter only (no core changes) |
| 264 | What is Domain-Driven Design (DDD)? | DDD, domain driven | • Design methodology focused on the core business domain and its logic<br>• Ubiquitous Language: shared vocabulary between developers and domain experts<br>• Bounded Contexts: clear boundaries around domain models<br>• Strategic patterns: context mapping, bounded contexts<br>• Tactical patterns: entities, value objects, aggregates, repositories, domain events |
| 265 | What is a Bounded Context? | bounded context, DDD | • Clear boundary around a specific domain model within which terms have specific meaning<br>• Example: "Customer" in Ordering = full profile; "Customer" in Shipping = just name + address<br>• Each bounded context can have its own database, team, and technology<br>• Communication between contexts via APIs or events (not shared database)<br>• Maps directly to microservice boundaries in many organizations |
| 266 | What is an Aggregate in DDD? | aggregate, DDD | • Cluster of related entities treated as a single transactional unit<br>• Aggregate Root: the entry point — all access goes through it<br>• Example: Order aggregate = Order (root) + OrderItems + ShippingInfo<br>• Rule: only the aggregate root is referenced from outside the aggregate<br>• Updates are atomic within an aggregate (ACID within, eventual consistency between aggregates) |
| 267 | What is the difference between Entity and Value Object? | entity vs value object | • Entity: has unique identity, persists over time (User identified by user_id)<br>  Two users with same name are different entities if different IDs<br>• Value Object: defined by attributes, no identity (Money = amount + currency)<br>  Two Money objects with same amount and currency are equal (interchangeable)<br>• Entities are mutable (user changes name, same entity). Value Objects are immutable (create new) |
| 268 | What is a Domain Event? | domain event, DDD | • Notification that something important happened in the domain<br>• Examples: OrderPlaced, PaymentReceived, UserRegistered<br>• Named in past tense (it already happened, it's a fact)<br>• Contains relevant data: {event: "OrderPlaced", order_id: 123, total: 500}<br>• Used for: inter-aggregate communication, event sourcing, audit logging |
| 269 | What is the Repository pattern? | repository, pattern | • Abstraction over data access — domain code talks to repository, not directly to database<br>• Interface: OrderRepository.find_by_id(id), OrderRepository.save(order)<br>• Implementation: PostgresOrderRepository, MongoOrderRepository (swappable)<br>• Benefits: testable (mock repository in tests), database-agnostic domain logic<br>• Part of DDD but widely used beyond DDD |
| 270 | Compare monolithic vs microservices architecture. | monolith vs microservices | • Monolith: single deployable unit containing all features → simple, fast in-process calls<br>• Microservices: independent services per domain → complex, network calls, but independently deployable<br>• Monolith: shared database, single tech stack, one team can manage initially<br>• Microservices: database per service, polyglot, small autonomous teams<br>• Start monolith → extract microservices as complexity grows (Strangler Fig pattern) |
| 271 | What is event storming? | event storming, workshop | • Collaborative workshop technique to discover domain events and business processes<br>• Participants: developers + domain experts + stakeholders<br>• Orange sticky notes: domain events (past tense: "OrderPlaced")<br>• Blue: commands (imperative: "PlaceOrder")<br>• Yellow: aggregates (business entities processing commands)<br>• Output: shared understanding of domain → informs bounded context boundaries |
| 272 | What is the SOLID principle in system design? | SOLID, principles | • S: Single Responsibility — each service/module does one thing well<br>• O: Open/Closed — extensible without modifying existing code (plugins, events)<br>• L: Liskov Substitution — substitutable implementations (interface-based design)<br>• I: Interface Segregation — small, focused interfaces (not one massive API)<br>• D: Dependency Inversion — depend on abstractions, not concrete implementations |
| 273 | What is the Strangler Fig pattern? | strangler fig, migration | • Incrementally replace a legacy system with new services<br>• Route new features to new service; old features still in legacy<br>• Over time: migrate more features until legacy is empty → retire it<br>• Named after strangler fig tree that grows around host tree<br>• Low risk: incremental, no big-bang rewrite, coexistence during transition |
| 274 | What is the Anti-Corruption Layer? | ACL, DDD | • Translation layer between new service and legacy/external system<br>• Prevents legacy concepts from "corrupting" the new service's domain model<br>• New service speaks its own language ← ACL translates → legacy format<br>• Example: new order service uses "ShoppingCart"; legacy uses "Basket" with different structure<br>• DDD concept: keeps bounded contexts clean and independent |
| 275 | What is the difference between a service and a module? | service vs module, architecture | • Module: code boundary within a process (package, namespace, library)<br>  Communication: function calls (nanoseconds), shared memory<br>• Service: deployable process with its own runtime<br>  Communication: network calls (milliseconds), serialized data<br>• Module: simpler, lower latency, but coupled deployment<br>• Service: independent deployment, scaling, technology — but operational overhead |


| # | Question | Keywords | Answer |
|---|----------|----------|--------|
| 276 | What is the Twelve-Factor App methodology? | twelve factor, methodology | • Set of principles for building modern cloud-native applications<br>• I: Codebase (one repo per app), II: Dependencies (explicit), III: Config (env vars)<br>• IV: Backing services (treat as attached resources), V: Build/Release/Run (separate stages)<br>• VI: Processes (stateless), VII: Port binding (self-contained), VIII: Concurrency (scale via processes)<br>• IX: Disposability (fast startup/shutdown), X: Dev/prod parity, XI: Logs (treat as streams), XII: Admin processes |
| 277 | What is Conway's Law? | Conway's Law, organization | • "Organizations design systems that mirror their communication structures"<br>• Three teams → system tends to have three major components<br>• Implication: to build microservices, organize into small, autonomous teams<br>• Inverse Conway Maneuver: deliberately structure teams to get desired architecture<br>• Very relevant for microservices: team boundaries = service boundaries |
| 278 | What is the difference between vertical and horizontal slicing? | vertical vs horizontal, architecture | • Horizontal slice: layer-based teams (frontend team, backend team, DB team<br>  → cross-team coordination for every feature → slow delivery<br>• Vertical slice: feature-based teams own all layers for their domain<br>  → each team independently delivers features → faster, more autonomous<br>• Microservices prefer vertical slicing (team owns entire service: API + logic + DB) |
| 279 | What is a modular monolith? | modular monolith, architecture | • Monolithic application with clear module boundaries and well-defined interfaces<br>• Each module owns its data (separate database schemas or tables)<br>• Communication between modules via internal APIs (not direct DB access)<br>• Benefits: simplicity of monolith + modularity of microservices<br>• Easy migration path: extract modules to microservices if needed |
| 280 | What is service-oriented architecture (SOA) vs microservices? | SOA vs microservices | • SOA: enterprise services communicating via ESB (Enterprise Service Bus) — heavy middleware<br>• Microservices: lightweight services communicating via HTTP/gRPC/events — no ESB<br>• SOA: shared database is acceptable; microservices: database per service<br>• SOA: heavyweight (SOAP, XML, WS-* standards); microservices: lightweight (REST, JSON, Protobuf)<br>• Microservices evolved from SOA with DevOps and container technologies |
| 281 | What is infrastructure as code? | IaC, infrastructure | • Managing infrastructure through code files rather than manual configuration<br>• Declarative: describe desired state, tool figures out how (Terraform, CloudFormation)<br>• Imperative: step-by-step instructions (Ansible, scripts)<br>• Benefits: version-controlled, reproducible, auditable, testable<br>• GitOps: infrastructure changes via pull requests → reviewed → merged → applied |
| 282 | What is the sidecar pattern vs ambassador pattern? | sidecar vs ambassador | • Sidecar: helper container deployed alongside main service for cross-cutting concerns<br>  Example: Envoy sidecar for service mesh (mTLS, tracing, retries)<br>• Ambassador: proxy for outbound connections (connection pooling, routing, TLS to external services)<br>  Example: ambassador handles complex database connection logic<br>• Both deploy alongside service in same pod/VM; differ in direction (inbound vs outbound) |
| 283 | What is the Backend-for-Frontend (BFF) pattern? | BFF, pattern | • Dedicated backend for each client type (mobile, web, third-party)<br>• Each BFF: aggregates microservice calls, transforms data for specific client needs<br>• Mobile BFF: smaller payloads, optimized for bandwidth<br>• Web BFF: richer data, supports server-side rendering<br>• Avoids: one-size-fits-all API that satisfies no client perfectly |
| 284 | What is an event-driven architecture pattern? | EDA, pattern | • Services communicate via events (notifications of state changes)<br>• Producer publishes event → event bus (Kafka, SNS) → consumers react<br>• Decoupling: producer doesn't know or care who consumes<br>• Scalability: consumers process independently, can be scaled separately<br>• Challenge: eventual consistency, debugging complex event chains |
| 285 | Scenario: You're building a new system — how do you choose the architecture pattern? | scenario, architecture choice | • Small team, simple domain → monolith or modular monolith<br>• Complex business domain → DDD + clean architecture<br>• Multiple teams, independent features → microservices<br>• High read/write ratio imbalance → CQRS<br>• Real-time processing, event-driven workflows → event-driven architecture<br>• Many external integrations → hexagonal (ports & adapters) for swappable adapters<br>• Start simple (monolith) → evolve toward microservices as complexity and team size grow |


---
<a id="design-problems"></a>
# Section 8: Classic System Design Interview Problems

This section covers the most commonly asked system design problems with detailed approach, architecture, and trade-off analysis.


## 8.1 How to Approach a System Design Interview

### The Framework (4 Steps, 35-45 minutes)

| Step | Time | Activities |
|------|------|-----------|
| **1. Requirements Clarification** | 5 min | Ask questions: who are the users? What are core features? What scale? Read-heavy or write-heavy? |
| **2. High-Level Design** | 10 min | Draw architecture: clients, load balancer, API servers, database, cache, message queue |
| **3. Deep Dive** | 15 min | Detail critical components: database schema, API design, data flow, scaling bottlenecks |
| **4. Trade-offs & Bottlenecks** | 5 min | Discuss: single points of failure, scaling limits, consistency vs availability, cost vs performance |

### Estimation Cheat Sheet

| Metric | Value |
|--------|-------|
| 1 day | 86,400 seconds ≈ 100K seconds |
| 1 month | ~2.5 million seconds |
| QPS from DAU | DAU × queries/user / 86,400 |
| Peak QPS | Average QPS × 2-5 |
| Storage per year | Daily new data × 365 |
| Text message | ~100 bytes |
| Metadata per image/video | ~1 KB |
| Image (compressed) | ~200 KB |
| Short video (1 min) | ~5 MB |
| 1M users × 1 KB each | 1 GB |

### Back-of-the-Envelope Calculation Template

```
Given: 100M DAU, each user sends 10 messages/day

Messages/day = 100M × 10 = 1 billion
QPS = 1B / 86400 ≈ 12K QPS (average)
Peak QPS = 12K × 3 = 36K QPS
Storage/day = 1B × 100 bytes = 100 GB/day
Storage/year = 100 GB × 365 = 36.5 TB/year
Storage/5 years = 182 TB (need distributed storage)
```


## 8.2 URL Shortener (TinyURL / Bit.ly)

### Requirements

| Feature | Details |
|---------|---------|
| **Core** | Given a long URL, generate a short URL. Redirect short URL → long URL |
| **Scale** | 100M URLs created/day, 10B redirects/day |
| **Properties** | Short URLs are 7 characters, permanent (don't expire by default) |
| **Non-functional** | Low latency redirects (< 50ms), highly available |

### Architecture

```
Client → Load Balancer → API Servers → Cache (Redis) → Database (NoSQL)
                                          ↑
                                   CDN (for popular redirects)

Create Short URL:
  POST /api/shorten {long_url: "https://..."}
  1. Generate unique short ID (Base62 encoding of counter or hash)
  2. Store mapping: short_id → long_url in database
  3. Return: https://tiny.url/{short_id}

Redirect:
  GET /{short_id}
  1. Check cache (Redis) → HIT? Redirect 301/302
  2. Cache MISS → query database → populate cache → redirect
  3. Return HTTP 301 (permanent) or 302 (temporary) redirect
```

### Key Design Decisions

| Decision | Options | Recommended |
|----------|---------|-------------|
| **ID Generation** | Hash (MD5), Counter + Base62, UUID | Counter + Base62 (no collisions, short) |
| **Database** | SQL vs NoSQL | NoSQL (DynamoDB/Cassandra) — simple key-value, massive scale |
| **Cache** | Redis, Memcached | Redis (80% hit rate for popular URLs) |
| **Redirect** | 301 vs 302 | 302 (temporary) if tracking clicks; 301 (permanent) if not |
| **Short ID length** | 6-8 chars | 7 chars (Base62^7 = 3.5 trillion unique URLs) |

### Base62 Encoding

```
Characters: a-z (26) + A-Z (26) + 0-9 (10) = 62 characters
7 characters: 62^7 = 3,521,614,606,208 ≈ 3.5 trillion unique URLs

Auto-increment counter: 1 → "1", 62 → "10", 3844 → "100"
  Counter value → convert to Base62 → use as short ID
  No collision: each counter value is unique
  
Distributed counter: use Zookeeper or pre-allocated ranges
  Server 1: counter range 1-1M
  Server 2: counter range 1M-2M
  No coordination needed within range
```


## 8.3 Chat System (WhatsApp / Slack)

### Requirements

| Feature | Details |
|---------|---------|
| **Core** | 1-on-1 chat, group chat (max 500 members), online presence |
| **Scale** | 500M DAU, 100B messages/day |
| **Properties** | Real-time delivery, message ordering, push notifications, media sharing |
| **Non-functional** | End-to-end encryption, message persistence, multi-device sync |

### Architecture

```
Clients ←WebSocket→ Chat Servers ──→ Message Queue (Kafka) ──→ Message Storage
                         │                                         │
                    User Service ←─── Presence Service         Media Storage (S3)
                         │                  │
                    Auth Service       Redis (online status)

Message Flow (User A → User B):
  1. A sends message via WebSocket to Chat Server
  2. Chat Server validates, assigns message_id + timestamp
  3. Chat Server checks: is B online?
     YES → push via B's WebSocket
     NO  → store in offline queue + send push notification
  4. Message persisted to Kafka → written to database
  5. B comes online → pull offline messages → mark as delivered
```

### Key Design Decisions

| Decision | Options | Choice |
|----------|---------|--------|
| **Protocol** | HTTP polling, Long polling, WebSocket, SSE | WebSocket (bidirectional, low latency) |
| **Message ID** | UUID, Snowflake, server timestamp | Snowflake ID (sortable, distributed, unique) |
| **Storage** | SQL, Cassandra, HBase | Cassandra (partition by chat_id, sorted by timestamp) |
| **Read receipt** | Client-side, server-side | Client sends "read" event → server updates status |
| **Group messages** | Fan-out-on-write, fan-out-on-read | Fan-out-on-write for small groups; on-read for large channels |

### Message Table Schema (Cassandra)

```
Partition Key: chat_id (all messages in a chat on same partition)
Clustering Key: message_id (ordered within partition)

CREATE TABLE messages (
  chat_id     UUID,
  message_id  BIGINT,      -- Snowflake ID (time-sortable)
  sender_id   UUID,
  content     TEXT,
  media_url   TEXT,
  created_at  TIMESTAMP,
  PRIMARY KEY (chat_id, message_id)
) WITH CLUSTERING ORDER BY (message_id ASC);
```


## 8.4 Video Streaming Platform (YouTube / Netflix)

### Requirements

| Feature | Details |
|---------|---------|
| **Core** | Upload videos, stream/watch videos, search, recommendations |
| **Scale** | 2B monthly users, 500 hours of video uploaded per minute |
| **Properties** | Adaptive bitrate streaming, multi-resolution transcoding |
| **Non-functional** | High availability, low buffering, global delivery via CDN |

### Architecture

```
Upload Flow:
  Creator → Upload Service → Object Storage (S3)
                    ↓
           Transcoding Pipeline (Queue + Workers)
              ├── 1080p, 720p, 480p, 360p
              ├── Generate thumbnails
              └── Extract metadata
                    ↓
            CDN Distribution → Edge Servers worldwide

Watch Flow:
  User → CDN Edge → Stream video chunks (adaptive bitrate)
  User → API Server → Video metadata, comments, recommendations
```

### Key Components

| Component | Technology | Purpose |
|-----------|-----------|---------|
| **Object Storage** | S3, GCS | Store original and transcoded video files |
| **Transcoding** | FFmpeg workers | Convert video to multiple resolutions and formats |
| **CDN** | CloudFront, Akamai | Serve video from nearest edge server |
| **Metadata DB** | PostgreSQL | Video info, user info, comments |
| **Search** | Elasticsearch | Full-text search on titles, descriptions |
| **Recommendations** | ML pipeline | Collaborative filtering, content-based |
| **Message Queue** | Kafka/SQS | Async transcoding job scheduling |

### Adaptive Bitrate Streaming (ABR)

```
Video transcoded into multiple bitrates:
  1080p → 5 Mbps
  720p  → 2.5 Mbps
  480p  → 1 Mbps
  360p  → 0.5 Mbps

Each resolution split into small chunks (2-10 seconds each)
Manifest file (.m3u8 for HLS or .mpd for DASH) lists all chunks and bitrates

Player monitors bandwidth:
  Good connection → request 1080p chunks
  Bandwidth drops → switch to 720p or 480p mid-stream
  Bandwidth recovers → switch back to higher quality

Result: minimal buffering, best possible quality for current network
```


## 8.5 Ride-Sharing Platform (Uber / Lyft)

### Requirements

| Feature | Details |
|---------|---------|
| **Core** | Request ride, match driver, real-time tracking, payments |
| **Scale** | 100M riders, 5M drivers, 15M trips/day |
| **Properties** | Real-time location updates (every 4 seconds), ETA calculation |
| **Non-functional** | Low latency matching (< 5 seconds), 99.99% for payment |

### Architecture

```
Rider App                                  Driver App
    │                                          │
    ├── Request Ride ──→ API Gateway           ├── Update Location (every 4s)
    │                     │                    │        │
    │               Trip Service          Location Service
    │                     │               (Redis GeoSpatial)
    │               Matching Engine            │
    │                     │   ← query nearby drivers
    │               Notification Service ──→ Push to driver
    │                     │
    │               Payment Service
    │                     │
    └── Track ride ← WebSocket → GPS tracking
```

### Key Design Decisions

| Component | Design Choice | Why |
|-----------|--------------|-----|
| **Location storage** | Redis GeoSpatial (GEOADD, GEORADIUS) | O(log N) nearest neighbor search, in-memory speed |
| **Matching** | Request → find nearest available drivers → ranked by ETA | Balance: driver proximity, rating, acceptance rate |
| **ETA** | Google Maps API + historical trip data + ML model | Accounts for traffic, road conditions, time of day |
| **Real-time tracking** | WebSocket for rider, HTTP polling every 4s for driver location | Rider needs smooth animation; driver sends periodic GPS updates |
| **Payment** | Stripe/payment processor with idempotency key | Exactly-once payment, PCI compliance |
| **Surge pricing** | Supply/demand ratio per geohash zone | Zone with low supply + high demand → price multiplier |

## 8.6 AI Chatbot System

### Requirements

| Feature | Details |
|---------|---------|
| **Core** | Conversational AI using LLMs, context-aware responses, document retrieval (RAG) |
| **Scale** | 10M users, 50M messages/day |
| **Properties** | Streaming responses, conversation history, document grounding |
| **Non-functional** | Low time-to-first-token (< 1s), content safety filtering |

### Architecture

```
User → API Gateway → Chat Service → LLM Router
                         │              │
                    History Service   ┌──┴──────────────┐
                    (Redis/DB)       │  RAG Pipeline     │
                                    │  Query → Embed    │
                                    │  → Vector Search  │
                                    │  → Context Inject │
                                    └──────────────────┘
                                          │
                                     Vector DB (Pinecone/Weaviate)
                                          │
                                     Document Store (S3 + Preprocessing)
```

### Key Components

| Component | Purpose | Technology |
|-----------|---------|-----------|
| **LLM Router** | Route to optimal model based on query complexity | Custom logic or LLM classifier |
| **Embedding Service** | Convert text to vector embeddings | OpenAI Ada, Sentence-BERT |
| **Vector Database** | Store and search document embeddings | Pinecone, Weaviate, Qdrant, pgvector |
| **Conversation Memory** | Store chat history per session | Redis (short-term), PostgreSQL (long-term) |
| **Safety Filter** | Content moderation pre/post LLM | OpenAI Moderation API, custom classifier |
| **Streaming** | Server-Sent Events for token-by-token response | SSE / WebSocket |


---
## Section 8 — Interview Questions: System Design Problems (100 Questions)


| # | Question | Keywords | Answer |
|---|----------|----------|--------|
| 286 | How do you approach a system design interview? | approach, framework | • Step 1 (5 min): Clarify requirements — functional, non-functional, scale, constraints<br>• Step 2 (10 min): High-level design — draw architecture diagram with major components<br>• Step 3 (15 min): Deep dive — detail critical components (DB schema, API, data flow)<br>• Step 4 (5 min): Trade-offs and bottlenecks — SPOF, scaling limits, consistency vs availability<br>• Always: ask questions, think aloud, discuss alternatives |
| 287 | How do you estimate QPS from DAU? | estimation, QPS | • QPS = DAU × actions_per_user / 86,400 seconds<br>• Example: 100M DAU, 10 actions/user → 100M × 10 / 86,400 ≈ 12K QPS (average)<br>• Peak QPS = average × 2-5 (depending on usage pattern)<br>• Write QPS is usually much lower than read QPS (10:1 or 100:1)<br>• State assumptions clearly and round for simplicity |
| 288 | Design a URL shortener. What's the core architecture? | URL shortener, architecture | • Write: POST /shorten → generate short_id (Base62 counter) → store mapping → return short URL<br>• Read: GET /{short_id} → check Redis cache → miss → query DB → cache → redirect (301/302)<br>• ID generation: auto-increment counter → Base62 encode → 7 chars = 3.5 trillion URLs<br>• Database: NoSQL (DynamoDB) for simple key-value at scale<br>• Cache: Redis for hot URLs (20% of URLs serve 80% of traffic) |
| 289 | How do you generate unique short IDs without collisions? | ID generation, collision | • Method 1: Centralized counter (Redis INCR) → Base62 encode — simple, no collisions<br>• Method 2: Pre-allocated ranges — each server gets 1M range, generates locally<br>• Method 3: Hash (MD5) → take first 7 chars — collision risk, need check-and-retry<br>• Method 4: Snowflake ID → timestamp + machine_id + sequence — distributed, sortable<br>• Recommended: counter + Base62 for simplicity; Snowflake for distributed |
| 290 | Should a URL shortener redirect with 301 or 302? | redirect, 301 vs 302 | • 301 Moved Permanently: browser caches redirect → fewer server hits but can't track clicks<br>• 302 Found (temporary): browser doesn't cache → every access hits server → can track analytics<br>• Use 301: if pure URL shortening (maximum performance, lower server load)<br>• Use 302: if analytics/click tracking needed (most commercial shorteners)<br>• Bit.ly uses 301 by default; analytics uses a separate tracking redirect layer |
| 291 | Design a chat system. What protocol should you use? | chat, protocol | • WebSocket: bidirectional, low latency, persistent connection — best for chat<br>• HTTP long polling: server holds request until new data — fallback for environments without WebSocket<br>• Server-Sent Events: server → client only — insufficient for chat (need bidirectional)<br>• Connection: one WebSocket per online user, maintained through heartbeats<br>• Fallback: if WebSocket unavailable, use long polling (older browsers, corporate firewalls) |
| 292 | How do you store messages in a chat system? | chat, message storage | • Cassandra: partition by chat_id, cluster by timestamp — all messages for a conversation together<br>• Write-optimized: append-only writes (fast), ordered by time (natural query pattern)<br>• Partition key: chat_id ensures all messages in a chat on same node<br>• Clustering key: message_id (Snowflake) for time-based ordering within partition<br>• Archive: move old messages to cold storage (S3) after retention period |
| 293 | How does a chat system handle group messages? | chat, group messaging | • Small groups (< 100 members): fan-out-on-write — when message sent, write to each group member's inbox<br>• Large channels (1000+ members): fan-out-on-read — store message once, each member reads on demand<br>• Hybrid: fan-out-on-write for active members, pull for inactive<br>• Delivery: iterate through member list → check online → WebSocket push or offline queue<br>• Optimization: batch push notifications for offline members |
| 294 | How does YouTube handle video uploads? | YouTube, upload | • User uploads to Upload Service → stored in S3 (original file)<br>• Transcoding pipeline: Kafka job → FFmpeg workers → multiple resolutions (1080p, 720p, 480p, 360p)<br>• Each resolution split into 2-10 second chunks for ABR (Adaptive Bitrate Streaming)<br>• Manifest file (.m3u8 or .mpd) generated listing all chunks and quality levels<br>• Chunks distributed to CDN edge servers worldwide<br>• Processing: async via job queue (may take minutes for long videos) |
| 295 | What is adaptive bitrate streaming? | ABR, streaming | • Video transcoded into multiple quality levels (bitrates)<br>• Each quality level split into small time-based chunks (2-10 seconds)<br>• Player monitors network bandwidth in real-time<br>• Good network → request high-quality chunks; slow network → switch to lower quality<br>• Protocols: HLS (Apple, .m3u8), DASH (standard, .mpd)<br>• Result: minimal buffering with best possible quality for current connection |
| 296 | How does Uber match riders with drivers? | Uber, matching | • Rider requests ride → Location Service queries Redis GeoSpatial for nearby available drivers<br>• GEORADIUS: find drivers within X km of rider's pickup point<br>• Rank candidates by: distance/ETA, driver rating, acceptance rate, vehicle type preference<br>• Send request to top-ranked driver with timeout (30 seconds to accept)<br>• If declined or timeout → send to next ranked driver<br>• Factors: surge pricing zone, estimated route efficiency |
| 297 | How does Uber track drivers in real-time? | Uber, tracking | • Driver app sends GPS coordinates every 4 seconds via HTTP to Location Service<br>• Location Service updates Redis GeoSpatial: GEOADD driver_locations {lng} {lat} {driver_id}<br>• Rider tracking: WebSocket connection → server pushes driver location updates to rider<br>• Map rendering: client interpolates between GPS updates for smooth animation<br>• Scale: 5M active drivers × update every 4s = 1.25M location updates/second |
| 298 | How do you design a notification system? | notification, design | • Types: push (mobile), email, SMS, in-app<br>• Architecture: Event → Notification Service → Priority Queue → Delivery Workers<br>• Priority: critical (payment failure) → high (message from friend) → low (marketing)<br>• Rate limiting: max notifications per user per hour (avoid spam)<br>• Preferences: user can opt out of specific notification types<br>• Delivery tracking: sent → delivered → read status |
| 299 | How do you design a news feed/timeline? | news feed, timeline | • Fan-out-on-write: when user posts, write to all followers' timelines (pre-computed)<br>• Fan-out-on-read: when user opens timeline, query all followees' posts in real-time<br>• Hybrid: fan-out-on-write for regular users; fan-out-on-read for celebrities (millions of followers)<br>• Storage: pre-computed timeline in Redis (list of post IDs per user)<br>• Feed: ranked by time, then re-ranked by ML model (relevance, engagement prediction) |
| 300 | How do you design a rate limiter service? | rate limiter, design | • Algorithm: Token Bucket (allows bursts) or Sliding Window (strict)<br>• Storage: Redis (INCR with TTL for fixed window, or Sorted Set for sliding window)<br>• Distributed: all API servers hit same Redis cluster for consistent rate limiting<br>• Response: 429 Too Many Requests with Retry-After header<br>• Rules engine: different limits per API key, per endpoint, per IP<br>• Multi-tenant: per-customer quotas stored in config (tiered pricing) |


| # | Question | Keywords | Answer |
|---|----------|----------|--------|
| 301 | How do you design a distributed cache? | cache, distributed design | • Architecture: cache cluster (Redis Cluster) with consistent hashing for key distribution<br>• Scaling: add nodes → consistent hashing minimizes key redistribution<br>• Replication: each master has 1+ replica for failover<br>• Eviction: LRU (remove least recently used) when memory full<br>• Client: cache-aside pattern (app checks cache → miss → fetch from DB → populate cache) |
| 302 | How do you design a search autocomplete system? | autocomplete, typeahead | • Data structure: Trie (prefix tree) stored in memory for O(length) prefix lookup<br>• Ranking: each node stores top-K completions (pre-computed by frequency/recency)<br>• Update: batch update trie hourly from search logs (not real-time — too expensive)<br>• Scale: distribute trie by first character range (A-G → shard 1, H-N → shard 2)<br>• Latency requirement: < 50ms for responsive autocomplete experience |
| 303 | How do you design a web crawler? | crawler, design | • Architecture: URL Frontier (queue) → Fetcher (download) → Parser (extract links/content) → Storage<br>• Politeness: respect robots.txt, rate limit per domain (1 request/second/domain)<br>• Deduplication: URL dedup (seen URLs) + content dedup (content hash)<br>• Frontier priority: prioritize by PageRank, freshness, domain authority<br>• Scale: distributed crawlers, consistent hashing for domain-to-crawler assignment |
| 304 | How do you design a payment system? | payment, design | • Idempotency: every payment request has unique idempotency_key — prevents double charges<br>• Double-entry bookkeeping: every transaction creates two entries (debit + credit)<br>• Payment states: PENDING → AUTHORIZED → CAPTURED → SETTLED (or FAILED/REFUNDED)<br>• Retry: exponential backoff for temporary failures from payment processor<br>• Reconciliation: daily reconciliation between internal records and bank statements |
| 305 | How do you design an e-commerce order system? | order, e-commerce | • Saga pattern: Create Order → Reserve Inventory → Process Payment → Create Shipment<br>• Compensations: if payment fails → release inventory → cancel order<br>• Database: Order table (PostgreSQL), status tracking (state machine)<br>• Idempotency: order_id as idempotency key for payment API call<br>• Consistency: eventual consistency across microservices (via events) |
| 306 | How do you design a file storage system (Google Drive / Dropbox)? | file storage, design | • Storage: chunked upload → S3 for file data, metadata in PostgreSQL<br>• Sync: file change → compute delta → upload changed chunks → notify other devices<br>• Deduplication: hash each chunk → if hash exists, reference existing (save storage)<br>• Version history: store each version as a set of chunks (immutable chunks, new version = new chunk list)<br>• Conflict resolution: if two devices edit same file → create conflict copy |
| 307 | How do you design a distributed task scheduler? | task scheduler, distributed | • Architecture: Scheduler → Task Queue (Kafka/SQS) → Worker Pool<br>• Scheduling: cron-like expressions stored in DB, scheduler checks every minute<br>• Deduplication: distributed lock (Redis) ensures each task scheduled once even with multiple schedulers<br>• Retry: failed tasks re-queued with exponential backoff, max retries<br>• At-least-once execution: worker commits completion after processing, not before |
| 308 | How do you design a leaderboard system? | leaderboard, design | • Data structure: Redis Sorted Set (ZADD for insert, ZREVRANGE for top-K)<br>• Operations: ZADD leaderboard {score} {user_id} — O(log N) insert<br>  ZREVRANGE leaderboard 0 99 — top 100, O(log N + 100)<br>  ZREVRANK leaderboard {user_id} — user's rank, O(log N)<br>• Scale: partition by game/category (separate sorted set per leaderboard)<br>• Real-time: scores update immediately on user action |
| 309 | How do you design an API rate limiter? | rate limiter, API design | • Token bucket algorithm: bucket capacity (max burst), refill rate (sustained rate)<br>• State: per-user/per-API-key counter in Redis (key = "rate:{user_id}", value = remaining tokens, TTL for window)<br>• Atomic: Redis MULTI/EXEC or Lua script to check + decrement atomically<br>• Distributed: all API servers share Redis → consistent limiting across servers<br>• Response: 429 with X-RateLimit-Remaining + Retry-After headers |
| 310 | How do you handle data consistency in a distributed transaction? | consistency, distributed | • Option 1: Two-Phase Commit (2PC) — strong consistency but blocking, slow<br>• Option 2: Saga pattern — eventual consistency with compensating transactions<br>• Option 3: Outbox pattern — write event to DB table, publish async (atomic with data)<br>• Choice depends on: consistency requirement (financial = strong, social = eventual)<br>• Most microservices: prefer Saga + eventual consistency for availability and performance |
| 311 | How do you design a content delivery network (CDN)? | CDN, design | • Architecture: origin server + distributed edge servers (PoPs worldwide)<br>• Request flow: client → nearest PoP → if cached, return; if not, fetch from origin → cache at edge<br>• Cache key: URL + query params + headers (Accept-Encoding, Accept-Language)<br>• Invalidation: TTL-based (30s-1h), purge API for immediate (emergency), versioned URLs<br>• Routing: DNS-based (Route53 latency routing) or Anycast (same IP, nearest server) |
| 312 | How do you design a recommendation system? | recommendation, design | • Content-based filtering: recommend items similar to what user liked (item features)<br>• Collaborative filtering: recommend items liked by similar users (user-item matrix)<br>• Hybrid: combine both approaches for better accuracy<br>• Pipeline: data collection → feature engineering → model training → online serving → A/B testing<br>• Real-time: use embedding similarity search (vector DB) for fast nearest-neighbor lookup |
| 313 | How do you design a social network graph? | social graph, design | • Data: graph database (Neo4j) or adjacency list in SQL/NoSQL<br>• Operations: follow/unfollow (add/remove edge), find friends, mutual friends, friend recommendations<br>• Scale: fan-out-on-write for feed (small graph), fan-out-on-read for feed (large graph)<br>• Storage: SQL for core profiles, NoSQL for activity feeds, graph DB for relationship queries<br>• Caching: cache friend lists in Redis (invalidated on follow/unfollow) |
| 314 | How do you design a ticket booking system? | ticket, booking | • Challenge: prevent double booking (two users booking same seat simultaneously)<br>• Solution 1: Pessimistic locking — lock seat row during booking process<br>• Solution 2: Optimistic locking — version column, retry on conflict<br>• Solution 3: Reservation hold — temporarily reserve for 10 minutes, release if not paid<br>• Payment integration: hold + charge within reservation window, release on expiry |
| 315 | How do you design a monitoring and alerting system? | monitoring, alerting | • Data collection: agents on each server → metrics (Prometheus) + logs (Fluentd) + traces (Jaeger)<br>• Storage: time-series DB (InfluxDB, Prometheus) for metrics, Elasticsearch for logs<br>• Alerting: define rules (CPU > 90% for 5 min → alert), escalation policies (PagerDuty)<br>• Dashboard: Grafana for visualization (real-time charts, heatmaps, SLO tracking)<br>• On-call: rotation, escalation, incident management workflow |


| # | Question | Keywords | Answer |
|---|----------|----------|--------|
| 316 | How do you design a unique ID generator for distributed systems? | ID generator, distributed | • Requirements: globally unique, time-sortable, distributed (no central coordination)<br>• Snowflake ID: 64-bit: 1 sign + 41 timestamp + 10 machine_id + 12 sequence<br>• Timestamp: milliseconds since epoch → sortable → 41 bits = 69 years<br>• Machine ID: 10 bits = 1024 machines. Sequence: 12 bits = 4096 IDs per ms per machine<br>• Other options: UUID v4 (random, 128 bits, not sortable), ULID (sortable UUID-like) |
| 317 | How do you handle hotspots in a distributed system? | hotspot, handling | • Problem: one shard/server gets disproportionate traffic (celebrity's data, viral content)<br>• Detection: monitor per-shard metrics (QPS, CPU, disk I/O), alert on imbalance<br>• Solution 1: add random suffix to hot key → spread across multiple shards<br>• Solution 2: dedicated shard for known hot keys (celebrity users)<br>• Solution 3: caching layer absorbs hot key reads (Redis with short TTL)<br>• Solution 4: rate limit writes to hot keys |
| 318 | How do you design a type-ahead search system? | typeahead, search | • Trie data structure: prefix tree storing all searchable terms<br>• Each node: letter + map of children + top-K suggestions (pre-computed)<br>• Lookup: traverse trie by prefix → return stored top-K suggestions → O(prefix length)<br>• Update: batch rebuild trie from aggregated search logs (hourly/daily)<br>• Scale: shard trie by prefix range → multiple servers handle different letter ranges |
| 319 | How do you design a webhook delivery system? | webhook, delivery | • Architecture: Event → Webhook Service → Delivery Queue → Delivery Workers → Customer URL<br>• Retry: exponential backoff (1min, 5min, 30min, 2h, 24h) for failed deliveries<br>• Idempotency: include event_id header → customer can deduplicate<br>• Signature: HMAC signature for customer to verify webhook authenticity<br>• Monitoring: delivery success rate, latency percentiles, DLQ for permanently failed |
| 320 | How do you design a geolocation service? | geolocation, design | • Storage: Redis GeoSpatial (GEOADD, GEORADIUS) for in-memory spatial queries<br>• Geohash: encode lat/lng to string — nearby points share prefix → efficient range queries<br>• QuadTree: spatial index dividing space recursively — efficient for point queries<br>• Use case: "find all restaurants within 5km" → GEORADIUS restaurants 77.6 12.9 5 km<br>• Scale: shard by geographic region (one Redis instance per city/country) |
| 321 | What are the trade-offs in system design? | trade-offs, design | • Consistency vs Availability (CAP): strong consistency = higher latency, lower availability<br>• Latency vs Throughput: batching increases throughput but adds latency<br>• SQL vs NoSQL: SQL = strong consistency, joins; NoSQL = scalability, flexibility<br>• Monolith vs Microservices: simplicity vs independent scalability<br>• Cost vs Performance: more servers = better performance = higher cost<br>• Always state trade-offs explicitly in interview |
| 322 | How do you handle data migration in system design? | data migration, strategy | • Strategy 1: dual-write — write to both old and new system, switch reads gradually<br>• Strategy 2: blue-green — migrate data, switch all traffic at once, rollback to old if issues<br>• Strategy 3: CDC-based — Debezium captures changes from old DB, streams to new DB<br>• Verify: run both systems in parallel, compare results (shadow traffic)<br>• Rollback plan: always have a way to switch back to old system |
| 323 | How do you design for multi-tenancy? | multi-tenancy, design | • Isolated databases: separate DB per tenant — strongest isolation, highest cost<br>• Shared database, separate schemas: one DB, schema per tenant — moderate isolation<br>• Shared everything: one DB, tenant_id column — easiest to manage, noisy neighbor risk<br>• Decision: regulated industries → isolated; SaaS → shared with row-level security<br>• Resource quotas: per-tenant rate limits, storage limits, compute limits |
| 324 | How do you design an event-driven notification system? | notification, event-driven | • Events: user actions → event bus (Kafka) → notification router<br>• Router: checks notification preferences and template → creates notification<br>• Channels: push, email, SMS, in-app — each has separate delivery service<br>• Priority queue: urgent notifications (payment failure) → immediate; marketing → batched<br>• Deduplication: don't send same notification twice (idempotency key per event+channel) |
| 325 | How do you design a distributed configuration management system? | config management, distributed | • Architecture: config store (etcd/Consul) → clients watch for changes → apply updates<br>• Versioning: each config change creates new version (rollback by reverting to previous version)<br>• Validation: schema validation before accepting config changes (prevent invalid configs)<br>• Feature flags: special case of configuration — enable/disable features per user/percentage<br>• Security: encrypt sensitive configs, RBAC for who can modify configs |
| 326 | How do you handle database schema changes with zero downtime? | schema migration, zero downtime | • Rule: only additive changes in production (add column, add table, add index)<br>• Phase 1: add new column (nullable or with default) → deploy code that writes to both old and new<br>• Phase 2: backfill old data into new column<br>• Phase 3: deploy code that reads from new column only<br>• Phase 4: drop old column (after verification period)<br>• Never: rename column, change type, drop column in single deployment |
| 327 | How do you design a full-text search engine? | search engine, design | • Components: crawler/ingester → indexer → query engine → ranker<br>• Inverted index: word → list of documents containing that word<br>• Tokenization: "hello world" → ["hello", "world"]. Stemming: "running" → "run"<br>• Ranking: TF-IDF (term frequency × inverse document frequency), BM25, PageRank<br>• Technology: Elasticsearch (distributed, REST API), Apache Solr |
| 328 | How do you design a system for handling peak traffic? | peak traffic, design | • CDN: cache static assets at edge (absorb 60-70% of traffic)<br>• Auto-scaling: target tracking scaling policy → add instances on CPU/request spike<br>• Cache warming: pre-populate cache before expected peak (Black Friday, product launch)<br>• Queue: buffer bursty writes in Kafka → workers process at sustainable rate<br>• Graceful degradation: disable non-essential features during peak (recommendations, analytics) |
| 329 | How do you ensure data durability across failures? | data durability, failures | • Write-ahead log (WAL): write to log before applying to database → replay on crash recovery<br>• Replication: synchronous replication to at least one replica (semi-sync minimum)<br>• Backups: regular snapshots + continuous WAL archiving (point-in-time recovery)<br>• Multi-AZ deployment: replicas in different availability zones<br>• Test: regular restore drills to verify backups actually work |
| 330 | Design a system that scales from 1K to 1M users. | scaling stages, growth | • 1K users: single server (app + DB on one machine), vertical scaling<br>• 10K users: separate DB server, add cache (Redis), CDN for static assets<br>• 100K users: load balancer + multiple app servers, read replicas for DB<br>• 500K users: database sharding, message queue for async processing, microservice extraction<br>• 1M users: full horizontal scaling, multi-AZ deployment, auto-scaling, comprehensive monitoring<br>• Key: add complexity only when needed — premature optimization wastes resources |


| # | Question | Keywords | Answer |
|---|----------|----------|--------|
| 331 | How do you design a distributed logging system? | logging, distributed | • Collection: Fluentd/Filebeat agents on each server → forward logs<br>• Transport: Kafka (buffer and durability) for log stream<br>• Processing: Logstash or Flink for parsing, enriching, filtering<br>• Storage: Elasticsearch (searchable) for recent logs, S3 for archive<br>• Visualization: Kibana/Grafana for dashboards, search, alerting<br>• Correlation: trace ID in every log line for cross-service debugging |
| 332 | Design a key-value store. What are the key components? | key-value store, design | • Write path: client → server → write to WAL → update in-memory hash table → return success<br>• Read path: check in-memory table → if not found → check on-disk SSTables → bloom filter to skip absent files<br>• Compaction: merge SSTables periodically (remove old versions, reclaim space)<br>• Replication: configurable replication factor, quorum reads/writes<br>• Partitioning: consistent hashing for key-to-node assignment |
| 333 | How do you design a messaging queue system? | message queue, design | • Producer → topic → partitions → consumer groups<br>• Storage: append-only log on disk (fast sequential writes)<br>• Retention: time-based (7 days) or size-based (100 GB)<br>• Ordering: guaranteed within partition (use message key for related messages)<br>• Delivery: at-least-once (default), exactly-once with idempotent producer + transactional consumer |
| 334 | How do you design a global DNS system? | DNS, design | • Hierarchical: Root servers → TLD servers (.com) → Authoritative servers (example.com)<br>• Caching: recursive resolvers cache responses (TTL-based)<br>• Load distribution: DNS round-robin, weighted, geolocation-based, latency-based<br>• Redundancy: multiple authoritative servers, anycast for root servers<br>• Scale: CDN providers operate DNS with thousandss of PoPs worldwide |
| 335 | How do you design a distributed counter? | counter, distributed | • Approach 1: centralized counter in Redis (INCR) — simple, accurate, ~100K INCR/s<br>• Approach 2: per-node local counters → periodic aggregation (eventual consistency)<br>• Approach 3: CRDT G-Counter — each node has own counter, total = sum of all<br>• Read: sum all local counters (eventually consistent) or read centralized (strongly consistent)<br>• Trade-off: real-time accuracy vs performance at extreme scale |
| 336 | How do you design an email delivery system? | email, delivery | • Components: API → queue (SQS) → email renderer → SMTP sender → bounce handler<br>• Templates: store email templates with variables ({name}, {link})<br>• Rate limiting: ISPs rate-limit senders — throttle per domain (Gmail: ~500/hour for new sender)<br>• Reputation: dedicated IP, SPF/DKIM/DMARC records, warm up IP gradually<br>• Tracking: pixel tracking for opens, link wrapping for clicks |
| 337 | How would you design Twitter's trending topics? | trending, design | • Count: sliding window counter for each hashtag (Redis Sorted Set + expire)<br>• Score: frequency in last hour / average frequency (detect sudden spike, not just popularity)<br>• Pipeline: tweet → extract hashtags → increment counters → compute trending scores hourly<br>• Personalization: filter by user's region, language, interests<br>• Anti-gaming: rate limit per user, detect bot patterns (same hashtag from many new accounts) |
| 338 | How do you design an image upload and serving system? | image, upload serving | • Upload: client → API → validate (size, type) → process (resize, compress) → store in S3<br>• Processing: generate thumbnails (150px, 300px, 600px, 1200px) via background workers<br>• Serving: CDN edge servers cache images (90%+ cache hit rate for popular images)<br>• URL: versioned or content-hashed (`img_abc123_300.webp`) for cache invalidation<br>• Optimization: WebP format (30% smaller than JPEG), lazy loading, responsive srcset |
| 339 | How do you design a real-time analytics dashboard? | analytics, real-time | • Ingestion: events → Kafka → stream processor (Flink/Spark Streaming)<br>• Processing: windowed aggregations (count, sum, average over 1min, 5min, 1hr windows)<br>• Storage: time-series DB (InfluxDB, TimescaleDB) for aggregated metrics<br>• Visualization: Grafana dashboards with auto-refresh (every 5-10 seconds)<br>• Pre-computation: compute aggregates at ingestion time, not at query time |
| 340 | How do you design a system for handling idempotent payments? | idempotent payment, design | • Client generates unique idempotency_key (UUID v4) for each payment intent<br>• Server: check Redis/DB for key → if exists, return cached result (no double charge)<br>• If new: process payment → save result under key with 24-48h TTL<br>• Database: unique constraint on idempotency_key (race condition protection)<br>• State machine: INITIATED → PROCESSING → SUCCEEDED/FAILED (prevents concurrent processing) |
| 341 | How do you design a system to handle 10 million concurrent WebSocket connections? | WebSocket, scale | • Challenge: each WebSocket = persistent TCP connection = memory per connection<br>• Server: Go or C++ (efficient per-connection memory, ~10K-50K connections per server)<br>• Servers needed: 10M / 50K = 200 servers minimum<br>• Pub/Sub: Redis Pub/Sub or Kafka for cross-server message delivery<br>• Connection management: reconnection logic, heartbeat/ping-pong, connection draining on deploy |
| 342 | How do you design a rate limiter that works across multiple data centers? | rate limiter, multi-region | • Approach 1: local rate limiting per region (each region gets proportional share of total limit)<br>• Approach 2: centralized Redis with cross-region replication (consistent but higher latency)<br>• Approach 3: eventual consistency — local counters + periodic sync (slight over-limit possible)<br>• Trade-off: strict global limit (high latency) vs approximate limit (low latency)<br>• Most APIs: approximate is fine (allow 5% over-limit during sync windows) |
| 343 | How do you design a system for serving machine learning models? | ML serving, design | • Architecture: API Gateway → Model Service → Model Runtime (TensorFlow Serving, TorchServe)<br>• Hosting: GPU servers for inference, auto-scaled based on request volume<br>• Model registry: store model versions (MLflow, S3) with metadata (accuracy, training data)<br>• A/B testing: route percentage of traffic to new model version, compare metrics<br>• Latency: batch prediction (offline, high throughput) vs online prediction (real-time, P99 < 100ms) |
| 344 | How do you design a URL redirect counting system at scale? | click counting, scale | • Challenge: billions of redirect requests per day, need accurate counts<br>• Write path: redirect happens → increment counter (don't block redirect on DB write)<br>• Async counting: log click to Kafka → stream processor aggregates → writes to DB periodically<br>• Approximate counts: HyperLogLog for unique visitors (12 KB per URL, ~0.81% error)<br>• Exact counts: sharded Redis counters, periodic flush to persistent DB |
| 345 | How do you design a feature flag system? | feature flags, system | • Architecture: Feature Flag Service (API) → Client SDKs (cached evaluation)<br>• Storage: PostgreSQL for flag definitions, Redis for fast evaluation cache<br>• Evaluation: flag rules (percentage rollout, user segment, allow-list) evaluated client-side from cache<br>• SDK: poll for updates every 30s or use SSE for real-time flag changes<br>• Safety: flags as kill switches — disable problematic features without deploy |


| # | Question | Keywords | Answer |
|---|----------|----------|--------|
| 346 | How do you design a batch processing pipeline? | batch processing, pipeline | • Architecture: Scheduler (Airflow) → Data Source → Processing (Spark) → Output → Monitoring<br>• DAG: Directed Acyclic Graph defines task dependencies (extract → transform → load)<br>• Idempotency: each batch run is rerunnable without side effects (overwrite, not append-duplicate)<br>• Monitoring: track job status, duration, record counts, failure alerts<br>• Recovery: failed tasks retry automatically, DAG restarts from failed step |
| 347 | How do you design a data lake? | data lake, design | • Storage: S3 or GCS with partitioned directory structure (year/month/day)<br>• Format: Parquet (columnar, compressed, efficient for analytics queries)<br>• Catalog: AWS Glue / Hive Metastore for table definitions and schema<br>• Processing: Spark/Presto/Athena for SQL queries on data lake files<br>• Governance: data quality checks, access control, PII encryption, retention policies |
| 348 | How do you design a system for deduplication at scale? | deduplication, scale | • Exact dedup: hash content (SHA-256) → check hash in DB/Redis → if exists, duplicate<br>• Probabilistic: Bloom filter (check membership without storing all items, false positives OK)<br>• Fuzzy dedup: similarity hashing (SimHash, MinHash) for near-duplicate detection<br>• Use case: email dedup (exact), document dedup (fuzzy), ad click dedup (exact + time window)<br>• Trade-off: exact is expensive (store all hashes); probabilistic is cheap but imprecise |
| 349 | How do you design a secret management system? | secrets, management | • Architecture: centralized vault (HashiCorp Vault, AWS Secrets Manager)<br>• Access: short-lived tokens with specific permissions (least privilege)<br>• Rotation: automatic rotation of database passwords, API keys (no hardcoded secrets)<br>• Encryption: encrypt at rest (AES-256) and in transit (TLS)<br>• Injection: secrets injected as environment variables or mounted files (not in code/config) |
| 350 | Design a real-world system: E-Commerce platform at scale. | e-commerce, complete design | • Services: User, Product Catalog, Cart, Order, Payment, Inventory, Shipping, Notification<br>• Database: PostgreSQL (transactional), Redis (cache, sessions), Elasticsearch (search)<br>• Payments: Stripe integration with idempotency keys, Saga pattern (order → payment → inventory)<br>• Scale: CDN for static assets, read replicas for catalog reads, sharding by user_id for orders<br>• Reliability: circuit breakers between services, async notifications, event-driven inventory updates<br>• Monitoring: RED metrics (Rate, Errors, Duration), distributed tracing, on-call rotation |
| 351 | What is the difference between vertical and horizontal scaling in databases? | database scaling, vertical horizontal | • Vertical: upgrade CPU/RAM of existing database server (limited by hardware max, ~96 cores, 768GB RAM)<br>• Horizontal: shard data across multiple database servers (theoretically unlimited)<br>• Vertical: simpler (no distributed logic needed), but has ceiling<br>• Horizontal: requires sharding strategy, cross-shard queries are expensive, rebalancing needed<br>• Approach: start vertical → switch to horizontal when approaching hardware limits |
| 352 | Scenario: Your system is experiencing cascading failures. How do you diagnose and fix? | cascading failure, diagnosis | • Symptoms: one service slows down → callers time out → callers' callers time out → domino effect<br>• Diagnosis: check metrics dashboard → identify the root service → check its dependencies<br>• Immediate fix: circuit breakers (stop calling failing service, return cached/default response)<br>• Short-term: add timeouts on all outbound calls, add bulkheads (isolated thread pools)<br>• Long-term: async communication (queues), graceful degradation, chaos engineering testing |
| 353 | How do you estimate storage requirements for a system? | storage estimation, calculation | • Identify: what data is stored (users, messages, media, logs, metadata)<br>• Estimate per-item size: user profile ~1KB, message ~100B, image ~200KB, video ~50MB<br>• Calculate daily growth: DAU × actions/day × item_size<br>• Multiply by retention: 5 years = 365 × 5 × daily_growth<br>• Add overhead: replication (3x), indexes (~30%), backups (2x), headroom (30%)<br>• Rule of thumb: multiply raw estimate by 10 for total storage with replication and backups |
| 354 | What are the key differences between SQL and NoSQL in system design? | SQL vs NoSQL, comparison | • SQL: structured schema, ACID transactions, powerful JOINs, vertical scaling primarily<br>• NoSQL: flexible schema, BASE model (eventual consistency), horizontal scaling natively<br>• Choose SQL: complex queries, transactions needed, data integrity critical (banking, inventory)<br>• Choose NoSQL: massive scale, flexible schema, simple key-value or document access (social feed, IoT)<br>• Many modern systems use both: SQL for transactional core, NoSQL for read-heavy/scalable components |
| 355 | How do you handle version conflicts in distributed systems? | version conflict, distributed | • Optimistic concurrency: version number on each record, update only if version matches<br>• Last-write-wins: simple but may lose data (acceptable for non-critical data)<br>• Vector clocks: detect concurrent writes, surface conflict to application for resolution<br>• CRDTs: data structures that merge automatically without conflicts<br>• Application-level: present both versions to user, let them choose (Google Docs, Git) |
| 356 | Design a system: How would you build a global content moderation system? | content moderation, design | • Pipeline: content uploaded → ML classifier (text/image/video) → confidence score<br>• High confidence (spam/obvious violation): auto-remove immediately<br>• Medium confidence: queue for human review (content moderation team)<br>• Low confidence (likely safe): publish, sample for quality review<br>• ML models: text classifier (hate speech, spam), image classifier (NSFW), video frame sampling<br>• Scale: parallel processing workers, multiple ML models, regional compliance rules |
| 357 | How would you design a multi-region active-active database? | multi-region, active-active | • Challenge: writes in multiple regions → conflict resolution needed<br>• Approach 1: CockroachDB/Spanner (distributed SQL with global transactions via TrueTime/hybrid logical clocks)<br>• Approach 2: Cassandra (leaderless, LWW conflict resolution, tunable consistency)<br>• Approach 3: DynamoDB Global Tables (multi-region, eventual consistency, LWW)<br>• Latency: cross-region replication adds 50-200ms<br>• Trade-off: strong global consistency = higher latency (Spanner); eventual = lower latency (Cassandra) |
| 358 | What are the common bottlenecks in a web application? | bottlenecks, web app | • Database: slow queries, connection exhaustion, lock contention → index, cache, read replicas<br>• CPU: computation-heavy endpoints → horizontal scaling, async processing<br>• Network: bandwidth saturation → CDN, compression, pagination<br>• Memory: large in-memory datasets, connection objects → right-size, pool, offload to Redis<br>• Single-threaded: GIL in Python, main thread in Node → worker processes, move to Go for CPU-bound |
| 359 | How do you make a system observable? | observability, implementation | • Structured logging: JSON logs with consistent fields (timestamp, trace_id, service, level, message)<br>• Metrics: Prometheus metrics endpoint, custom counters and histograms for business metrics<br>• Tracing: OpenTelemetry SDK for automatic span creation and propagation<br>• Dashboards: Grafana for real-time visualization, Prometheus rules for alerting<br>• On-call: PagerDuty integration, runbooks linked to alerts, escalation policies |
| 360 | Scenario: Design an architecture for a startup that expects to grow from 0 to 10M users in 2 years. | startup scaling, architecture | • Month 1-3: monolith on single server, PostgreSQL, basic caching<br>• Month 3-6: add Redis cache, move to managed DB (RDS), CDN for static assets<br>• Month 6-12: load balancer + 2-3 app servers, read replicas, background workers (Celery/SQS)<br>• Year 1-2: extract hot paths to microservices, add monitoring stack, database sharding for write-heavy tables<br>• Year 2+: full microservices, event-driven architecture, multi-AZ, auto-scaling, chaos engineering<br>• Principle: add complexity only when needed — premature optimization is the root of all evil |
